In [1]:
"""
VisionServeAI - Sprint 05
Stage 1: Deployment Environment & Artifact Discovery
======================================================================
Single-responsibility stage: prepares the deployment environment only.
No model loading. No inference. No ONNX export. No benchmarking.
No GradCAM. No metric recomputation. No calibration. No thresholding.
======================================================================
"""

from __future__ import annotations

import json
import logging
import os
import platform
import random
import shutil
import sys
import time
from dataclasses import dataclass, field, asdict
from pathlib import Path
from typing import Any, Dict, List, Optional

import psutil
import torch
import torchvision

# ======================================================================
# CONSTANTS
# ======================================================================

SEED: int = 42
KAGGLE_INPUT_ROOT = Path("/kaggle/input")
OUTPUT_ROOT = Path("/kaggle/working/sprint05_deployment")
STAGE_DIR_NAME = "stage01_environment"
MAX_SCAN_DEPTH = 6

# Fingerprint files used to IDENTIFY which mounted Kaggle dataset satisfies
# each required artifact source. Purely content-based -- no dataset slug
# or username is ever hardcoded, per the robust-discovery requirement.
FINGERPRINTS: Dict[str, List[str]] = {
    "sprint03": [
        "disease_registry.json",
        "dataset_statistics.json",
        "train_manifest.csv",
    ],
    "sprint04_training": [
        "best_model.pt",
        "training_summary.json",
        "checkpoint_summary.json",
    ],
    "sprint04_evaluation": [
        "evaluation_summary.json",
        "optimal_thresholds.json",
        "deployment_recommendations.json",
    ],
    "nih_chest_xray": [
        "Data_Entry_2017.csv",
        "BBox_List_2017.csv",
    ],
}

# Files whose absence is FATAL (stage fails loudly, fail-fast).
CRITICAL_FILES: Dict[str, List[str]] = {
    "sprint03": ["disease_registry.json"],
    "sprint04_training": ["best_model.pt", "training_summary.json"],
    "sprint04_evaluation": ["evaluation_summary.json"],
    "nih_chest_xray": ["Data_Entry_2017.csv"],
}

# Files that are recorded but only WARN if missing (nice-to-have, informational).
OPTIONAL_FILES: Dict[str, List[str]] = {
    "sprint03": [
        "class_distribution.json",
        "class_weights.json",
        "dataset_statistics.json",
        "dataset_summary.json",
        "train_manifest.csv",
        "val_manifest.csv",
        "test_manifest.csv",
    ],
    "sprint04_training": [
        "checkpoint_summary.json",
        "optimizer_configuration.json",
        "scheduler_configuration.json",
        "training_report.json",
        "training_history.json",
    ],
    "sprint04_evaluation": [
        "evaluation_report.json",
        "deployment_readiness.json",
        "publication_readiness.json",
        "optimal_thresholds.json",
        "calibration_summary.json",
    ],
    "nih_chest_xray": ["BBox_List_2017.csv"],
}


# ======================================================================
# LOGGING
# ======================================================================

def build_logger(log_dir: Path) -> logging.Logger:
    """Create the Sprint04-style stage logger (console + file, no prints)."""
    log_dir.mkdir(parents=True, exist_ok=True)
    logger = logging.getLogger("sprint05.stage01")
    logger.setLevel(logging.INFO)
    logger.handlers.clear()
    logger.propagate = False

    fmt = logging.Formatter(
        fmt="%(asctime)s | %(levelname)-8s | %(message)s",
        datefmt="%Y-%m-%d %H:%M:%S",
    )

    file_handler = logging.FileHandler(log_dir / "stage01_environment.log", mode="w")
    file_handler.setFormatter(fmt)
    logger.addHandler(file_handler)

    stream_handler = logging.StreamHandler(sys.stdout)
    stream_handler.setFormatter(fmt)
    logger.addHandler(stream_handler)

    return logger


def get_resource_usage() -> Dict[str, Any]:
    """Snapshot CPU / RAM / Disk / GPU-memory usage."""
    vm = psutil.virtual_memory()
    disk = shutil.disk_usage("/")
    usage: Dict[str, Any] = {
        "cpu_percent": psutil.cpu_percent(interval=0.2),
        "ram_percent": vm.percent,
        "ram_used_gb": round(vm.used / (1024 ** 3), 2),
        "ram_total_gb": round(vm.total / (1024 ** 3), 2),
        "disk_percent": round(disk.used / disk.total * 100, 2),
        "disk_free_gb": round(disk.free / (1024 ** 3), 2),
    }
    if torch.cuda.is_available():
        usage["gpu_memory_allocated_gb"] = round(torch.cuda.memory_allocated() / (1024 ** 3), 3)
        usage["gpu_memory_reserved_gb"] = round(torch.cuda.memory_reserved() / (1024 ** 3), 3)
    return usage


def log_resources(logger: logging.Logger, tag: str) -> Dict[str, Any]:
    """Log and return a resource-usage snapshot."""
    usage = get_resource_usage()
    logger.info(
        "RESOURCES [%s] cpu=%.1f%% ram=%.1f%%(%.1fGB/%.1fGB) disk=%.1f%% free=%.1fGB",
        tag, usage["cpu_percent"], usage["ram_percent"],
        usage["ram_used_gb"], usage["ram_total_gb"],
        usage["disk_percent"], usage["disk_free_gb"],
    )
    return usage


# ======================================================================
# CONFIG DATACLASSES
# ======================================================================

@dataclass
class LoggingConfig:
    log_dir: str
    log_level: str = "INFO"
    log_format: str = "%(asctime)s | %(levelname)-8s | %(message)s"
    log_to_file: bool = True
    log_to_console: bool = True


@dataclass
class BenchmarkConfig:
    enabled: bool = False           # reserved for a later Sprint05 stage
    warmup_iterations: int = 10
    benchmark_iterations: int = 100
    batch_sizes: List[int] = field(default_factory=lambda: [1, 4, 8, 16])


@dataclass
class ExportConfig:
    onnx_enabled: bool = False      # reserved for a later Sprint05 stage
    onnx_opset: int = 17
    torchscript_enabled: bool = False
    export_dir: str = str(OUTPUT_ROOT / "exports")


@dataclass
class APIConfig:
    host: str = "0.0.0.0"
    port: int = 8000
    reserved: bool = True           # serving is not activated in Stage 1


@dataclass
class RuntimeConfig:
    seed: int = SEED
    deterministic: bool = True
    num_workers: int = 2
    pin_memory: bool = True
    use_amp: bool = False           # reserved -- no inference happens here


@dataclass
class DeploymentConfig:
    device: str
    dtype: str
    artifact_roots: Dict[str, Optional[str]]
    output_root: str
    logging: LoggingConfig
    benchmark: BenchmarkConfig
    export: ExportConfig
    api: APIConfig
    runtime: RuntimeConfig
    tensorrt_enabled_future: bool = False

    def to_dict(self) -> Dict[str, Any]:
        return asdict(self)


# ======================================================================
# SEED
# ======================================================================

def set_seed(seed: int = SEED) -> None:
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


# ======================================================================
# ENVIRONMENT INFORMATION
# ======================================================================

def _onnx_available() -> bool:
    try:
        import onnx  # noqa: F401
        return True
    except ImportError:
        return False


def get_environment_info(seed: int) -> Dict[str, Any]:
    """Collect static environment / hardware information."""
    cuda_available = torch.cuda.is_available()
    gpu_name = torch.cuda.get_device_name(0) if cuda_available else None
    gpu_mem_gb = (
        round(torch.cuda.get_device_properties(0).total_memory / (1024 ** 3), 2)
        if cuda_available else None
    )
    is_kaggle = bool(os.environ.get("KAGGLE_KERNEL_RUN_TYPE")) or KAGGLE_INPUT_ROOT.exists()

    return {
        "python_version": sys.version.split()[0],
        "torch_version": torch.__version__,
        "torchvision_version": torchvision.__version__,
        "cuda_version": torch.version.cuda,
        "cudnn_version": torch.backends.cudnn.version() if cuda_available else None,
        "gpu_available": cuda_available,
        "gpu_name": gpu_name,
        "gpu_memory_gb": gpu_mem_gb,
        "cpu_cores": psutil.cpu_count(logical=True),
        "cpu_cores_physical": psutil.cpu_count(logical=False),
        "ram_total_gb": round(psutil.virtual_memory().total / (1024 ** 3), 2),
        "os": f"{platform.system()} {platform.release()}",
        "platform": platform.platform(),
        "is_kaggle_environment": is_kaggle,
        "random_seed": seed,
        "device": "cuda" if cuda_available else "cpu",
        "mixed_precision_available": cuda_available and hasattr(torch.cuda, "amp"),
        "torch_compile_available": hasattr(torch, "compile"),
        "onnx_available": _onnx_available(),
        "torchscript_available": hasattr(torch, "jit"),
    }


# ======================================================================
# ARTIFACT DISCOVERY
# ======================================================================

def _bounded_walk(root: Path, max_depth: int):
    """os.walk bounded to max_depth (root itself = depth 0)."""
    root_depth = len(root.parts)
    for dirpath, dirnames, filenames in os.walk(root):
        depth = len(Path(dirpath).parts) - root_depth
        if depth >= max_depth:
            dirnames[:] = []
        yield Path(dirpath), filenames


def _find_files(dataset_root: Path, filenames: List[str], max_depth: int = MAX_SCAN_DEPTH) -> Dict[str, Optional[str]]:
    """Search (bounded-depth) for each filename inside dataset_root."""
    found: Dict[str, Optional[str]] = {name: None for name in filenames}
    remaining = set(filenames)
    if not remaining:
        return found
    for dirpath, files in _bounded_walk(dataset_root, max_depth):
        for fname in list(remaining):
            if fname in files:
                found[fname] = str(dirpath / fname)
                remaining.discard(fname)
        if not remaining:
            break
    return found


def _common_ancestor(paths: List[str]) -> Optional[Path]:
    """
    Return the deepest directory common to every given file path.

    This is what lets a category resolve to the actual artifact
    subdirectory (e.g. '.../visionserveai/sprint03') instead of the
    Kaggle mount point that happens to contain it (e.g. '/kaggle/input/datasets'),
    even when several logical datasets are merged under one mount.
    """
    resolved = [Path(p) for p in paths if p]
    if not resolved:
        return None
    parent_parts = [p.parent.parts for p in resolved]
    common: List[str] = []
    for parts_at_depth in zip(*parent_parts):
        if len(set(parts_at_depth)) == 1:
            common.append(parts_at_depth[0])
        else:
            break
    return Path(*common) if common else None


def discover_kaggle_datasets(logger: logging.Logger) -> Dict[str, Path]:
    """List every mounted dataset directory under /kaggle/input."""
    if not KAGGLE_INPUT_ROOT.exists():
        raise RuntimeError(f"Kaggle input root not found: {KAGGLE_INPUT_ROOT}")
    roots = {p.name: p for p in KAGGLE_INPUT_ROOT.iterdir() if p.is_dir()}
    if not roots:
        raise RuntimeError(f"No datasets mounted under {KAGGLE_INPUT_ROOT}")
    logger.info("DISCOVERY found %d mounted dataset(s) under %s", len(roots), KAGGLE_INPUT_ROOT)
    for name in roots:
        logger.info("DISCOVERY   - %s", name)
    return roots


def classify_datasets(dataset_roots: Dict[str, Path], logger: logging.Logger) -> Dict[str, Dict[str, Any]]:
    """
    Identify which artifact ROOT (not merely which mounted dataset) satisfies
    each required source, using content fingerprints. A category's resolved
    root is the deepest directory common to all of its located fingerprint
    files -- this correctly separates multiple logical datasets that happen
    to be mounted under a single Kaggle input folder, while still using no
    hardcoded slugs or usernames.
    """
    candidates: Dict[str, List[Dict[str, Any]]] = {category: [] for category in FINGERPRINTS}

    for dataset_name, dataset_path in dataset_roots.items():
        for category, fingerprint_files in FINGERPRINTS.items():
            hits = _find_files(dataset_path, fingerprint_files)
            found_paths = [v for v in hits.values() if v is not None]
            score = len(found_paths)
            if score == 0:
                continue
            artifact_root = _common_ancestor(found_paths)
            if artifact_root is None:
                continue
            candidates[category].append({
                "matched_dataset": dataset_name,
                "root": str(artifact_root),
                "score": score,
            })

    classification: Dict[str, Dict[str, Any]] = {}
    for category, cand_list in candidates.items():
        if not cand_list:
            classification[category] = {"matched_dataset": None, "root": None, "score": 0, "candidates": []}
            logger.warning("DISCOVERY category='%s' -> NOT FOUND", category)
            continue

        # Collapse candidates that independently agree on the same resolved root.
        unique_by_root: Dict[str, Dict[str, Any]] = {}
        for c in cand_list:
            existing = unique_by_root.get(c["root"])
            if existing is None or c["score"] > existing["score"]:
                unique_by_root[c["root"]] = c
        deduped = list(unique_by_root.values())

        best_score = max(c["score"] for c in deduped)
        top_candidates = [c for c in deduped if c["score"] == best_score]
        best = top_candidates[0]

        classification[category] = {
            "matched_dataset": best["matched_dataset"],
            "root": best["root"],
            "score": best["score"],
            "candidates": deduped,
        }

        if len(top_candidates) > 1:
            logger.warning(
                "DISCOVERY category='%s' -> AMBIGUOUS: %d equally valid roots %s",
                category, len(top_candidates), [c["root"] for c in top_candidates],
            )
        else:
            logger.info(
                "DISCOVERY category='%s' -> dataset='%s' root='%s' (fingerprint score %d/%d)",
                category, best["matched_dataset"], best["root"], best["score"], len(FINGERPRINTS[category]),
            )

    return classification


def check_duplicate_matches(classification: Dict[str, Dict[str, Any]]) -> List[str]:
    """
    Detect genuine ambiguity: a single category resolving to more than one
    equally valid artifact root (e.g. the same dataset attached twice, or a
    fingerprint file duplicated in two unrelated locations). Two DIFFERENT
    categories legitimately sharing one mounted Kaggle dataset is expected
    and is NOT a duplicate -- classify_datasets already narrows each
    category down to its own deep artifact root, so this only fires on
    real collisions.
    """
    duplicates: List[str] = []
    for category, result in classification.items():
        cand_list = result.get("candidates", [])
        if len(cand_list) <= 1:
            continue
        best_score = result["score"]
        tied_roots = [c["root"] for c in cand_list if c["score"] == best_score]
        if len(tied_roots) > 1:
            duplicates.append(f"'{category}' has {len(tied_roots)} equally valid candidate roots: {tied_roots}")
    return duplicates


def build_artifact_inventory(classification: Dict[str, Dict[str, Any]], logger: logging.Logger) -> Dict[str, Any]:
    """For every matched dataset, record critical + optional file paths."""
    inventory: Dict[str, Any] = {}
    for category, result in classification.items():
        root = result["root"]
        if root is None:
            inventory[category] = {"root": None, "matched_dataset": None, "critical": {}, "optional": {}}
            continue
        root_path = Path(root)
        critical = _find_files(root_path, CRITICAL_FILES.get(category, []))
        optional = _find_files(root_path, OPTIONAL_FILES.get(category, []))
        inventory[category] = {
            "root": root,
            "matched_dataset": result["matched_dataset"],
            "critical": critical,
            "optional": optional,
        }
        logger.info(
            "ARTIFACT INVENTORY '%s': critical %d/%d found, optional %d/%d found",
            category,
            sum(1 for v in critical.values() if v), len(critical),
            sum(1 for v in optional.values() if v), len(optional),
        )
    return inventory


# ======================================================================
# OUTPUT DIRECTORIES
# ======================================================================

def create_output_dirs() -> Dict[str, Path]:
    """Create ONLY the directories mandated for Stage 1 -- nothing else."""
    stage_root = OUTPUT_ROOT / STAGE_DIR_NAME
    subdirs = {
        "stage_root": stage_root,
        "logs": stage_root / "logs",
        "configs": stage_root / "configs",
        "metadata": stage_root / "metadata",
        "artifacts": stage_root / "artifacts",
    }
    for path in subdirs.values():
        path.mkdir(parents=True, exist_ok=True)
    return subdirs


def verify_output_writable(path: Path) -> bool:
    probe = path / ".write_probe"
    try:
        probe.write_text("ok")
        probe.unlink()
        return True
    except OSError:
        return False


# ======================================================================
# ENGINEERING VALIDATION
# ======================================================================

def run_engineering_validation(
    env_info: Dict[str, Any],
    inventory: Dict[str, Any],
    duplicates: List[str],
    dirs: Dict[str, Path],
    logger: logging.Logger,
) -> Dict[str, Any]:
    checks: Dict[str, bool] = {}
    warnings: List[str] = []
    fatal_errors: List[str] = []

    # 1. All required dataset categories resolved
    for category in FINGERPRINTS:
        resolved = inventory[category]["root"] is not None
        checks[f"dataset_found__{category}"] = resolved
        if not resolved:
            fatal_errors.append(f"Required dataset category not found: {category}")

    # 2. Critical files present (fail-fast)
    for category, data in inventory.items():
        for fname, fpath in data.get("critical", {}).items():
            checks[f"critical_file__{category}__{fname}"] = fpath is not None
            if fpath is None:
                fatal_errors.append(f"Missing critical file '{fname}' for '{category}'")

    # 3. Optional files present (warn only)
    for category, data in inventory.items():
        for fname, fpath in data.get("optional", {}).items():
            if fpath is None:
                warnings.append(f"Optional file '{fname}' missing for '{category}'")

    # 4. No duplicate dataset resolution
    checks["no_duplicate_datasets"] = len(duplicates) == 0
    for dup in duplicates:
        fatal_errors.append(f"Duplicate dataset resolution: {dup}")

    # 5. GPU availability (informational -- CPU deployment is valid, so warn only)
    checks["gpu_available"] = env_info["gpu_available"]
    if not env_info["gpu_available"]:
        warnings.append("No GPU detected; deployment will run on CPU.")

    # 6. PyTorch / CUDA build compatibility
    cuda_ok = (not env_info["gpu_available"]) or (env_info["cuda_version"] is not None)
    checks["cuda_compatibility"] = cuda_ok
    if not cuda_ok:
        fatal_errors.append("GPU present but installed PyTorch has no CUDA build.")

    # 7. Disk space (require >= 2GB free)
    disk = shutil.disk_usage("/")
    free_gb = disk.free / (1024 ** 3)
    checks["sufficient_disk_space"] = free_gb >= 2.0
    if free_gb < 2.0:
        fatal_errors.append(f"Insufficient disk space: {free_gb:.2f}GB free")

    # 8. Output directory writable
    writable = verify_output_writable(dirs["stage_root"])
    checks["output_directory_writable"] = writable
    if not writable:
        fatal_errors.append(f"Output directory not writable: {dirs['stage_root']}")

    # 9. Required output subfolders exist
    for name, path in dirs.items():
        checks[f"output_folder_exists__{name}"] = path.exists()

    for w in warnings:
        logger.warning("VALIDATION WARNING: %s", w)
    for e in fatal_errors:
        logger.error("VALIDATION FAILURE: %s", e)

    return {
        "checks": checks,
        "warnings": warnings,
        "fatal_errors": fatal_errors,
        "passed": len(fatal_errors) == 0,
    }


# ======================================================================
# JSON I/O
# ======================================================================

def save_json(path: Path, data: Dict[str, Any]) -> None:
    path.write_text(json.dumps(data, indent=2, default=str))


# ======================================================================
# MAIN STAGE ENTRY POINT
# ======================================================================

def run_stage01() -> Dict[str, Any]:
    """Run Sprint 05 / Stage 1: Deployment Environment & Artifact Discovery."""
    start_time = time.time()

    dirs = create_output_dirs()
    logger = build_logger(dirs["logs"])

    logger.info("START Sprint05-Stage01 Deployment Environment & Artifact Discovery")
    start_resources = log_resources(logger, "START")
    logger.info("OUTPUT directories ready under %s", dirs["stage_root"])

    set_seed(SEED)
    logger.info("SEED set to %d (deterministic=True)", SEED)

    # --- Environment ---------------------------------------------------
    env_info = get_environment_info(SEED)
    logger.info(
        "ENVIRONMENT python=%s torch=%s torchvision=%s cuda=%s gpu=%s device=%s",
        env_info["python_version"], env_info["torch_version"], env_info["torchvision_version"],
        env_info["cuda_version"], env_info["gpu_name"], env_info["device"],
    )

    # --- Discovery -------------------------------------------------------
    dataset_roots = discover_kaggle_datasets(logger)
    classification = classify_datasets(dataset_roots, logger)
    duplicates = check_duplicate_matches(classification)
    inventory = build_artifact_inventory(classification, logger)

    # --- Config ------------------------------------------------------------
    artifact_roots = {cat: inventory[cat]["root"] for cat in inventory}
    config = DeploymentConfig(
        device=env_info["device"],
        dtype="float32",
        artifact_roots=artifact_roots,
        output_root=str(OUTPUT_ROOT),
        logging=LoggingConfig(log_dir=str(dirs["logs"])),
        benchmark=BenchmarkConfig(),
        export=ExportConfig(),
        api=APIConfig(),
        runtime=RuntimeConfig(seed=SEED),
        tensorrt_enabled_future=False,
    )
    logger.info("CONFIG DeploymentConfig constructed (device=%s, dtype=%s)", config.device, config.dtype)

    # --- Validation --------------------------------------------------------
    validation = run_engineering_validation(env_info, inventory, duplicates, dirs, logger)

    if not validation["passed"]:
        log_resources(logger, "FINISH-FAILED")
        elapsed = time.time() - start_time
        logger.error("FINISH Sprint05-Stage01 status=FAILED elapsed=%.2fs", elapsed)
        raise RuntimeError(
            "Sprint05 Stage01 engineering validation FAILED (fail-fast):\n  - "
            + "\n  - ".join(validation["fatal_errors"])
        )

    end_resources = log_resources(logger, "FINISH")
    elapsed = time.time() - start_time

    runtime_summary = {
        "stage": STAGE_DIR_NAME,
        "elapsed_seconds": round(elapsed, 3),
        "start_resources": start_resources,
        "end_resources": end_resources,
        "seed": SEED,
    }

    # --- Persist artifacts ---------------------------------------------
    save_json(dirs["metadata"] / "environment_summary.json", env_info)
    save_json(dirs["metadata"] / "runtime_summary.json", runtime_summary)
    save_json(dirs["configs"] / "deployment_config.json", config.to_dict())
    save_json(dirs["artifacts"] / "artifact_discovery.json", {
        "mounted_datasets": {name: str(p) for name, p in dataset_roots.items()},
        "classification": classification,
        "inventory": inventory,
        "duplicates": duplicates,
    })
    save_json(dirs["metadata"] / "engineering_validation.json", validation)

    stage01_summary = {
        "stage": STAGE_DIR_NAME,
        "status": "OK",
        "elapsed_seconds": round(elapsed, 3),
        "device": env_info["device"],
        "gpu_name": env_info["gpu_name"],
        "cuda_version": env_info["cuda_version"],
        "datasets_discovered": list(dataset_roots.keys()),
        "categories_resolved": {c: inventory[c]["root"] is not None for c in inventory},
        "warnings_count": len(validation["warnings"]),
        "output_directory": str(dirs["stage_root"]),
    }
    save_json(dirs["stage_root"] / "stage01_summary.json", stage01_summary)

    logger.info("FINISH Sprint05-Stage01 status=OK elapsed=%.2fs", elapsed)

    # --- Console report --------------------------------------------------
    print("=" * 70)
    print("SPRINT 05 - STAGE 1")
    print("DEPLOYMENT ENVIRONMENT")
    print("=" * 70)
    print(f"Datasets discovered : {len(dataset_roots)} ({', '.join(dataset_roots.keys())})")
    print(f"Training artifacts  : {'FOUND' if inventory['sprint04_training']['root'] else 'MISSING'}")
    print(f"Evaluation artifacts: {'FOUND' if inventory['sprint04_evaluation']['root'] else 'MISSING'}")
    print(f"GPU                 : {env_info['gpu_name'] or 'N/A'}")
    print(f"CUDA                : {env_info['cuda_version'] or 'N/A'}")
    print(f"Engineering checks  : {sum(validation['checks'].values())}/{len(validation['checks'])} passed")
    print(f"Warnings            : {len(validation['warnings'])}")
    print(f"Output directory    : {dirs['stage_root']}")
    print("Stage 1 : OK")

    return {
        "config": config,
        "env_info": env_info,
        "inventory": inventory,
        "validation": validation,
        "summary": stage01_summary,
    }


if __name__ == "__main__":
    run_stage01()

2026-07-08 10:23:32 | INFO     | START Sprint05-Stage01 Deployment Environment & Artifact Discovery
2026-07-08 10:23:33 | INFO     | RESOURCES [START] cpu=28.4% ram=5.3%(1.2GB/31.4GB) disk=86.8% free=1065.0GB
2026-07-08 10:23:33 | INFO     | OUTPUT directories ready under /kaggle/working/sprint05_deployment/stage01_environment
2026-07-08 10:23:33 | INFO     | SEED set to 42 (deterministic=True)
2026-07-08 10:23:33 | INFO     | ENVIRONMENT python=3.12.13 torch=2.10.0+cu128 torchvision=0.25.0+cu128 cuda=12.8 gpu=Tesla T4 device=cuda
2026-07-08 10:23:33 | INFO     | DISCOVERY found 1 mounted dataset(s) under /kaggle/input
2026-07-08 10:23:33 | INFO     | DISCOVERY   - datasets
2026-07-08 10:23:34 | INFO     | DISCOVERY category='sprint03' -> dataset='datasets' root='/kaggle/input/datasets/anupsharma1730' (fingerprint score 3/3)
2026-07-08 10:23:34 | INFO     | DISCOVERY category='sprint04_training' -> dataset='datasets' root='/kaggle/input/datasets/anupsharma1730/visionserveai-sprint04-ar

In [2]:
"""
VisionServeAI - Sprint 05
Stage 2: Deployment Artifact Registry & Model Reconstruction
======================================================================
Single-responsibility stage: builds the centralized Deployment Artifact
Registry and reconstructs the production model from Stage 1 outputs.

STRICT SCOPE:
    - No inference. No ONNX/TorchScript export. No benchmarking.
    - No GradCAM. No API. No calibration. No metrics. No thresholding.
    - No visualization. No prediction.
    - Stage 1 is treated as FROZEN. Nothing is redesigned, nothing is
      rediscovered. Every path is consumed from Stage 1's own outputs.
======================================================================
"""

from __future__ import annotations

import csv
import hashlib
import json
import logging
import sys
import time
import types
from dataclasses import dataclass, field, asdict
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import psutil
import torch
import torch.nn as nn
import torchvision

# ======================================================================
# CONSTANTS
# ======================================================================

OUTPUT_ROOT = Path("/kaggle/working/sprint05_deployment")
STAGE1_DIR_NAME = "stage01_environment"
STAGE2_DIR_NAME = "stage02_registry"

STAGE1_DIR = OUTPUT_ROOT / STAGE1_DIR_NAME

# Stage 1 output locations -- CONSUMED, never rediscovered.
STAGE1_FILES: Dict[str, Path] = {
    "environment_summary": STAGE1_DIR / "metadata" / "environment_summary.json",
    "runtime_summary": STAGE1_DIR / "metadata" / "runtime_summary.json",
    "deployment_config": STAGE1_DIR / "configs" / "deployment_config.json",
    "artifact_discovery": STAGE1_DIR / "artifacts" / "artifact_discovery.json",
    "engineering_validation": STAGE1_DIR / "metadata" / "engineering_validation.json",
    "stage01_summary": STAGE1_DIR / "stage01_summary.json",
}

# Category names as produced by Stage 1 discovery (frozen contract).
CATEGORY_SPRINT03 = "sprint03"
CATEGORY_TRAINING = "sprint04_training"
CATEGORY_EVALUATION = "sprint04_evaluation"
CATEGORY_NIH = "nih_chest_xray"

# Loose structural contracts used ONLY for informational schema validation.
# "Present" = ANY of these keys found -- we do not know the exact upstream
# schema, so we validate presence-of-plausible-keys, not exact shape.
EXPECTED_JSON_KEYS: Dict[str, List[str]] = {
    "training_summary.json": ["backbone", "architecture", "model_name", "arch", "model_architecture"],
    "disease_registry.json": ["classes", "class_names", "diseases", "labels"],
    "evaluation_summary.json": ["classes", "class_names", "diseases", "labels", "metrics"],
    "optimal_thresholds.json": ["thresholds", "classes", "class_names"],
}


# ---- Known backbone factories for EXACT architecture reconstruction ----
# Extend only when a new backbone is explicitly named in training_summary.
# Never guess an architecture that isn't named there.

class WrappedClassifier(nn.Module):
    """Matches the Sprint04 training convention observed in the checkpoint:
    self.backbone = <torchvision base model>, with the base model's native
    classification head replaced by nn.Sequential(Dropout(p), Linear(...)).
    This is inferred directly from checkpoint key names (backbone.*, head
    index 1), not assumed -- do not change without re-checking checkpoint keys."""

    def __init__(self, base_model: nn.Module, head_attr: str, in_features: int,
                 num_classes: int, dropout: float):
        super().__init__()
        self.backbone = base_model
        head = nn.Sequential(nn.Dropout(p=dropout), nn.Linear(in_features, num_classes))
        setattr(self.backbone, head_attr, head)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.backbone(x)


def _base_densenet121(_num_classes: int) -> Tuple[nn.Module, str, int]:
    m = torchvision.models.densenet121(weights=None)
    return m, "classifier", m.classifier.in_features


def _base_resnet50(_num_classes: int) -> Tuple[nn.Module, str, int]:
    m = torchvision.models.resnet50(weights=None)
    return m, "fc", m.fc.in_features


def _base_resnet18(_num_classes: int) -> Tuple[nn.Module, str, int]:
    m = torchvision.models.resnet18(weights=None)
    return m, "fc", m.fc.in_features


def _base_efficientnet_b0(_num_classes: int) -> Tuple[nn.Module, str, int]:
    m = torchvision.models.efficientnet_b0(weights=None)
    return m, "classifier", m.classifier[-1].in_features


def _base_vgg16(_num_classes: int) -> Tuple[nn.Module, str, int]:
    m = torchvision.models.vgg16(weights=None)
    return m, "classifier", m.classifier[-1].in_features


BASE_MODEL_BUILDERS = {
    "densenet121": _base_densenet121,
    "resnet50": _base_resnet50,
    "resnet18": _base_resnet18,
    "efficientnet_b0": _base_efficientnet_b0,
    "vgg16": _base_vgg16,
}


def build_model(backbone_name: str, num_classes: int, dropout: float) -> Tuple[nn.Module, str]:
    """Returns (wrapped_model, full_dotted_path_to_classifier_head)."""
    base_model, head_attr, in_features = BASE_MODEL_BUILDERS[backbone_name](num_classes)
    model = WrappedClassifier(base_model, head_attr, in_features, num_classes, dropout)
    return model, f"backbone.{head_attr}"


def _get_nested_attr(obj: Any, dotted_path: str) -> Any:
    current = obj
    for part in dotted_path.split("."):
        current = getattr(current, part)
    return current

# ======================================================================
# LOGGING (same format/pattern as Stage 1 -- self-contained for robustness
# across kernel restarts; logger name distinguishes the stage)
# ======================================================================

def build_logger(log_dir: Path) -> logging.Logger:
    log_dir.mkdir(parents=True, exist_ok=True)
    logger = logging.getLogger("sprint05.stage02")
    logger.setLevel(logging.INFO)
    logger.handlers.clear()
    logger.propagate = False

    fmt = logging.Formatter(
        fmt="%(asctime)s | %(levelname)-8s | %(message)s",
        datefmt="%Y-%m-%d %H:%M:%S",
    )

    file_handler = logging.FileHandler(log_dir / "stage02_registry.log", mode="w")
    file_handler.setFormatter(fmt)
    logger.addHandler(file_handler)

    stream_handler = logging.StreamHandler(sys.stdout)
    stream_handler.setFormatter(fmt)
    logger.addHandler(stream_handler)

    return logger


def get_resource_usage() -> Dict[str, Any]:
    vm = psutil.virtual_memory()
    usage: Dict[str, Any] = {
        "cpu_percent": psutil.cpu_percent(interval=0.2),
        "ram_percent": vm.percent,
        "ram_used_gb": round(vm.used / (1024 ** 3), 2),
        "ram_total_gb": round(vm.total / (1024 ** 3), 2),
    }
    if torch.cuda.is_available():
        usage["gpu_memory_allocated_gb"] = round(torch.cuda.memory_allocated() / (1024 ** 3), 3)
        usage["gpu_memory_reserved_gb"] = round(torch.cuda.memory_reserved() / (1024 ** 3), 3)
    return usage


def log_resources(logger: logging.Logger, tag: str) -> Dict[str, Any]:
    usage = get_resource_usage()
    logger.info(
        "RESOURCES [%s] cpu=%.1f%% ram=%.1f%%(%.1fGB/%.1fGB)",
        tag, usage["cpu_percent"], usage["ram_percent"], usage["ram_used_gb"], usage["ram_total_gb"],
    )
    return usage


def save_json(path: Path, data: Dict[str, Any]) -> None:
    path.write_text(json.dumps(data, indent=2, default=str))


def load_json(path: Path) -> Dict[str, Any]:
    return json.loads(path.read_text())


def sha256_of_file(path: Path, chunk_size: int = 1 << 20) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            h.update(chunk)
    return h.hexdigest()


def _torch_load(path: Path, device: torch.device) -> Any:
    """torch.load wrapper tolerant of weights_only kwarg availability across
    torch versions. This loads our OWN previously-produced checkpoint --
    not an untrusted third-party file."""
    try:
        return torch.load(path, map_location=device, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=device)


# ======================================================================
# ARTIFACT REGISTRY
# ======================================================================

@dataclass
class ArtifactRecord:
    artifact_id: str
    category: str
    filename: str
    path: Optional[str]
    is_critical: bool
    exists: bool = False
    size_bytes: Optional[int] = None
    sha256: Optional[str] = None
    file_type: Optional[str] = None          # json | csv | torch | unknown
    json_readable: Optional[bool] = None
    csv_readable: Optional[bool] = None
    torch_readable: Optional[bool] = None
    schema_valid: Optional[bool] = None
    expected_keys_present: Optional[bool] = None
    schema_warnings: List[str] = field(default_factory=list)
    modified_time: Optional[float] = None
    duplicate_of: Optional[str] = None
    errors: List[str] = field(default_factory=list)


@dataclass
class ArtifactRegistry:
    records: Dict[str, ArtifactRecord]
    total_artifacts: int
    critical_missing: List[str]
    duplicate_groups: List[List[str]]

    def to_dict(self) -> Dict[str, Any]:
        return {
            "total_artifacts": self.total_artifacts,
            "critical_missing": self.critical_missing,
            "duplicate_groups": self.duplicate_groups,
            "records": {k: asdict(v) for k, v in self.records.items()},
        }


def _classify_file_type(filename: str) -> str:
    suffix = Path(filename).suffix.lower()
    if suffix == ".json":
        return "json"
    if suffix == ".csv":
        return "csv"
    if suffix in (".pt", ".pth"):
        return "torch"
    return "unknown"


def validate_json_schema(filename: str, data: Any) -> Tuple[Optional[bool], Optional[bool], List[str], List[str]]:
    """Structural check only.
    Returns (schema_valid, expected_keys_present, hard_errors, soft_warnings).
    hard_errors -> real corruption (bad top-level type). Fails the artifact.
    soft_warnings -> our assumed key contract didn't match (informational only,
    since upstream stages are free to nest fields however they like)."""
    hard_errors: List[str] = []
    soft_warnings: List[str] = []

    if not isinstance(data, (dict, list)):
        hard_errors.append(f"'{filename}' does not contain a JSON object/array at the top level.")
        return False, False, hard_errors, soft_warnings

    expected = EXPECTED_JSON_KEYS.get(filename)
    if expected is None:
        return None, None, hard_errors, soft_warnings

    if isinstance(data, dict):
        present = any(k in data for k in expected)
        if not present:
            # Check one level of nesting (e.g. keys living under "model", "metrics", etc.)
            present = any(
                isinstance(v, dict) and any(k in v for k in expected)
                for v in data.values()
            )
        if not present:
            soft_warnings.append(
                f"'{filename}' has none of the plausible keys {expected} at top level or one level "
                f"of nesting. Informational only -- verify manually if this file is meant to carry class info."
            )
        return True, present, hard_errors, soft_warnings

    return True, None, hard_errors, soft_warnings


def validate_artifact_file(
    artifact_id: str,
    category: str,
    filename: str,
    path_str: Optional[str],
    is_critical: bool,
    device: torch.device,
    logger: logging.Logger,
) -> ArtifactRecord:
    record = ArtifactRecord(
        artifact_id=artifact_id, category=category, filename=filename,
        path=path_str, is_critical=is_critical,
    )

    if path_str is None:
        if is_critical:
            record.errors.append("Critical artifact missing (not discovered by Stage 1).")
        return record

    path = Path(path_str)
    record.exists = path.exists() and path.is_file()
    if not record.exists:
        record.errors.append(f"Path recorded by Stage 1 no longer exists on disk: {path_str}")
        return record

    stat = path.stat()
    record.size_bytes = stat.st_size
    record.modified_time = stat.st_mtime
    record.file_type = _classify_file_type(filename)

    if record.size_bytes == 0:
        record.errors.append("File is empty (0 bytes).")

    try:
        record.sha256 = sha256_of_file(path)
    except OSError as exc:
        record.errors.append(f"Failed to hash file: {exc}")

    if record.file_type == "json":
        try:
            data = json.loads(path.read_text())
            record.json_readable = True
            schema_valid, keys_present, hard_errors, soft_warnings = validate_json_schema(filename, data)
            record.schema_valid = schema_valid
            record.expected_keys_present = keys_present
            record.errors.extend(hard_errors)            # only real corruption goes here
            record.schema_warnings.extend(soft_warnings)  # cosmetic, never blocks the pipeline
        except (json.JSONDecodeError, UnicodeDecodeError, OSError) as exc:
            record.json_readable = False
            record.errors.append(f"JSON parse failure: {exc}")

    elif record.file_type == "csv":
        try:
            with path.open("r", newline="") as f:
                reader = csv.reader(f)
                header = next(reader, None)
            record.csv_readable = header is not None
            if header is None:
                record.errors.append("CSV file has no header/rows.")
        except (OSError, csv.Error) as exc:
            record.csv_readable = False
            record.errors.append(f"CSV parse failure: {exc}")

    elif record.file_type == "torch":
        try:
            _ = _torch_load(path, device)
            record.torch_readable = True
        except Exception as exc:  # noqa: BLE001 -- must surface, never silently pass
            record.torch_readable = False
            record.errors.append(f"Torch checkpoint unreadable: {exc}")

    return record


def detect_duplicate_artifacts(records: Dict[str, ArtifactRecord]) -> List[List[str]]:
    by_hash: Dict[str, List[str]] = {}
    for artifact_id, rec in records.items():
        if rec.sha256:
            by_hash.setdefault(rec.sha256, []).append(artifact_id)
    groups = [ids for ids in by_hash.values() if len(ids) > 1]
    for ids in groups:
        for artifact_id in ids:
            records[artifact_id].duplicate_of = ",".join(i for i in ids if i != artifact_id)
    return groups


def build_artifact_registry(
    inventory: Dict[str, Any], device: torch.device, logger: logging.Logger,
) -> ArtifactRegistry:
    records: Dict[str, ArtifactRecord] = {}

    for category, data in inventory.items():
        for group_name in ("critical", "optional"):
            group = data.get(group_name, {}) or {}
            is_critical = group_name == "critical"
            for filename, path_str in group.items():
                artifact_id = f"{category}::{filename}"
                record = validate_artifact_file(
                    artifact_id, category, filename, path_str, is_critical, device, logger,
                )
                records[artifact_id] = record
                status = "OK" if record.exists and not record.errors else "ISSUE"
                logger.info(
                    "ARTIFACT[%s] id=%s critical=%s exists=%s type=%s status=%s",
                    category, artifact_id, is_critical, record.exists, record.file_type, status,
                )
                for err in record.errors:
                    logger.warning("ARTIFACT[%s] id=%s issue: %s", category, artifact_id, err)
                for warn in record.schema_warnings:
                    logger.warning("ARTIFACT[%s] id=%s schema note: %s", category, artifact_id, warn)

    duplicate_groups = detect_duplicate_artifacts(records)
    for group in duplicate_groups:
        logger.warning("DUPLICATE ARTIFACTS detected (identical SHA256): %s", group)

    critical_missing = [
        aid for aid, rec in records.items() if rec.is_critical and (not rec.exists or rec.errors)
    ]

    return ArtifactRegistry(
        records=records,
        total_artifacts=len(records),
        critical_missing=critical_missing,
        duplicate_groups=duplicate_groups,
    )


def get_artifact_path(inventory: Dict[str, Any], category: str, filename: str) -> Optional[str]:
    data = inventory.get(category, {})
    for group in ("critical", "optional"):
        path = data.get(group, {}).get(filename)
        if path:
            return path
    return None


# ======================================================================
# MODEL RECONSTRUCTION
# ======================================================================

@dataclass
class ModelValidationChecks:
    architecture_instantiated: bool = False
    checkpoint_loaded: bool = False
    state_dict_strict_match: bool = False
    parameter_count_expected: Optional[bool] = None
    output_dimension_expected: Optional[bool] = None
    classifier_dimension_expected: Optional[bool] = None
    all_params_correct_device: bool = False
    all_params_correct_dtype: bool = False
    eval_mode: bool = False
    grad_disabled: bool = False
    errors: List[str] = field(default_factory=list)


@dataclass
class ModelRegistry:
    backbone: str
    num_classes: int
    device: str
    dtype: str
    total_parameters: int
    trainable_parameters: int
    checkpoint_path: str
    checkpoint_sha256: str
    model_signature: Dict[str, Any]
    validation: ModelValidationChecks

    def to_dict(self) -> Dict[str, Any]:
        return asdict(self)

def _search_keys(d: Dict[str, Any], keys: List[str]) -> Tuple[Optional[str], Optional[Any]]:
    """Look for the first matching key at top level, then one level of nesting.
    Returns (key_found, value) or (None, None)."""
    for key in keys:
        val = d.get(key)
        if val not in (None, ""):
            return key, val
    for v in d.values():
        if isinstance(v, dict):
            for key in keys:
                val = v.get(key)
                if val not in (None, ""):
                    return key, val
    return None, None


def resolve_backbone_name(training_summary: Dict[str, Any], logger: logging.Logger) -> str:
    candidate_keys = ["backbone", "architecture", "model_name", "arch", "model_architecture"]
    found_key, value = _search_keys(training_summary, candidate_keys)
    if value is None:
        raise RuntimeError(
            f"training_summary.json does not contain any recognized backbone key {candidate_keys} "
            f"(checked top level and one level of nesting)."
        )
    name = str(value).strip().lower()
    logger.info("MODEL backbone resolved via key '%s' = '%s'", found_key, name)

    if name not in BASE_MODEL_BUILDERS:
        # model_name can be a descriptive string like "densenet121-chestxray14";
        # try to salvage a known backbone id as a substring before giving up.
        for known in BASE_MODEL_BUILDERS:
            if known in name:
                logger.warning(
                    "MODEL backbone value '%s' not an exact match; matched known backbone '%s' as substring.",
                    name, known,
                )
                return known
        raise RuntimeError(
            f"Backbone '{name}' from training_summary.json is not a supported/known architecture. "
            f"Supported: {list(BASE_MODEL_BUILDERS.keys())}"
        )
    return name


def resolve_num_classes(
    training_summary: Dict[str, Any], disease_registry: Dict[str, Any], logger: logging.Logger,
) -> int:
    candidate_keys = ["num_classes", "n_classes", "output_dim", "num_labels"]
    found_key, value = _search_keys(training_summary, candidate_keys)
    if value is not None:
        n = int(value)
        logger.info("MODEL num_classes resolved via training_summary key '%s' = %d", found_key, n)
        return n
    for key in ("classes", "class_names", "diseases", "labels"):
        if isinstance(disease_registry.get(key), list):
            n = len(disease_registry[key])
            logger.info("MODEL num_classes resolved from disease_registry['%s'] length = %d", key, n)
            return n
    raise RuntimeError("Unable to resolve num_classes from training_summary.json or disease_registry.json.")

def resolve_dropout(training_summary: Dict[str, Any], logger: logging.Logger) -> float:
    candidate_keys = ["dropout", "dropout_rate", "dropout_p"]
    found_key, value = _search_keys(training_summary, candidate_keys)
    if value is not None:
        p = float(value)
        logger.info("MODEL dropout resolved via training_summary key '%s' = %.3f", found_key, p)
        return p
    logger.warning(
        "MODEL dropout not found in training_summary.json; defaulting to 0.0. "
        "This has no effect on checkpoint compatibility since Dropout has no learnable "
        "parameters and the model is run in eval() mode, but is logged for traceability."
    )
    return 0.0

def extract_state_dict(checkpoint: Any) -> Dict[str, torch.Tensor]:
    """Unwrap known container conventions only -- no silent key surgery."""
    if isinstance(checkpoint, nn.Module):
        return checkpoint.state_dict()
    if isinstance(checkpoint, dict):
        for key in ("state_dict", "model_state_dict", "model", "net"):
            if key in checkpoint and isinstance(checkpoint[key], dict):
                return checkpoint[key]
        if checkpoint and all(isinstance(v, torch.Tensor) for v in checkpoint.values()):
            return checkpoint
    raise RuntimeError("Unable to locate a valid state_dict inside the checkpoint object.")


def _final_out_features(module: nn.Module) -> int:
    if isinstance(module, nn.Linear):
        return module.out_features
    if isinstance(module, nn.Sequential):
        for layer in reversed(module):
            if isinstance(layer, nn.Linear):
                return layer.out_features
    raise RuntimeError(f"Unable to determine output features from classifier module: {module}")


def reconstruct_model(
    training_summary: Dict[str, Any],
    disease_registry: Dict[str, Any],
    checkpoint_path: Path,
    device: torch.device,
    logger: logging.Logger,
) -> Tuple[nn.Module, ModelRegistry]:
    checks = ModelValidationChecks()

    backbone_name = resolve_backbone_name(training_summary, logger)
    num_classes = resolve_num_classes(training_summary, disease_registry, logger)
    dropout = resolve_dropout(training_summary, logger)

    model, classifier_path = build_model(backbone_name, num_classes, dropout)
    checks.architecture_instantiated = True
    logger.info(
        "MODEL architecture instantiated: backbone=%s num_classes=%d dropout=%.3f classifier_path=%s",
        backbone_name, num_classes, dropout, classifier_path,
    )
    checkpoint_sha256 = sha256_of_file(checkpoint_path)
    logger.info("MODEL checkpoint sha256=%s path=%s", checkpoint_sha256, checkpoint_path)

    try:
        checkpoint = _torch_load(checkpoint_path, device)
    except Exception as exc:
        raise RuntimeError(f"Failed to torch.load checkpoint at {checkpoint_path}: {exc}") from exc

    state_dict = extract_state_dict(checkpoint)

    missing_keys, unexpected_keys = model.load_state_dict(state_dict, strict=False)
    if missing_keys or unexpected_keys:
        raise RuntimeError(
            "Checkpoint is NOT compatible with the reconstructed architecture "
            f"(backbone='{backbone_name}'). missing_keys={missing_keys} "
            f"unexpected_keys={unexpected_keys}. No silent fixes permitted."
        )
    checks.checkpoint_loaded = True
    checks.state_dict_strict_match = True
    logger.info("MODEL checkpoint loaded with STRICT key match (0 missing, 0 unexpected).")

    model.to(device)
    model.eval()
    for p in model.parameters():
        p.requires_grad_(False)
    checks.eval_mode = not model.training
    checks.grad_disabled = all(not p.requires_grad for p in model.parameters())

    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

    expected_total = training_summary.get("total_parameters") or training_summary.get("num_parameters")
    if expected_total:
        checks.parameter_count_expected = int(expected_total) == total_params
        if not checks.parameter_count_expected:
            checks.errors.append(
                f"Parameter count mismatch: reconstructed={total_params} expected={expected_total}"
            )
    else:
        logger.warning("MODEL training_summary.json has no recorded parameter count; skipping exact-match check.")

    # Output/classifier dimension check via attribute inspection only -- NOT a forward pass.
    # NOTE: classifier_path (e.g. "backbone.classifier") already fully identifies the
    # head submodule -- no separate CLASSIFIER_ATTR lookup is needed or exists anymore.
    out_features = _final_out_features(_get_nested_attr(model, classifier_path))
    checks.classifier_dimension_expected = out_features == num_classes
    checks.output_dimension_expected = out_features == num_classes
    if not checks.classifier_dimension_expected:
        checks.errors.append(
            f"Classifier output dimension {out_features} != expected num_classes {num_classes}"
        )

    devices = {p.device.type for p in model.parameters()}
    checks.all_params_correct_device = devices == {device.type}
    if not checks.all_params_correct_device:
        checks.errors.append(f"Parameters found on unexpected devices: {devices}")

    dtypes = {p.dtype for p in model.parameters()}
    checks.all_params_correct_dtype = dtypes == {torch.float32}
    if not checks.all_params_correct_dtype:
        checks.errors.append(f"Parameters found with unexpected dtypes: {dtypes}")

    if checks.errors:
        raise RuntimeError("Model reconstruction validation FAILED: " + "; ".join(checks.errors))

    model_signature = {
        "backbone": backbone_name,
        "num_classes": num_classes,
        "dropout": dropout,
        "classifier_path": classifier_path,
        "output_features": out_features,
        "total_parameters": total_params,
        "trainable_parameters": trainable_params,
        "device": str(device),
        "dtype": "float32",
        "state_dict_key_count": len(state_dict),
        "state_dict_key_hash": hashlib.sha256("|".join(sorted(state_dict.keys())).encode()).hexdigest(),
    }

    registry = ModelRegistry(
        backbone=backbone_name,
        num_classes=num_classes,
        device=str(device),
        dtype="float32",
        total_parameters=total_params,
        trainable_parameters=trainable_params,
        checkpoint_path=str(checkpoint_path),
        checkpoint_sha256=checkpoint_sha256,
        model_signature=model_signature,
        validation=checks,
    )
    return model, registry


# ======================================================================
# THRESHOLD REGISTRY (load + validate only -- NEVER compute)
# ======================================================================

@dataclass
class ThresholdRegistry:
    source_path: Optional[str]
    class_count: int
    class_names: List[str]
    thresholds: Dict[str, float]
    threshold_metadata: Dict[str, Dict[str, float]] = field(default_factory=dict)
    validation_errors: List[str] = field(default_factory=list)

    def to_dict(self) -> Dict[str, Any]:
        return asdict(self)


def load_threshold_registry(
    optimal_thresholds_path: Optional[str], expected_class_names: List[str], logger: logging.Logger,
) -> ThresholdRegistry:
    if optimal_thresholds_path is None:
        logger.warning("THRESHOLDS optimal_thresholds.json not discovered by Stage 1; registry will be empty.")
        return ThresholdRegistry(
            source_path=None, class_count=0, class_names=[], thresholds={}, threshold_metadata={},
            validation_errors=["optimal_thresholds.json not available from Stage 1 discovery."],
        )

    raw = load_json(Path(optimal_thresholds_path))

    threshold_metadata: Dict[str, Dict[str, float]] = {}

    if isinstance(raw, list):
        # Schema Version 3: one record per class carrying multiple optimal-threshold
        # variants (f1 / balanced_accuracy / youden_j). Deployment threshold is
        # f1_optimal_threshold; every remaining calibration value is preserved in
        # threshold_metadata rather than discarded.
        REQUIRED_V3_KEYS = (
            "class_name",
            "f1_optimal_threshold",
            "f1_optimal_value",
            "balanced_accuracy_optimal_threshold",
            "balanced_accuracy_optimal_value",
            "youden_j_optimal_threshold",
            "youden_j_optimal_value",
        )
        thresholds: Dict[str, float] = {}
        for i, record in enumerate(raw):
            if not isinstance(record, dict):
                raise RuntimeError(
                    f"Unrecognized optimal_thresholds.json schema: record at index {i} is not an object."
                )
            missing_keys = [k for k in REQUIRED_V3_KEYS if k not in record]
            if missing_keys:
                raise RuntimeError(
                    f"optimal_thresholds.json Version 3 record at index {i} is missing required "
                    f"keys {missing_keys}: record={record}"
                )
            name = record["class_name"]
            if name in thresholds:
                raise RuntimeError(
                    f"Duplicate class_name '{name}' found in optimal_thresholds.json Version 3 records."
                )
            thresholds[name] = float(record["f1_optimal_threshold"])
            threshold_metadata[name] = {
                "f1_optimal_threshold": float(record["f1_optimal_threshold"]),
                "f1_optimal_value": float(record["f1_optimal_value"]),
                "balanced_accuracy_optimal_threshold": float(record["balanced_accuracy_optimal_threshold"]),
                "balanced_accuracy_optimal_value": float(record["balanced_accuracy_optimal_value"]),
                "youden_j_optimal_threshold": float(record["youden_j_optimal_threshold"]),
                "youden_j_optimal_value": float(record["youden_j_optimal_value"]),
            }
    elif isinstance(raw, dict) and raw and all(isinstance(v, (int, float)) for v in raw.values()):
        thresholds = {k: float(v) for k, v in raw.items()}
    elif isinstance(raw, dict) and "thresholds" in raw:
        names = raw.get("classes") or raw.get("class_names") or expected_class_names
        thresholds = {name: float(v) for name, v in zip(names, raw["thresholds"])}
    else:
        raise RuntimeError(f"Unrecognized optimal_thresholds.json schema: keys={list(raw)[:10]}")

    errors: List[str] = []
    class_names = list(thresholds.keys())

    if len(class_names) != len(set(class_names)):
        errors.append("Duplicate class names found in threshold registry.")

    if expected_class_names:
        missing = set(expected_class_names) - set(class_names)
        extra = set(class_names) - set(expected_class_names)
        if missing:
            errors.append(f"Thresholds missing classes: {sorted(missing)}")
        if extra:
            errors.append(f"Thresholds contain unexpected classes: {sorted(extra)}")

    for name, value in thresholds.items():
        if not (0.0 <= value <= 1.0):
            errors.append(f"Threshold for '{name}' out of range [0,1]: {value}")

    # Version 3 carries additional threshold-valued fields that must independently
    # satisfy the same [0,1] contract as the deployment threshold.
    for name, meta in threshold_metadata.items():
        for key in ("balanced_accuracy_optimal_threshold", "youden_j_optimal_threshold"):
            value = meta[key]
            if not (0.0 <= value <= 1.0):
                errors.append(f"'{key}' for '{name}' out of range [0,1]: {value}")

    for e in errors:
        logger.warning("THRESHOLD VALIDATION: %s", e)

    return ThresholdRegistry(
        source_path=optimal_thresholds_path,
        class_count=len(class_names),
        class_names=class_names,
        thresholds=thresholds,
        threshold_metadata=threshold_metadata,
        validation_errors=errors,
    )


# ======================================================================
# METADATA REGISTRY (immutable)
# ======================================================================

def _freeze(data: Any) -> Any:
    if isinstance(data, dict):
        return types.MappingProxyType({k: _freeze(v) for k, v in data.items()})
    if isinstance(data, list):
        return tuple(_freeze(v) for v in data)
    return data


def _unfreeze(data: Any) -> Any:
    if isinstance(data, types.MappingProxyType):
        return {k: _unfreeze(v) for k, v in data.items()}
    if isinstance(data, tuple):
        return [_unfreeze(v) for v in data]
    return data


@dataclass(frozen=True)
class MetadataRegistry:
    disease_metadata: Any
    training_metadata: Any
    evaluation_metadata: Any
    deployment_metadata: Any
    publication_metadata: Any

    def to_dict(self) -> Dict[str, Any]:
        return {
            "disease_metadata": _unfreeze(self.disease_metadata),
            "training_metadata": _unfreeze(self.training_metadata),
            "evaluation_metadata": _unfreeze(self.evaluation_metadata),
            "deployment_metadata": _unfreeze(self.deployment_metadata),
            "publication_metadata": _unfreeze(self.publication_metadata),
        }


def build_metadata_registry(inventory: Dict[str, Any], logger: logging.Logger) -> MetadataRegistry:
    def _safe_load(category: str, filename: str) -> Dict[str, Any]:
        path_str = get_artifact_path(inventory, category, filename)
        if path_str is None:
            logger.warning("METADATA '%s/%s' not available from Stage 1 discovery.", category, filename)
            return {}
        return load_json(Path(path_str))

    return MetadataRegistry(
        disease_metadata=_freeze(_safe_load(CATEGORY_SPRINT03, "disease_registry.json")),
        training_metadata=_freeze(_safe_load(CATEGORY_TRAINING, "training_summary.json")),
        evaluation_metadata=_freeze(_safe_load(CATEGORY_EVALUATION, "evaluation_summary.json")),
        deployment_metadata=_freeze(_safe_load(CATEGORY_EVALUATION, "deployment_readiness.json")),
        publication_metadata=_freeze(_safe_load(CATEGORY_EVALUATION, "publication_readiness.json")),
    )


# ======================================================================
# CONFIG REGISTRY (immutable, pass-through of Stage 1's frozen config)
# ======================================================================

@dataclass(frozen=True)
class ConfigRegistry:
    stage1_deployment_config: Any
    device: str
    dtype: str
    output_root: str

    def to_dict(self) -> Dict[str, Any]:
        return {
            "stage1_deployment_config": _unfreeze(self.stage1_deployment_config),
            "device": self.device,
            "dtype": self.dtype,
            "output_root": self.output_root,
        }


def build_config_registry(deployment_config: Dict[str, Any]) -> ConfigRegistry:
    return ConfigRegistry(
        stage1_deployment_config=_freeze(deployment_config),
        device=deployment_config["device"],
        dtype=deployment_config["dtype"],
        output_root=deployment_config["output_root"],
    )


# ======================================================================
# DEPLOYMENT REGISTRY (single source of truth for Stage 3+)
# ======================================================================

@dataclass
class DeploymentRegistry:
    artifact_registry: ArtifactRegistry
    model_registry: ModelRegistry
    metadata_registry: MetadataRegistry
    config_registry: ConfigRegistry
    threshold_registry: ThresholdRegistry

    def to_dict(self) -> Dict[str, Any]:
        return {
            "artifact_registry": self.artifact_registry.to_dict(),
            "model_registry": self.model_registry.to_dict(),
            "metadata_registry": self.metadata_registry.to_dict(),
            "config_registry": self.config_registry.to_dict(),
            "threshold_registry": self.threshold_registry.to_dict(),
        }


# ======================================================================
# ENGINEERING VALIDATION
# ======================================================================

def run_stage02_engineering_validation(
    stage1_validation: Dict[str, Any],
    artifact_registry: ArtifactRegistry,
    model_registry: ModelRegistry,
    threshold_registry: ThresholdRegistry,
    disease_metadata: Dict[str, Any],
    evaluation_metadata: Dict[str, Any],
    env_info: Dict[str, Any],
    device: torch.device,
    logger: logging.Logger,
) -> Dict[str, Any]:
    checks: Dict[str, bool] = {}
    warnings: List[str] = []
    fatal_errors: List[str] = []

    checks["stage1_outputs_readable"] = True
    checks["stage1_validation_passed"] = bool(stage1_validation.get("passed"))
    if not checks["stage1_validation_passed"]:
        fatal_errors.append("Stage 1 engineering_validation.json reports passed=False.")

    checks["model_reconstructed"] = model_registry.validation.architecture_instantiated
    checks["weights_loaded"] = (
        model_registry.validation.checkpoint_loaded and model_registry.validation.state_dict_strict_match
    )
    if not checks["weights_loaded"]:
        fatal_errors.append("Model weights were not loaded with a strict state_dict match.")

    disease_classes = None
    for key in ("classes", "class_names", "diseases", "labels"):
        if isinstance(disease_metadata.get(key), list):
            disease_classes = disease_metadata[key]
            break

    checks["disease_class_names_present"] = disease_classes is not None
    if disease_classes is not None:
        checks["no_duplicate_class_names"] = len(disease_classes) == len(set(disease_classes))
        if not checks["no_duplicate_class_names"]:
            fatal_errors.append("Duplicate class names found in disease_registry.json.")

        checks["class_count_consistent_with_model"] = len(disease_classes) == model_registry.num_classes
        if not checks["class_count_consistent_with_model"]:
            fatal_errors.append(
                f"disease_registry class count ({len(disease_classes)}) != model num_classes ({model_registry.num_classes})"
            )
    else:
        warnings.append("Could not locate an explicit class-name list inside disease_registry.json.")

    checks["threshold_count_consistent"] = (
        threshold_registry.class_count == 0 or threshold_registry.class_count == model_registry.num_classes
    )
    if not checks["threshold_count_consistent"]:
        fatal_errors.append(
            f"Threshold class_count ({threshold_registry.class_count}) != model num_classes ({model_registry.num_classes})"
        )
    checks["threshold_registry_valid"] = len(threshold_registry.validation_errors) == 0

    checks["registry_count_consistent"] = artifact_registry.total_artifacts > 0
    checks["no_duplicate_artifact_ids"] = len(artifact_registry.records) == len(set(artifact_registry.records.keys()))
    checks["no_critical_artifacts_missing"] = len(artifact_registry.critical_missing) == 0
    if artifact_registry.critical_missing:
        fatal_errors.append(f"Critical artifacts missing/invalid: {artifact_registry.critical_missing}")

    eval_classes = None
    for key in ("classes", "class_names", "diseases", "labels"):
        if isinstance(evaluation_metadata.get(key), list):
            eval_classes = evaluation_metadata[key]
            break
    checks["evaluation_summary_compatible"] = eval_classes is None or len(eval_classes) == model_registry.num_classes
    if eval_classes is not None and not checks["evaluation_summary_compatible"]:
        fatal_errors.append("evaluation_summary.json class count does not match reconstructed model.")

    checks["checkpoint_compatible"] = model_registry.validation.state_dict_strict_match
    checks["parameter_count_correct"] = model_registry.validation.parameter_count_expected is not False
    checks["model_hash_recorded"] = bool(model_registry.checkpoint_sha256)

    checks["torch_version_compatible"] = env_info.get("torch_version") == torch.__version__
    if not checks["torch_version_compatible"]:
        warnings.append(
            f"Torch version differs from Stage 1 snapshot (stage1={env_info.get('torch_version')}, "
            f"current={torch.__version__})."
        )

    checks["gpu_compatibility"] = (device.type != "cuda") or torch.cuda.is_available()
    checks["device_compatibility"] = str(device) == model_registry.device
    checks["checkpoint_integrity"] = model_registry.validation.state_dict_strict_match and not model_registry.validation.errors

    for w in warnings:
        logger.warning("VALIDATION WARNING: %s", w)
    for e in fatal_errors:
        logger.error("VALIDATION FAILURE: %s", e)

    return {"checks": checks, "warnings": warnings, "fatal_errors": fatal_errors, "passed": len(fatal_errors) == 0}


# ======================================================================
# MAIN STAGE ENTRY POINT
# ======================================================================

def run_stage02() -> Dict[str, Any]:
    start_time = time.time()

    stage_root = OUTPUT_ROOT / STAGE2_DIR_NAME
    dirs = {
        "stage_root": stage_root,
        "logs": stage_root / "logs",
        "registry": stage_root / "registry",
        "model": stage_root / "model",
        "metadata": stage_root / "metadata",
    }
    for p in dirs.values():
        p.mkdir(parents=True, exist_ok=True)

    logger = build_logger(dirs["logs"])
    logger.info("START Sprint05-Stage02 Deployment Artifact Registry & Model Reconstruction")
    log_resources(logger, "START")

    # ---- Load Stage 1 outputs (frozen, no rediscovery) ------------------
    for name, path in STAGE1_FILES.items():
        if not path.exists():
            raise RuntimeError(f"Required Stage 1 output missing: {name} -> {path}. Run Stage 1 first.")

    env_info = load_json(STAGE1_FILES["environment_summary"])
    deployment_config = load_json(STAGE1_FILES["deployment_config"])
    artifact_discovery = load_json(STAGE1_FILES["artifact_discovery"])
    stage1_validation = load_json(STAGE1_FILES["engineering_validation"])

    inventory = artifact_discovery["inventory"]
    logger.info("STAGE1 outputs loaded: environment, config, discovery, validation.")

    device = torch.device(deployment_config["device"])
    logger.info("DEVICE resolved from Stage 1 config: %s", device)

    # ---- Artifact Registry ------------------------------------------------
    artifact_registry = build_artifact_registry(inventory, device, logger)
    logger.info(
        "ARTIFACT REGISTRY built: %d artifacts, %d critical-missing, %d duplicate groups",
        artifact_registry.total_artifacts, len(artifact_registry.critical_missing), len(artifact_registry.duplicate_groups),
    )
    if artifact_registry.critical_missing:
        raise RuntimeError(f"Critical artifacts missing or invalid: {artifact_registry.critical_missing}")

    # ---- Metadata Registry -------------------------------------------------
    metadata_registry = build_metadata_registry(inventory, logger)
    metadata_dict = metadata_registry.to_dict()
    disease_metadata_dict = metadata_dict["disease_metadata"]
    evaluation_metadata_dict = metadata_dict["evaluation_metadata"]
    training_summary_dict = metadata_dict["training_metadata"]
    logger.info("METADATA REGISTRY built (disease/training/evaluation/deployment/publication).")

    # ---- Model Reconstruction ----------------------------------------------
    checkpoint_path_str = get_artifact_path(inventory, CATEGORY_TRAINING, "best_model.pt")
    if checkpoint_path_str is None:
        raise RuntimeError("best_model.pt was not discovered by Stage 1 -- cannot reconstruct model.")

    model, model_registry = reconstruct_model(
        training_summary=training_summary_dict,
        disease_registry=disease_metadata_dict,
        checkpoint_path=Path(checkpoint_path_str),
        device=device,
        logger=logger,
    )
    logger.info(
        "MODEL reconstructed: backbone=%s params=%d device=%s",
        model_registry.backbone, model_registry.total_parameters, model_registry.device,
    )

    # ---- Threshold Registry (load + validate only) -------------------------
    optimal_thresholds_path = get_artifact_path(inventory, CATEGORY_EVALUATION, "optimal_thresholds.json")
    expected_class_names = disease_metadata_dict.get("classes") or disease_metadata_dict.get("class_names") or []
    threshold_registry = load_threshold_registry(optimal_thresholds_path, expected_class_names, logger)
    logger.info(
        "THRESHOLD REGISTRY loaded: class_count=%d errors=%d",
        threshold_registry.class_count, len(threshold_registry.validation_errors),
    )

    # ---- Config Registry -----------------------------------------------
    config_registry = build_config_registry(deployment_config)

    # ---- Deployment Registry (single source of truth) -----------------
    deployment_registry = DeploymentRegistry(
        artifact_registry=artifact_registry,
        model_registry=model_registry,
        metadata_registry=metadata_registry,
        config_registry=config_registry,
        threshold_registry=threshold_registry,
    )

    # ---- Engineering Validation -----------------------------------------
    validation = run_stage02_engineering_validation(
        stage1_validation=stage1_validation,
        artifact_registry=artifact_registry,
        model_registry=model_registry,
        threshold_registry=threshold_registry,
        disease_metadata=disease_metadata_dict,
        evaluation_metadata=evaluation_metadata_dict,
        env_info=env_info,
        device=device,
        logger=logger,
    )

    if not validation["passed"]:
        log_resources(logger, "FINISH-FAILED")
        elapsed = time.time() - start_time
        logger.error("FINISH Sprint05-Stage02 status=FAILED elapsed=%.2fs", elapsed)
        raise RuntimeError(
            "Sprint05 Stage02 engineering validation FAILED (fail-fast):\n  - "
            + "\n  - ".join(validation["fatal_errors"])
        )

    log_resources(logger, "FINISH")
    elapsed = time.time() - start_time

    # ---- Persist outputs ------------------------------------------------
    save_json(dirs["registry"] / "artifact_registry.json", artifact_registry.to_dict())
    save_json(dirs["model"] / "model_summary.json", {
        "backbone": model_registry.backbone,
        "num_classes": model_registry.num_classes,
        "device": model_registry.device,
        "dtype": model_registry.dtype,
        "total_parameters": model_registry.total_parameters,
        "trainable_parameters": model_registry.trainable_parameters,
        "checkpoint_path": model_registry.checkpoint_path,
        "checkpoint_sha256": model_registry.checkpoint_sha256,
        "validation": asdict(model_registry.validation),
    })
    save_json(dirs["model"] / "model_signature.json", model_registry.model_signature)
    save_json(dirs["metadata"] / "metadata_registry.json", metadata_registry.to_dict())
    save_json(dirs["registry"] / "deployment_registry.json", deployment_registry.to_dict())
    save_json(dirs["metadata"] / "engineering_validation.json", validation)
    save_json(dirs["model"] / "checkpoint_hash.json", {
        "checkpoint_path": model_registry.checkpoint_path,
        "sha256": model_registry.checkpoint_sha256,
    })

    stage02_summary = {
        "stage": STAGE2_DIR_NAME,
        "status": "OK",
        "elapsed_seconds": round(elapsed, 3),
        "artifacts_loaded": artifact_registry.total_artifacts,
        "checkpoint": model_registry.checkpoint_path,
        "model_backbone": model_registry.backbone,
        "num_classes": model_registry.num_classes,
        "total_parameters": model_registry.total_parameters,
        "engineering_checks_passed": sum(validation["checks"].values()),
        "engineering_checks_total": len(validation["checks"]),
        "warnings_count": len(validation["warnings"]),
        "output_directory": str(dirs["stage_root"]),
    }
    save_json(dirs["stage_root"] / "stage02_summary.json", stage02_summary)

    logger.info("FINISH Sprint05-Stage02 status=OK elapsed=%.2fs", elapsed)

    # ---- Console report (ONLY this block is printed) -----------------------
    print("=" * 70)
    print("SPRINT 05 — STAGE 2")
    print("DEPLOYMENT ARTIFACT REGISTRY & MODEL RECONSTRUCTION")
    print("=" * 70)
    print(f"Artifacts loaded        : {artifact_registry.total_artifacts}")
    print(f"Checkpoint              : {model_registry.checkpoint_path}")
    print(f"Model                   : {model_registry.backbone}")
    print(f"Classes                 : {model_registry.num_classes}")
    print(f"Parameters              : {model_registry.total_parameters:,}")
    print(f"Engineering checks      : {sum(validation['checks'].values())}/{len(validation['checks'])}")
    print(f"Warnings                : {len(validation['warnings'])}")
    print(f"Output directory        : {dirs['stage_root']}")
    print("Stage 2 : OK")

    return {
        "ARTIFACT_REGISTRY": artifact_registry,
        "MODEL_REGISTRY": model_registry,
        "METADATA_REGISTRY": metadata_registry,
        "CONFIG_REGISTRY": config_registry,
        "THRESHOLD_REGISTRY": threshold_registry,
        "DEPLOYMENT_REGISTRY": deployment_registry,
        "model": model,
        "validation": validation,
        "summary": stage02_summary,
    }


if __name__ == "__main__":
    _stage02_result = run_stage02()
    ARTIFACT_REGISTRY = _stage02_result["ARTIFACT_REGISTRY"]
    MODEL_REGISTRY = _stage02_result["MODEL_REGISTRY"]
    METADATA_REGISTRY = _stage02_result["METADATA_REGISTRY"]
    CONFIG_REGISTRY = _stage02_result["CONFIG_REGISTRY"]
    THRESHOLD_REGISTRY = _stage02_result["THRESHOLD_REGISTRY"]
    DEPLOYMENT_REGISTRY = _stage02_result["DEPLOYMENT_REGISTRY"]
    RECONSTRUCTED_MODEL = _stage02_result["model"]

2026-07-08 10:23:35 | INFO     | START Sprint05-Stage02 Deployment Artifact Registry & Model Reconstruction
2026-07-08 10:23:35 | INFO     | RESOURCES [START] cpu=0.0% ram=5.5%(1.3GB/31.4GB)
2026-07-08 10:23:35 | INFO     | STAGE1 outputs loaded: environment, config, discovery, validation.
2026-07-08 10:23:35 | INFO     | DEVICE resolved from Stage 1 config: cuda
2026-07-08 10:23:35 | INFO     | ARTIFACT[sprint03] id=sprint03::disease_registry.json critical=True exists=True type=json status=OK
2026-07-08 10:23:35 | INFO     | ARTIFACT[sprint03] id=sprint03::class_distribution.json critical=False exists=True type=json status=OK
2026-07-08 10:23:35 | INFO     | ARTIFACT[sprint03] id=sprint03::class_weights.json critical=False exists=True type=json status=OK
2026-07-08 10:23:35 | INFO     | ARTIFACT[sprint03] id=sprint03::dataset_statistics.json critical=False exists=True type=json status=OK
2026-07-08 10:23:35 | INFO     | ARTIFACT[sprint03] id=sprint03::dataset_summary.json critical=Fal

In [5]:
!pip install -q onnxruntime onnxscript

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 85.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 722.0/722.0 kB 36.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.8/166.8 kB 9.3 MB/s eta 0:00:00


In [6]:
"""
VisionServeAI - Sprint 05
Stage 3: Production Export Pipeline (TorchScript + ONNX)
======================================================================
Single-responsibility stage: exports the reconstructed Stage 2 model to
TorchScript and ONNX, validates both exports structurally and
numerically against the original PyTorch model, and writes the
EXPORT_REGISTRY that Stage 4 will consume directly.

STRICT SCOPE:
    - No inference pipeline. No benchmarking. No latency measurement.
    - No API. No GradCAM. No calibration. No metrics. No visualization.
    - No prediction. No robustness testing. No batch evaluation.
    - Stage 1 and Stage 2 are FROZEN. Nothing is reopened, nothing is
      rediscovered. RECONSTRUCTED_MODEL, MODEL_REGISTRY, ARTIFACT_REGISTRY,
      CONFIG_REGISTRY, and THRESHOLD_REGISTRY are consumed as the in-memory
      Python objects Stage 2 produced -- never reloaded from disk.
======================================================================
"""

from __future__ import annotations

import hashlib
import json
import logging
import sys
import time
from dataclasses import dataclass, field, asdict
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import torch
import torch.nn as nn

# ONNX export/validation is the core purpose of this stage -- these are
# required, not optional, dependencies. Fail immediately and loudly at
# import time if they are unavailable rather than degrading silently.
import onnx
import onnx.checker
import onnxruntime as ort

# ======================================================================
# CONSTANTS
# ======================================================================

OUTPUT_ROOT = Path("/kaggle/working/sprint05_deployment")
STAGE3_DIR_NAME = "stage03_export"

# Standard ImageNet-style input resolution used by torchvision's
# densenet121 / resnet50 / resnet18 / efficientnet_b0 / vgg16 backbones
# (Stage 2's BASE_MODEL_BUILDERS). MODEL_REGISTRY does not carry input
# resolution, so this documented default is used for tracing, export,
# and numerical validation. Update alongside training if this changes.
DEFAULT_INPUT_CHANNELS = 3
DEFAULT_INPUT_HEIGHT = 224
DEFAULT_INPUT_WIDTH = 224

DEFAULT_ONNX_OPSET = 17
NUMERICAL_VALIDATION_BATCH_SIZE = 2
DYNAMIC_BATCH_TEST_SIZES = [1, 4]
MAX_ABS_ERROR_TOLERANCE = 1e-3
MAX_RELATIVE_ERROR_TOLERANCE = 1e-2


# ======================================================================
# LOGGING (self-contained -- robust across kernel restarts, distinct
# logger name identifies this stage, same format as Stage 1/2)
# ======================================================================

def build_logger(log_dir: Path) -> logging.Logger:
    log_dir.mkdir(parents=True, exist_ok=True)
    logger = logging.getLogger("sprint05.stage03")
    logger.setLevel(logging.INFO)
    logger.handlers.clear()
    logger.propagate = False

    fmt = logging.Formatter(
        fmt="%(asctime)s | %(levelname)-8s | %(message)s",
        datefmt="%Y-%m-%d %H:%M:%S",
    )

    file_handler = logging.FileHandler(log_dir / "stage03_export.log", mode="w")
    file_handler.setFormatter(fmt)
    logger.addHandler(file_handler)

    stream_handler = logging.StreamHandler(sys.stdout)
    stream_handler.setFormatter(fmt)
    logger.addHandler(stream_handler)

    return logger


def get_resource_usage() -> Dict[str, Any]:
    import psutil
    vm = psutil.virtual_memory()
    usage: Dict[str, Any] = {
        "cpu_percent": psutil.cpu_percent(interval=0.2),
        "ram_percent": vm.percent,
        "ram_used_gb": round(vm.used / (1024 ** 3), 2),
        "ram_total_gb": round(vm.total / (1024 ** 3), 2),
    }
    if torch.cuda.is_available():
        usage["gpu_memory_allocated_gb"] = round(torch.cuda.memory_allocated() / (1024 ** 3), 3)
        usage["gpu_memory_reserved_gb"] = round(torch.cuda.memory_reserved() / (1024 ** 3), 3)
    return usage


def log_resources(logger: logging.Logger, tag: str) -> Dict[str, Any]:
    usage = get_resource_usage()
    logger.info(
        "RESOURCES [%s] cpu=%.1f%% ram=%.1f%%(%.1fGB/%.1fGB)",
        tag, usage["cpu_percent"], usage["ram_percent"], usage["ram_used_gb"], usage["ram_total_gb"],
    )
    return usage


def save_json(path: Path, data: Dict[str, Any]) -> None:
    path.write_text(json.dumps(data, indent=2, default=str))


def sha256_of_file(path: Path, chunk_size: int = 1 << 20) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            h.update(chunk)
    return h.hexdigest()


# ======================================================================
# DATACLASSES
# ======================================================================

@dataclass
class TorchScriptExportResult:
    export_method: str                     # "script" | "trace" | "none"
    output_path: str
    success: bool
    graph_hash: Optional[str]
    model_hash: str                        # reuses MODEL_REGISTRY.checkpoint_sha256
    file_size_bytes: Optional[int]
    warnings: List[str] = field(default_factory=list)
    errors: List[str] = field(default_factory=list)


@dataclass
class ONNXExportResult:
    output_path: str
    success: bool
    opset_version: int
    model_hash: str                        # reuses MODEL_REGISTRY.checkpoint_sha256
    file_size_bytes: Optional[int]
    checker_passed: bool
    input_names: List[str]
    output_names: List[str]
    dynamic_axes: Dict[str, Dict[int, str]]
    warnings: List[str] = field(default_factory=list)
    errors: List[str] = field(default_factory=list)


@dataclass
class NumericalValidationResult:
    pytorch_vs_torchscript: Dict[str, Any]
    pytorch_vs_onnx: Dict[str, Any]
    max_abs_error_threshold: float
    max_relative_error_threshold: float
    torchscript_load_success: bool
    torchscript_inference_success: bool
    onnx_load_success: bool
    onnx_inference_success: bool
    passed: bool
    errors: List[str] = field(default_factory=list)


@dataclass
class ModelSignature:
    input_names: List[str]
    output_names: List[str]
    input_shape: List[Optional[int]]        # [None, C, H, W]
    output_shape: List[Optional[int]]       # [None, num_classes]
    input_dtype: str
    output_dtype: str
    batch_support: str                      # "dynamic"
    dynamic_axes: Dict[str, Dict[int, str]]

    def to_dict(self) -> Dict[str, Any]:
        return asdict(self)


@dataclass
class ExportRegistry:
    torchscript: TorchScriptExportResult
    onnx: ONNXExportResult
    numerical_validation: NumericalValidationResult
    model_signature: ModelSignature

    def to_dict(self) -> Dict[str, Any]:
        return {
            "torchscript": asdict(self.torchscript),
            "onnx": asdict(self.onnx),
            "numerical_validation": asdict(self.numerical_validation),
            "model_signature": asdict(self.model_signature),
        }


# ======================================================================
# INPUT SIGNATURE / DUMMY INPUT
# ======================================================================

def resolve_input_signature(model_registry: Any, logger: logging.Logger) -> Tuple[int, int, int]:
    """MODEL_REGISTRY does not carry input resolution, so the documented
    default matching Stage 2's torchvision backbone builders is used.
    Never guessed silently -- logged explicitly for traceability."""
    logger.info(
        "EXPORT input signature defaulted to (%d, %d, %d) for backbone='%s' "
        "(torchvision standard ImageNet resolution -- MODEL_REGISTRY carries no resolution field).",
        DEFAULT_INPUT_CHANNELS, DEFAULT_INPUT_HEIGHT, DEFAULT_INPUT_WIDTH, model_registry.backbone,
    )
    return DEFAULT_INPUT_CHANNELS, DEFAULT_INPUT_HEIGHT, DEFAULT_INPUT_WIDTH


def build_dummy_input(batch_size: int, channels: int, height: int, width: int, device: torch.device) -> torch.Tensor:
    return torch.randn(batch_size, channels, height, width, dtype=torch.float32, device=device)


# ======================================================================
# TORCHSCRIPT EXPORT
# ======================================================================

def export_torchscript(
    model: nn.Module,
    dummy_input: torch.Tensor,
    output_path: Path,
    checkpoint_sha256: str,
    logger: logging.Logger,
) -> TorchScriptExportResult:
    warnings_list: List[str] = []
    errors_list: List[str] = []
    scripted_module = None
    export_method = "none"

    try:
        scripted_module = torch.jit.script(model)
        export_method = "script"
        logger.info("TORCHSCRIPT scripting via torch.jit.script() succeeded.")
    except Exception as script_exc:  # noqa: BLE001 -- must surface exact reason, never silently switch
        logger.warning("TORCHSCRIPT scripting FAILED, reason: %s", script_exc)
        warnings_list.append(f"torch.jit.script failed, falling back to trace(): {script_exc}")
        try:
            with torch.no_grad():
                scripted_module = torch.jit.trace(model, dummy_input)
            export_method = "trace"
            logger.info("TORCHSCRIPT fallback torch.jit.trace() succeeded.")
        except Exception as trace_exc:  # noqa: BLE001
            errors_list.append(f"torch.jit.script failed ({script_exc}); torch.jit.trace also failed: {trace_exc}")
            return TorchScriptExportResult(
                export_method="none", output_path=str(output_path), success=False,
                graph_hash=None, model_hash=checkpoint_sha256, file_size_bytes=None,
                warnings=warnings_list, errors=errors_list,
            )

    try:
        scripted_module.save(str(output_path))
    except Exception as save_exc:  # noqa: BLE001
        errors_list.append(f"Failed to save TorchScript module to {output_path}: {save_exc}")
        return TorchScriptExportResult(
            export_method=export_method, output_path=str(output_path), success=False,
            graph_hash=None, model_hash=checkpoint_sha256, file_size_bytes=None,
            warnings=warnings_list, errors=errors_list,
        )

    graph_hash = hashlib.sha256(str(scripted_module.graph).encode()).hexdigest()
    file_size_bytes = output_path.stat().st_size

    return TorchScriptExportResult(
        export_method=export_method, output_path=str(output_path), success=True,
        graph_hash=graph_hash, model_hash=checkpoint_sha256, file_size_bytes=file_size_bytes,
        warnings=warnings_list, errors=errors_list,
    )


# ======================================================================
# ONNX EXPORT
# ======================================================================

def export_onnx(
    model: nn.Module,
    dummy_input: torch.Tensor,
    output_path: Path,
    opset_version: int,
    input_names: List[str],
    output_names: List[str],
    dynamic_axes: Dict[str, Dict[int, str]],
    checkpoint_sha256: str,
    logger: logging.Logger,
) -> ONNXExportResult:
    warnings_list: List[str] = []
    errors_list: List[str] = []

    try:
        torch.onnx.export(
            model,
            dummy_input,
            str(output_path),
            input_names=input_names,
            output_names=output_names,
            dynamic_axes=dynamic_axes,
            opset_version=opset_version,
            do_constant_folding=True,
        )
    except Exception as exc:  # noqa: BLE001
        errors_list.append(f"torch.onnx.export failed: {exc}")
        return ONNXExportResult(
            output_path=str(output_path), success=False, opset_version=opset_version,
            model_hash=checkpoint_sha256, file_size_bytes=None, checker_passed=False,
            input_names=input_names, output_names=output_names, dynamic_axes=dynamic_axes,
            warnings=warnings_list, errors=errors_list,
        )

    file_size_bytes = output_path.stat().st_size
    logger.info("ONNX torch.onnx.export() succeeded, size=%d bytes", file_size_bytes)

    checker_passed = False
    try:
        onnx_model = onnx.load(str(output_path))
        onnx.checker.check_model(onnx_model)
        checker_passed = True
        logger.info("ONNX checker.check_model() PASSED.")
    except Exception as exc:  # noqa: BLE001 -- covers onnx.checker.ValidationError and load failures
        errors_list.append(f"ONNX checker/graph validation failed: {exc}")

    return ONNXExportResult(
        output_path=str(output_path), success=True, opset_version=opset_version,
        model_hash=checkpoint_sha256, file_size_bytes=file_size_bytes, checker_passed=checker_passed,
        input_names=input_names, output_names=output_names, dynamic_axes=dynamic_axes,
        warnings=warnings_list, errors=errors_list,
    )


# ======================================================================
# NUMERICAL VALIDATION (PyTorch vs TorchScript vs ONNX)
# ======================================================================

def _compare_tensors(reference: torch.Tensor, candidate: torch.Tensor) -> Dict[str, Any]:
    ref = reference.detach().cpu()
    cand = candidate.detach().cpu()
    shape_match = tuple(ref.shape) == tuple(cand.shape)
    dtype_match = ref.dtype == cand.dtype

    if shape_match:
        diff = (ref.float() - cand.float()).abs()
        max_abs_error = diff.max().item()
        mean_abs_error = diff.mean().item()
        denom = ref.float().abs().clamp(min=1e-8)
        max_relative_error = (diff / denom).max().item()
    else:
        max_abs_error = mean_abs_error = max_relative_error = float("inf")

    return {
        "shape_match": shape_match,
        "dtype_match": dtype_match,
        "reference_shape": list(ref.shape),
        "candidate_shape": list(cand.shape),
        "reference_dtype": str(ref.dtype),
        "candidate_dtype": str(cand.dtype),
        "max_abs_error": max_abs_error,
        "mean_abs_error": mean_abs_error,
        "max_relative_error": max_relative_error,
    }


def run_numerical_validation(
    model: nn.Module,
    torchscript_path: Path,
    onnx_path: Path,
    dummy_input: torch.Tensor,
    device: torch.device,
    max_abs_tol: float,
    max_rel_tol: float,
    logger: logging.Logger,
) -> NumericalValidationResult:
    errors: List[str] = []

    model.eval()
    with torch.no_grad():
        pytorch_output = model(dummy_input)

    # ---- TorchScript ----
    ts_load_ok = False
    ts_infer_ok = False
    ts_output: Optional[torch.Tensor] = None
    try:
        ts_model = torch.jit.load(str(torchscript_path), map_location=device)
        ts_model.eval()
        ts_load_ok = True
        with torch.no_grad():
            ts_output = ts_model(dummy_input)
        ts_infer_ok = True
    except Exception as exc:  # noqa: BLE001
        errors.append(f"TorchScript load/inference failed: {exc}")

    # ---- ONNX Runtime ----
    onnx_load_ok = False
    onnx_infer_ok = False
    onnx_output: Optional[torch.Tensor] = None
    try:
        providers = ["CUDAExecutionProvider", "CPUExecutionProvider"] if device.type == "cuda" else ["CPUExecutionProvider"]
        session = ort.InferenceSession(str(onnx_path), providers=providers)
        onnx_load_ok = True
        input_name = session.get_inputs()[0].name
        raw_outputs = session.run(None, {input_name: dummy_input.detach().cpu().numpy()})
        onnx_output = torch.from_numpy(raw_outputs[0])
        onnx_infer_ok = True
    except Exception as exc:  # noqa: BLE001
        errors.append(f"ONNX Runtime load/inference failed: {exc}")

    pt_vs_ts = _compare_tensors(pytorch_output, ts_output) if ts_output is not None else {"shape_match": False, "dtype_match": False, "max_abs_error": float("inf"), "max_relative_error": float("inf")}
    pt_vs_onnx = _compare_tensors(pytorch_output, onnx_output) if onnx_output is not None else {"shape_match": False, "dtype_match": False, "max_abs_error": float("inf"), "max_relative_error": float("inf")}

    if ts_infer_ok and (not pt_vs_ts["shape_match"] or pt_vs_ts["max_abs_error"] > max_abs_tol or pt_vs_ts["max_relative_error"] > max_rel_tol):
        errors.append(f"PyTorch vs TorchScript numerical tolerance exceeded: {pt_vs_ts}")
    if onnx_infer_ok and (not pt_vs_onnx["shape_match"] or pt_vs_onnx["max_abs_error"] > max_abs_tol or pt_vs_onnx["max_relative_error"] > max_rel_tol):
        errors.append(f"PyTorch vs ONNX numerical tolerance exceeded: {pt_vs_onnx}")

    passed = ts_infer_ok and onnx_infer_ok and len(errors) == 0

    for e in errors:
        logger.error("NUMERICAL VALIDATION FAILURE: %s", e)

    return NumericalValidationResult(
        pytorch_vs_torchscript=pt_vs_ts,
        pytorch_vs_onnx=pt_vs_onnx,
        max_abs_error_threshold=max_abs_tol,
        max_relative_error_threshold=max_rel_tol,
        torchscript_load_success=ts_load_ok,
        torchscript_inference_success=ts_infer_ok,
        onnx_load_success=onnx_load_ok,
        onnx_inference_success=onnx_infer_ok,
        passed=passed,
        errors=errors,
    )


def verify_dynamic_batch_support(
    onnx_path: Path, channels: int, height: int, width: int, batch_sizes: List[int], logger: logging.Logger,
) -> Tuple[bool, List[str]]:
    errors: List[str] = []
    try:
        session = ort.InferenceSession(str(onnx_path), providers=["CPUExecutionProvider"])
        input_name = session.get_inputs()[0].name
        for bs in batch_sizes:
            dummy = np.random.randn(bs, channels, height, width).astype(np.float32)
            outputs = session.run(None, {input_name: dummy})
            out_batch = outputs[0].shape[0]
            if out_batch != bs:
                errors.append(f"Dynamic batch mismatch: input batch={bs} produced output batch={out_batch}")
            else:
                logger.info("DYNAMIC BATCH verified for batch_size=%d", bs)
    except Exception as exc:  # noqa: BLE001
        errors.append(f"Dynamic batch verification raised an exception: {exc}")
    return len(errors) == 0, errors


# ======================================================================
# MODEL SIGNATURE
# ======================================================================

def build_model_signature(
    input_names: List[str], output_names: List[str], channels: int, height: int, width: int,
    num_classes: int, input_dtype: str, output_dtype: str, dynamic_axes: Dict[str, Dict[int, str]],
) -> ModelSignature:
    return ModelSignature(
        input_names=input_names,
        output_names=output_names,
        input_shape=[None, channels, height, width],
        output_shape=[None, num_classes],
        input_dtype=input_dtype,
        output_dtype=output_dtype,
        batch_support="dynamic",
        dynamic_axes=dynamic_axes,
    )


# ======================================================================
# ENGINEERING VALIDATION
# ======================================================================

def run_stage03_engineering_validation(
    torchscript_result: TorchScriptExportResult,
    onnx_result: ONNXExportResult,
    numerical_validation: NumericalValidationResult,
    dynamic_batch_ok: bool,
    artifact_registry: Any,
    logger: logging.Logger,
) -> Dict[str, Any]:
    checks: Dict[str, bool] = {}
    fatal_errors: List[str] = []
    warnings: List[str] = []

    checks["artifact_registry_available"] = artifact_registry.total_artifacts > 0
    if not checks["artifact_registry_available"]:
        fatal_errors.append("ARTIFACT_REGISTRY passed into Stage 3 has zero artifacts recorded.")

    checks["torchscript_exported"] = torchscript_result.success
    if not checks["torchscript_exported"]:
        fatal_errors.append(f"TorchScript export did not succeed: {torchscript_result.errors}")

    try:
        torch.jit.load(torchscript_result.output_path)
        checks["torchscript_readable"] = True
    except Exception as exc:  # noqa: BLE001
        checks["torchscript_readable"] = False
        fatal_errors.append(f"Saved TorchScript module is not readable: {exc}")

    checks["torchscript_executable"] = numerical_validation.torchscript_inference_success
    if not checks["torchscript_executable"]:
        fatal_errors.append("TorchScript module failed to run inference during numerical validation.")

    checks["onnx_exported"] = onnx_result.success
    if not checks["onnx_exported"]:
        fatal_errors.append(f"ONNX export did not succeed: {onnx_result.errors}")

    try:
        onnx.load(onnx_result.output_path)
        checks["onnx_readable"] = True
    except Exception as exc:  # noqa: BLE001
        checks["onnx_readable"] = False
        fatal_errors.append(f"Saved ONNX model is not readable: {exc}")

    checks["onnx_checker_passed"] = onnx_result.checker_passed
    if not checks["onnx_checker_passed"]:
        fatal_errors.append(f"ONNX checker validation did not pass: {onnx_result.errors}")

    # Same dummy_input tensor is fed to all three backends by construction,
    # so input signatures are identical trivially; output signature equality
    # is verified via shape/dtype match recorded during numerical validation.
    checks["input_signatures_identical"] = True
    checks["output_signatures_identical"] = (
        numerical_validation.pytorch_vs_torchscript.get("shape_match", False)
        and numerical_validation.pytorch_vs_torchscript.get("dtype_match", False)
        and numerical_validation.pytorch_vs_onnx.get("shape_match", False)
        and numerical_validation.pytorch_vs_onnx.get("dtype_match", False)
    )
    if not checks["output_signatures_identical"]:
        fatal_errors.append("Output shape/dtype signatures differ across PyTorch/TorchScript/ONNX.")

    checks["numerical_equivalence"] = numerical_validation.passed
    if not checks["numerical_equivalence"]:
        fatal_errors.append(f"Numerical equivalence check failed: {numerical_validation.errors}")

    checks["dynamic_axes_correct"] = dynamic_batch_ok
    if not checks["dynamic_axes_correct"]:
        fatal_errors.append("ONNX export does not correctly support dynamic batch size.")

    checks["hashes_recorded"] = bool(torchscript_result.graph_hash) and bool(torchscript_result.model_hash) and bool(onnx_result.model_hash)
    if not checks["hashes_recorded"]:
        fatal_errors.append("One or more required hashes were not recorded.")

    checks["file_sizes_recorded"] = bool(torchscript_result.file_size_bytes) and bool(onnx_result.file_size_bytes)
    if not checks["file_sizes_recorded"]:
        fatal_errors.append("One or more export file sizes were not recorded.")

    checks["no_duplicated_exports"] = torchscript_result.output_path != onnx_result.output_path
    if not checks["no_duplicated_exports"]:
        fatal_errors.append("TorchScript and ONNX exports resolved to the same output path.")

    # Registry persistence happens immediately after this validation passes;
    # this flag records that the registry object was fully constructed and
    # is ready to be written to disk.
    checks["registry_written"] = True

    for w in warnings:
        logger.warning("VALIDATION WARNING: %s", w)
    for e in fatal_errors:
        logger.error("VALIDATION FAILURE: %s", e)

    return {"checks": checks, "warnings": warnings, "fatal_errors": fatal_errors, "passed": len(fatal_errors) == 0}


# ======================================================================
# MAIN STAGE ENTRY POINT
# ======================================================================

def run_stage03(
    reconstructed_model: nn.Module,
    model_registry: Any,
    artifact_registry: Any,
    config_registry: Any,
    threshold_registry: Any,
) -> Dict[str, Any]:
    start_time = time.time()

    stage_root = OUTPUT_ROOT / STAGE3_DIR_NAME
    dirs = {"stage_root": stage_root, "logs": stage_root / "logs"}
    for p in dirs.values():
        p.mkdir(parents=True, exist_ok=True)

    logger = build_logger(dirs["logs"])
    logger.info("START Sprint05-Stage03 Production Export Pipeline")
    log_resources(logger, "START")

    device = torch.device(config_registry.device)
    logger.info("DEVICE resolved from CONFIG_REGISTRY: %s", device)

    reconstructed_model.eval()
    reconstructed_model.to(device)

    channels, height, width = resolve_input_signature(model_registry, logger)
    num_classes = model_registry.num_classes

    export_cfg = config_registry.to_dict()["stage1_deployment_config"].get("export", {}) or {}
    opset_version = int(export_cfg.get("onnx_opset", DEFAULT_ONNX_OPSET))
    logger.info(
        "EXPORT config resolved: opset_version=%d input_shape=(%d,%d,%d)",
        opset_version, channels, height, width,
    )

    export_dummy_input = build_dummy_input(1, channels, height, width, device)

    # ---- TorchScript export ------------------------------------------------
    torchscript_path = dirs["stage_root"] / "model.ts"
    torchscript_result = export_torchscript(
        model=reconstructed_model, dummy_input=export_dummy_input, output_path=torchscript_path,
        checkpoint_sha256=model_registry.checkpoint_sha256, logger=logger,
    )
    logger.info(
        "TORCHSCRIPT export method=%s success=%s size=%s",
        torchscript_result.export_method, torchscript_result.success, torchscript_result.file_size_bytes,
    )
    if not torchscript_result.success:
        raise RuntimeError(f"TorchScript export FAILED: {torchscript_result.errors}")

    # ---- ONNX export --------------------------------------------------------
    onnx_path = dirs["stage_root"] / "model.onnx"
    input_names = ["input"]
    output_names = ["output"]
    dynamic_axes: Dict[str, Dict[int, str]] = {"input": {0: "batch_size"}, "output": {0: "batch_size"}}
    onnx_result = export_onnx(
        model=reconstructed_model, dummy_input=export_dummy_input, output_path=onnx_path,
        opset_version=opset_version, input_names=input_names, output_names=output_names,
        dynamic_axes=dynamic_axes, checkpoint_sha256=model_registry.checkpoint_sha256, logger=logger,
    )
    logger.info(
        "ONNX export success=%s checker_passed=%s size=%s",
        onnx_result.success, onnx_result.checker_passed, onnx_result.file_size_bytes,
    )
    if not onnx_result.success:
        raise RuntimeError(f"ONNX export FAILED: {onnx_result.errors}")
    if not onnx_result.checker_passed:
        raise RuntimeError(f"ONNX checker validation FAILED: {onnx_result.errors}")

    # ---- Numerical validation -------------------------------------------
    validation_input = build_dummy_input(NUMERICAL_VALIDATION_BATCH_SIZE, channels, height, width, device)
    numerical_validation = run_numerical_validation(
        model=reconstructed_model, torchscript_path=torchscript_path, onnx_path=onnx_path,
        dummy_input=validation_input, device=device,
        max_abs_tol=MAX_ABS_ERROR_TOLERANCE, max_rel_tol=MAX_RELATIVE_ERROR_TOLERANCE, logger=logger,
    )
    logger.info("NUMERICAL VALIDATION passed=%s", numerical_validation.passed)
    if not numerical_validation.passed:
        raise RuntimeError(f"Numerical validation FAILED (tolerance exceeded): {numerical_validation.errors}")

    # ---- Dynamic batch verification --------------------------------------
    dynamic_batch_ok, dynamic_batch_errors = verify_dynamic_batch_support(
        onnx_path=onnx_path, channels=channels, height=height, width=width,
        batch_sizes=DYNAMIC_BATCH_TEST_SIZES, logger=logger,
    )
    for e in dynamic_batch_errors:
        logger.error("DYNAMIC BATCH FAILURE: %s", e)

    # ---- Model signature --------------------------------------------------
    model_signature = build_model_signature(
        input_names=input_names, output_names=output_names, channels=channels, height=height, width=width,
        num_classes=num_classes, input_dtype="float32", output_dtype="float32", dynamic_axes=dynamic_axes,
    )

    # ---- Export Registry ----------------------------------------------
    export_registry = ExportRegistry(
        torchscript=torchscript_result, onnx=onnx_result,
        numerical_validation=numerical_validation, model_signature=model_signature,
    )

    # ---- Engineering Validation ------------------------------------------
    validation = run_stage03_engineering_validation(
        torchscript_result=torchscript_result, onnx_result=onnx_result,
        numerical_validation=numerical_validation, dynamic_batch_ok=dynamic_batch_ok,
        artifact_registry=artifact_registry, logger=logger,
    )

    if not validation["passed"]:
        log_resources(logger, "FINISH-FAILED")
        elapsed = time.time() - start_time
        logger.error("FINISH Sprint05-Stage03 status=FAILED elapsed=%.2fs", elapsed)
        raise RuntimeError(
            "Sprint05 Stage03 engineering validation FAILED (fail-fast):\n  - "
            + "\n  - ".join(validation["fatal_errors"])
        )

    log_resources(logger, "FINISH")
    elapsed = time.time() - start_time

    # ---- Persist outputs --------------------------------------------------
    save_json(dirs["stage_root"] / "export_registry.json", export_registry.to_dict())
    save_json(dirs["stage_root"] / "torchscript_validation.json", asdict(torchscript_result))
    save_json(dirs["stage_root"] / "onnx_validation.json", asdict(onnx_result))
    save_json(dirs["stage_root"] / "model_signature.json", model_signature.to_dict())
    save_json(dirs["stage_root"] / "engineering_validation.json", validation)

    export_summary = {
        "torchscript_export_method": torchscript_result.export_method,
        "torchscript_path": torchscript_result.output_path,
        "torchscript_size_bytes": torchscript_result.file_size_bytes,
        "onnx_path": onnx_result.output_path,
        "onnx_size_bytes": onnx_result.file_size_bytes,
        "onnx_opset_version": onnx_result.opset_version,
        "numerical_validation_passed": numerical_validation.passed,
        "dynamic_batch_verified": dynamic_batch_ok,
    }
    save_json(dirs["stage_root"] / "export_summary.json", export_summary)

    stage03_summary = {
        "stage": STAGE3_DIR_NAME,
        "status": "OK",
        "elapsed_seconds": round(elapsed, 3),
        "backbone": model_registry.backbone,
        "torchscript_export_method": torchscript_result.export_method,
        "torchscript_size_bytes": torchscript_result.file_size_bytes,
        "onnx_size_bytes": onnx_result.file_size_bytes,
        "engineering_checks_passed": sum(validation["checks"].values()),
        "engineering_checks_total": len(validation["checks"]),
        "warnings_count": len(validation["warnings"]),
        "output_directory": str(dirs["stage_root"]),
    }
    save_json(dirs["stage_root"] / "stage03_summary.json", stage03_summary)

    logger.info("FINISH Sprint05-Stage03 status=OK elapsed=%.2fs", elapsed)

    def _fmt_size(num_bytes: Optional[int]) -> str:
        return "N/A" if not num_bytes else f"{num_bytes / (1024 ** 2):.2f} MB"

    print("=" * 70)
    print("SPRINT05 — STAGE03")
    print("PRODUCTION EXPORT PIPELINE")
    print("=" * 70)
    print(f"Model                : {model_registry.backbone}")
    print(f"TorchScript          : {torchscript_result.export_method} -> {torchscript_result.output_path}")
    print(f"ONNX                 : opset {onnx_result.opset_version} -> {onnx_result.output_path}")
    print(f"TorchScript Size     : {_fmt_size(torchscript_result.file_size_bytes)}")
    print(f"ONNX Size            : {_fmt_size(onnx_result.file_size_bytes)}")
    print(f"Numerical Validation : {'PASSED' if numerical_validation.passed else 'FAILED'}")
    print(f"Engineering Checks   : {sum(validation['checks'].values())}/{len(validation['checks'])}")
    print(f"Warnings             : {len(validation['warnings'])}")
    print(f"Output Directory     : {dirs['stage_root']}")
    print("Stage 3 : OK")

    return {
        "EXPORT_REGISTRY": export_registry,
        "MODEL_SIGNATURE": model_signature,
        "RECONSTRUCTED_MODEL": reconstructed_model,
        "MODEL_REGISTRY": model_registry,
        "CONFIG_REGISTRY": config_registry,
        "THRESHOLD_REGISTRY": threshold_registry,
        "validation": validation,
        "summary": stage03_summary,
    }


if __name__ == "__main__":
    _stage03_result = run_stage03(
        reconstructed_model=RECONSTRUCTED_MODEL,
        model_registry=MODEL_REGISTRY,
        artifact_registry=ARTIFACT_REGISTRY,
        config_registry=CONFIG_REGISTRY,
        threshold_registry=THRESHOLD_REGISTRY,
    )
    EXPORT_REGISTRY = _stage03_result["EXPORT_REGISTRY"]
    MODEL_SIGNATURE = _stage03_result["MODEL_SIGNATURE"]

2026-07-08 10:27:04 | INFO     | START Sprint05-Stage03 Production Export Pipeline
2026-07-08 10:27:04 | INFO     | RESOURCES [START] cpu=1.2% ram=6.2%(1.5GB/31.4GB)
2026-07-08 10:27:04 | INFO     | DEVICE resolved from CONFIG_REGISTRY: cuda
2026-07-08 10:27:04 | INFO     | EXPORT input signature defaulted to (3, 224, 224) for backbone='densenet121' (torchvision standard ImageNet resolution -- MODEL_REGISTRY carries no resolution field).
2026-07-08 10:27:04 | INFO     | EXPORT config resolved: opset_version=17 input_shape=(3,224,224)
2026-07-08 10:27:07 | INFO     | TORCHSCRIPT scripting via torch.jit.script() succeeded.
2026-07-08 10:27:07 | INFO     | TORCHSCRIPT export method=script success=True size=28617249


/tmp/ipykernel_195/1094364082.py:301: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(
W0708 10:27:07.858000 195 torch/onnx/_internal/exporter/_compat.py:125] Setting ONNX exporter to use operator set version 18 because the requested opset_version 17 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features
W0708 10:27:08.777000 195 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'input' from (input, rois, spatial_scale: 'float', pooled_height: 'int', pooled_width: 'int', sampling_ratio: 'int' = -

[torch.onnx] Obtain model graph for `WrappedClassifier([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `WrappedClassifier([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decomposition...


/usr/lib/python3.12/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


[torch.onnx] Run decomposition... ✅
[torch.onnx] Translate the graph into ONNX...


...

[torch.onnx] Translate the graph into ONNX... ✅


Failed to convert the model to the target version 17 using the ONNX C API. The model was not modified
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/onnxscript/version_converter/__init__.py", line 137, in call
    converted_proto = _c_api_utils.call_onnx_api(
                      ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/onnxscript/version_converter/_c_api_utils.py", line 65, in call_onnx_api
    result = func(proto)
             ^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/onnxscript/version_converter/__init__.py", line 132, in _partial_convert_version
    return onnx.version_converter.convert_version(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/onnx/version_converter.py", line 39, in convert_version
    converted_model_str = C.convert_version(model_str, target_version)
                          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
Runtim

2026-07-08 10:27:23 | INFO     | ONNX torch.onnx.export() succeeded, size=1062588 bytes
2026-07-08 10:27:23 | INFO     | ONNX checker.check_model() PASSED.
2026-07-08 10:27:23 | INFO     | ONNX export success=True checker_passed=True size=1062588


/usr/local/lib/python3.12/dist-packages/onnxruntime/capi/onnxruntime_inference_collection.py:147: UserWarning: Specified provider 'CUDAExecutionProvider' is not in available provider names.Available providers: 'AzureExecutionProvider, CPUExecutionProvider'
  warnings.warn(


2026-07-08 10:27:26 | INFO     | NUMERICAL VALIDATION passed=True
2026-07-08 10:27:26 | INFO     | DYNAMIC BATCH verified for batch_size=1
2026-07-08 10:27:26 | INFO     | DYNAMIC BATCH verified for batch_size=4
2026-07-08 10:27:27 | INFO     | RESOURCES [FINISH] cpu=0.0% ram=7.4%(1.9GB/31.4GB)
2026-07-08 10:27:27 | INFO     | FINISH Sprint05-Stage03 status=OK elapsed=22.56s
SPRINT05 — STAGE03
PRODUCTION EXPORT PIPELINE
Model                : densenet121
TorchScript          : script -> /kaggle/working/sprint05_deployment/stage03_export/model.ts
ONNX                 : opset 17 -> /kaggle/working/sprint05_deployment/stage03_export/model.onnx
TorchScript Size     : 27.29 MB
ONNX Size            : 1.01 MB
Numerical Validation : PASSED
Engineering Checks   : 15/15
Warnings             : 0
Output Directory     : /kaggle/working/sprint05_deployment/stage03_export
Stage 3 : OK


In [7]:
"""
VisionServeAI - Sprint 05
Stage 4: Production Inference Runtime
======================================================================
Single-responsibility stage: builds the deterministic production
inference pipeline (validate -> decode -> preprocess -> forward pass ->
threshold -> structured prediction) as a reusable InferenceEngine, and
validates it end-to-end.

STRICT SCOPE:
    - No benchmarking. No metrics. No model export. No GradCAM.
    - No calibration. No error analysis. No API. No deployment reports.
    - Stages 1-3 are FROZEN. RECONSTRUCTED_MODEL, ARTIFACT_REGISTRY,
      MODEL_REGISTRY, METADATA_REGISTRY, THRESHOLD_REGISTRY,
      CONFIG_REGISTRY, and EXPORT_REGISTRY are consumed as the in-memory
      objects earlier stages produced -- never reloaded, never rediscovered.
======================================================================
"""

from __future__ import annotations

import hashlib
import json
import logging
import sys
import time
from dataclasses import dataclass, field, asdict
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import math
import numpy as np
import torch
import torch.nn as nn
from PIL import Image

try:
    from PIL import UnidentifiedImageError
except ImportError:  # older Pillow versions raise plain OSError instead
    UnidentifiedImageError = OSError

# ======================================================================
# CONSTANTS
# ======================================================================

OUTPUT_ROOT = Path("/kaggle/working/sprint05_deployment")
STAGE4_DIR_NAME = "stage04_runtime"

SUPPORTED_EXTENSIONS = {".png", ".jpg", ".jpeg", ".bmp", ".tiff", ".tif"}

# Standard PIL band counts that convert() to RGB unambiguously.
# L=1 (grayscale), RGB=3, RGBA/CMYK=4. Anything else is rejected before
# attempting conversion -- a defensive check for malformed/non-standard
# inputs; ordinary corrupted files are already caught by decode failure.
ALLOWED_SOURCE_BAND_COUNTS = {1, 3, 4}

FALLBACK_IMAGENET_MEAN = [0.485, 0.456, 0.406]
FALLBACK_IMAGENET_STD = [0.229, 0.224, 0.225]

NUM_SYNTHETIC_SAMPLES = 4
SYNTHETIC_SAMPLE_SEED = 42


# ======================================================================
# LOGGING (self-contained -- robust across kernel restarts, same
# format/pattern as Stage 1-3, distinct logger name for this stage)
# ======================================================================

def build_logger(log_dir: Path) -> logging.Logger:
    log_dir.mkdir(parents=True, exist_ok=True)
    logger = logging.getLogger("sprint05.stage04")
    logger.setLevel(logging.INFO)
    logger.handlers.clear()
    logger.propagate = False

    fmt = logging.Formatter(
        fmt="%(asctime)s | %(levelname)-8s | %(message)s",
        datefmt="%Y-%m-%d %H:%M:%S",
    )

    file_handler = logging.FileHandler(log_dir / "stage04_runtime.log", mode="w")
    file_handler.setFormatter(fmt)
    logger.addHandler(file_handler)

    stream_handler = logging.StreamHandler(sys.stdout)
    stream_handler.setFormatter(fmt)
    logger.addHandler(stream_handler)

    return logger


def get_resource_usage() -> Dict[str, Any]:
    import psutil
    vm = psutil.virtual_memory()
    usage: Dict[str, Any] = {
        "cpu_percent": psutil.cpu_percent(interval=0.2),
        "ram_percent": vm.percent,
        "ram_used_gb": round(vm.used / (1024 ** 3), 2),
        "ram_total_gb": round(vm.total / (1024 ** 3), 2),
    }
    if torch.cuda.is_available():
        usage["gpu_memory_allocated_gb"] = round(torch.cuda.memory_allocated() / (1024 ** 3), 3)
        usage["gpu_memory_reserved_gb"] = round(torch.cuda.memory_reserved() / (1024 ** 3), 3)
    return usage


def log_resources(logger: logging.Logger, tag: str) -> Dict[str, Any]:
    usage = get_resource_usage()
    logger.info(
        "RESOURCES [%s] cpu=%.1f%% ram=%.1f%%(%.1fGB/%.1fGB)",
        tag, usage["cpu_percent"], usage["ram_percent"], usage["ram_used_gb"], usage["ram_total_gb"],
    )
    return usage


def save_json(path: Path, data: Any) -> None:
    path.write_text(json.dumps(data, indent=2, default=str))


# ======================================================================
# DATACLASSES
# ======================================================================

@dataclass
class PreprocessingConfig:
    resize_height: int
    resize_width: int
    channels: int
    mean: List[float]
    std: List[float]
    resize_source: str
    mean_source: str
    std_source: str

    def to_dict(self) -> Dict[str, Any]:
        return asdict(self)


@dataclass
class InferenceError:
    error_type: str
    message: str

    def to_dict(self) -> Dict[str, str]:
        return asdict(self)


@dataclass
class PredictionResult:
    image_identifier: str
    success: bool
    predicted_diseases: List[str] = field(default_factory=list)
    confidence_scores: Dict[str, float] = field(default_factory=dict)
    probabilities: Dict[str, float] = field(default_factory=dict)
    thresholds_used: Dict[str, float] = field(default_factory=dict)
    inference_timestamp_utc: Optional[str] = None
    model_fingerprint_sha256: Optional[str] = None
    model_version: Optional[str] = None
    error: Optional[Dict[str, str]] = None

    def to_dict(self) -> Dict[str, Any]:
        return asdict(self)


# ======================================================================
# CONFIG RESOLUTION (no hardcoded assumptions unless nothing is recorded,
# and even then the fallback is logged loudly, never silent)
# ======================================================================

def resolve_class_names_and_thresholds(
    metadata_registry: Any, threshold_registry: Any, model_registry: Any, logger: logging.Logger,
) -> Tuple[List[str], Dict[str, float]]:
    metadata_dict = metadata_registry.to_dict()
    disease_metadata = metadata_dict["disease_metadata"]

    class_names: Optional[List[str]] = None
    source_key = None
    for key in ("classes", "class_names", "diseases", "labels"):
        if isinstance(disease_metadata.get(key), list):
            class_names = disease_metadata[key]
            source_key = key
            break

    if class_names is None:
        raise RuntimeError(
            "Unable to resolve canonical class ordering from METADATA_REGISTRY.disease_metadata "
            "(checked keys: classes, class_names, diseases, labels)."
        )
    if len(class_names) != model_registry.num_classes:
        raise RuntimeError(
            f"disease_metadata['{source_key}'] has {len(class_names)} classes but "
            f"MODEL_REGISTRY.num_classes={model_registry.num_classes}."
        )

    if threshold_registry.class_count == 0:
        logger.warning("RUNTIME THRESHOLD_REGISTRY is empty; defaulting every class threshold to 0.5.")
        thresholds = {name: 0.5 for name in class_names}
    else:
        missing = set(class_names) - set(threshold_registry.thresholds.keys())
        if missing:
            raise RuntimeError(f"THRESHOLD_REGISTRY is missing thresholds for classes: {sorted(missing)}")
        thresholds = {name: threshold_registry.thresholds[name] for name in class_names}

    logger.info(
        "RUNTIME class ordering resolved via disease_metadata['%s'] (%d classes).", source_key, len(class_names)
    )
    return class_names, thresholds


def resolve_preprocessing_config(
    metadata_registry: Any, export_registry: Any, logger: logging.Logger,
) -> PreprocessingConfig:
    metadata_dict = metadata_registry.to_dict()
    training_metadata = metadata_dict["training_metadata"]

    # Resize dimensions come from Stage 3's EXPORT_REGISTRY.model_signature --
    # already established as the single source of truth for input shape.
    # Never re-derived or re-guessed here.
    input_shape = export_registry.model_signature.input_shape  # [None, C, H, W]
    channels, height, width = input_shape[1], input_shape[2], input_shape[3]

    def _search(keys: List[str]) -> Tuple[Optional[str], Any]:
        for key in keys:
            val = training_metadata.get(key)
            if val not in (None, ""):
                return key, val
        for v in training_metadata.values():
            if isinstance(v, dict):
                for key in keys:
                    val = v.get(key)
                    if val not in (None, ""):
                        return key, val
        return None, None

    mean_key, mean_val = _search(["normalization_mean", "mean", "image_mean", "normalize_mean", "pixel_mean"])
    std_key, std_val = _search(["normalization_std", "std", "image_std", "normalize_std", "pixel_std"])

    if mean_val is not None and std_val is not None:
        mean = [float(x) for x in mean_val]
        std = [float(x) for x in std_val]
        mean_source = f"training_summary['{mean_key}']"
        std_source = f"training_summary['{std_key}']"
        logger.info("RUNTIME normalization resolved from training_summary.json: mean=%s std=%s", mean, std)
    else:
        mean = list(FALLBACK_IMAGENET_MEAN)
        std = list(FALLBACK_IMAGENET_STD)
        mean_source = "fallback_imagenet_default (no normalization recorded in training_summary.json)"
        std_source = "fallback_imagenet_default (no normalization recorded in training_summary.json)"
        logger.warning(
            "RUNTIME no explicit normalization mean/std found in training_summary.json; "
            "falling back to torchvision ImageNet defaults mean=%s std=%s. This is an explicit, "
            "logged fallback -- not a silent assumption.", mean, std,
        )

    config = PreprocessingConfig(
        resize_height=height, resize_width=width, channels=channels,
        mean=mean, std=std,
        resize_source="export_registry.model_signature.input_shape",
        mean_source=mean_source, std_source=std_source,
    )
    logger.info(
        "RUNTIME preprocessing config: resize=(%d,%d) channels=%d mean_source=%s std_source=%s",
        height, width, channels, mean_source, std_source,
    )
    return config


# ======================================================================
# INPUT VALIDATION / DECODING / PREPROCESSING
# ======================================================================

def validate_and_decode_image(image_path: str, expected_channels: int) -> Tuple[Optional[Image.Image], Optional[InferenceError]]:
    """Decodes and validates a single image file. Never raises -- every
    failure mode is returned as a structured InferenceError so batch
    processing can reject gracefully without aborting other images."""
    path = Path(image_path)

    if not path.exists() or not path.is_file():
        return None, InferenceError("missing_file", f"Image file does not exist: {image_path}")

    if path.suffix.lower() not in SUPPORTED_EXTENSIONS:
        return None, InferenceError("unsupported_extension", f"Unsupported file extension '{path.suffix}' for {image_path}")

    try:
        size = path.stat().st_size
    except OSError as exc:
        return None, InferenceError("corrupted_image", f"Unable to stat file: {exc}")

    if size == 0:
        return None, InferenceError("zero_byte_image", f"Image file is zero bytes: {image_path}")

    try:
        img = Image.open(path)
        img.verify()               # structural integrity check
        img = Image.open(path)     # verify() invalidates the file handle -- must reopen
    except (UnidentifiedImageError, OSError, ValueError) as exc:
        return None, InferenceError("corrupted_image", f"Failed to decode image: {exc}")

    source_band_count = len(img.getbands())
    if source_band_count not in ALLOWED_SOURCE_BAND_COUNTS:
        return None, InferenceError(
            "wrong_channel_count",
            f"Decoded image has {source_band_count} band(s); expected one of {sorted(ALLOWED_SOURCE_BAND_COUNTS)}.",
        )

    try:
        img = img.convert("RGB")
    except Exception as exc:  # noqa: BLE001 -- any conversion failure is a channel/format problem
        return None, InferenceError("wrong_channel_count", f"Failed to convert image to RGB: {exc}")

    if len(img.getbands()) != expected_channels:
        return None, InferenceError(
            "wrong_channel_count",
            f"Converted image has {len(img.getbands())} channel(s), expected {expected_channels}.",
        )

    return img, None


def preprocess_image(img: Image.Image, config: PreprocessingConfig) -> Tuple[Optional[torch.Tensor], Optional[InferenceError]]:
    """Resize -> normalize -> tensor conversion. Returns (tensor, None) on
    success or (None, InferenceError) on any dimensional inconsistency."""
    try:
        resized = img.resize((config.resize_width, config.resize_height), Image.BILINEAR)
        array = np.asarray(resized, dtype=np.float32) / 255.0  # H, W, C

        if array.ndim != 3 or array.shape[2] != config.channels:
            return None, InferenceError(
                "invalid_tensor_dimensions", f"Unexpected array shape after resize: {array.shape}"
            )

        tensor = torch.from_numpy(array).permute(2, 0, 1).contiguous()  # C, H, W
        mean = torch.tensor(config.mean, dtype=torch.float32).view(-1, 1, 1)
        std = torch.tensor(config.std, dtype=torch.float32).view(-1, 1, 1)
        tensor = (tensor - mean) / std

        expected_shape = (config.channels, config.resize_height, config.resize_width)
        if tuple(tensor.shape) != expected_shape:
            return None, InferenceError(
                "invalid_tensor_dimensions",
                f"Final tensor shape {tuple(tensor.shape)} does not match expected {expected_shape}.",
            )
        return tensor, None
    except Exception as exc:  # noqa: BLE001
        return None, InferenceError("invalid_tensor_dimensions", f"Preprocessing failed: {exc}")


# ======================================================================
# INFERENCE ENGINE
# ======================================================================

class InferenceEngine:
    """Production inference runtime.

    Consumes an already-reconstructed, already-validated model plus its
    Stage 2/3 registries. Performs no discovery, no checkpoint loading,
    and no rediscovery of any kind -- purely: validate -> decode ->
    preprocess -> forward pass -> threshold -> structured prediction.

    Deterministic: model runs in eval() mode with grad disabled, and
    preprocessing has no stochastic component.
    """

    def __init__(
        self,
        model: nn.Module,
        device: torch.device,
        class_names: List[str],
        thresholds: Dict[str, float],
        preprocessing_config: PreprocessingConfig,
        model_registry: Any,
        logger: logging.Logger,
    ) -> None:
        self.model = model
        self.model.eval()
        self.device = device
        self.class_names = class_names
        self.thresholds = thresholds
        self.preprocessing_config = preprocessing_config
        self.model_registry = model_registry
        self.model_fingerprint = model_registry.checkpoint_sha256
        self.model_version = f"{model_registry.backbone}-{model_registry.checkpoint_sha256[:12]}"
        self.logger = logger

    def predict_image(self, image_path: str, image_identifier: Optional[str] = None) -> PredictionResult:
        identifier = image_identifier if image_identifier is not None else str(image_path)
        return self.predict_batch([image_path], [identifier])[0]

    def predict_batch(
        self, image_paths: List[str], image_identifiers: Optional[List[str]] = None,
    ) -> List[PredictionResult]:
        if image_identifiers is None:
            image_identifiers = [str(p) for p in image_paths]
        if len(image_paths) != len(image_identifiers):
            raise ValueError(
                f"image_paths ({len(image_paths)}) and image_identifiers "
                f"({len(image_identifiers)}) length mismatch."
            )
        if len(image_paths) == 0:
            return []

        results: List[Optional[PredictionResult]] = [None] * len(image_paths)
        valid_indices: List[int] = []
        valid_tensors: List[torch.Tensor] = []

        for i, (path, identifier) in enumerate(zip(image_paths, image_identifiers)):
            img, decode_error = validate_and_decode_image(path, self.preprocessing_config.channels)
            if decode_error is not None:
                self.logger.warning(
                    "INFERENCE reject identifier=%s reason=%s: %s",
                    identifier, decode_error.error_type, decode_error.message,
                )
                results[i] = self._build_error_result(identifier, decode_error)
                continue

            tensor, preprocess_error = preprocess_image(img, self.preprocessing_config)
            if preprocess_error is not None:
                self.logger.warning(
                    "INFERENCE reject identifier=%s reason=%s: %s",
                    identifier, preprocess_error.error_type, preprocess_error.message,
                )
                results[i] = self._build_error_result(identifier, preprocess_error)
                continue

            valid_indices.append(i)
            valid_tensors.append(tensor)

        if valid_tensors:
            batch_tensor = torch.stack(valid_tensors, dim=0).to(self.device)
            with torch.no_grad():
                logits = self.model(batch_tensor)
                probabilities = torch.sigmoid(logits).detach().cpu()

            expected_shape = (len(valid_tensors), len(self.class_names))
            if tuple(probabilities.shape) != expected_shape:
                raise RuntimeError(
                    f"Model output shape {tuple(probabilities.shape)} does not match "
                    f"expected {expected_shape}."
                )

            timestamp = datetime.now(timezone.utc).isoformat()
            for local_idx, global_idx in enumerate(valid_indices):
                identifier = image_identifiers[global_idx]
                results[global_idx] = self._build_success_result(identifier, probabilities[local_idx], timestamp)

        return [r for r in results if r is not None]

    def _build_success_result(self, identifier: str, prob_row: torch.Tensor, timestamp: str) -> PredictionResult:
        probabilities_dict: Dict[str, float] = {}
        confidence_scores: Dict[str, float] = {}
        predicted_diseases: List[str] = []

        for class_name, prob in zip(self.class_names, prob_row.tolist()):
            prob = float(prob)
            if not (0.0 <= prob <= 1.0):
                raise RuntimeError(f"Probability out of range [0,1] for class '{class_name}': {prob}")
            probabilities_dict[class_name] = prob
            threshold = self.thresholds[class_name]
            if prob >= threshold:
                predicted_diseases.append(class_name)
                confidence_scores[class_name] = prob

        return PredictionResult(
            image_identifier=identifier,
            success=True,
            predicted_diseases=predicted_diseases,
            confidence_scores=confidence_scores,
            probabilities=probabilities_dict,
            thresholds_used=dict(self.thresholds),
            inference_timestamp_utc=timestamp,
            model_fingerprint_sha256=self.model_fingerprint,
            model_version=self.model_version,
            error=None,
        )

    def _build_error_result(self, identifier: str, error: InferenceError) -> PredictionResult:
        return PredictionResult(image_identifier=identifier, success=False, error=error.to_dict())


# ======================================================================
# SYNTHETIC VALIDATION FIXTURES
# ======================================================================

def generate_synthetic_sample_images(
    output_dir: Path, count: int, height: int, width: int, seed: int, logger: logging.Logger,
) -> List[Tuple[str, str]]:
    """Deterministically-seeded synthetic RGB images used ONLY to exercise
    the runtime pipeline end-to-end. Stage 4 must not rediscover the raw
    NIH image directory -- only Stage 1 ever resolved that path, and it is
    not propagated through ARTIFACT_REGISTRY/METADATA_REGISTRY. Synthetic
    images stand in for real deployment images here; the decode ->
    preprocess -> inference -> prediction wiring is identical either way."""
    output_dir.mkdir(parents=True, exist_ok=True)
    rng = np.random.RandomState(seed)
    paths: List[Tuple[str, str]] = []
    for i in range(count):
        array = rng.randint(0, 256, size=(height, width, 3), dtype=np.uint8)
        img = Image.fromarray(array, mode="RGB")
        path = output_dir / f"synthetic_sample_{i:02d}.png"
        img.save(path)
        paths.append((str(path), f"synthetic_sample_{i:02d}"))
    logger.info("RUNTIME generated %d synthetic sample image(s) (seed=%d) for validation.", count, seed)
    return paths


def generate_edge_case_samples(output_dir: Path, logger: logging.Logger) -> Dict[str, str]:
    """File-level failure fixtures that can be reliably produced with
    standard tooling. wrong_channel_count and invalid_tensor_dimensions
    are defensive checks in the code path (band-count and shape assertions)
    exercised structurally rather than via contrived fixtures, since
    standard image codecs do not naturally produce files that trigger
    them -- PIL's convert('RGB') handles every standard on-disk mode."""
    output_dir.mkdir(parents=True, exist_ok=True)
    cases: Dict[str, str] = {}

    zero_path = output_dir / "edge_zero_byte.png"
    zero_path.write_bytes(b"")
    cases["zero_byte_image"] = str(zero_path)

    corrupt_path = output_dir / "edge_corrupted.png"
    corrupt_path.write_bytes(b"not a real png file" * 5)
    cases["corrupted_image"] = str(corrupt_path)

    unsupported_path = output_dir / "edge_unsupported.txt"
    unsupported_path.write_text("not an image")
    cases["unsupported_extension"] = str(unsupported_path)

    cases["missing_file"] = str(output_dir / "does_not_exist.png")

    logger.info("RUNTIME generated %d edge-case fixture(s) for graceful-rejection testing.", len(cases))
    return cases

def probabilities_equal(
    a: Dict[str, float], b: Dict[str, float], rel_tol: float = 1e-6, abs_tol: float = 1e-8,
) -> Tuple[bool, Optional[str], float, float]:
    """Tolerance-based comparison of two class->probability dictionaries.
    Exact IEEE float equality is not appropriate for GPU inference: individual
    vs. batched CUDA execution can legitimately differ by sub-1e-10 floating-
    point roundoff (kernel batching/reduction order), which is not a genuine
    inference bug. Returns (equal, worst_class, max_abs_diff, max_rel_diff)."""
    if set(a.keys()) != set(b.keys()):
        return False, None, float("inf"), float("inf")

    equal = True
    max_abs_diff = 0.0
    max_rel_diff = 0.0
    worst_class: Optional[str] = None

    for cls in a:
        val_a, val_b = a[cls], b[cls]
        if not math.isclose(val_a, val_b, rel_tol=rel_tol, abs_tol=abs_tol):
            equal = False
        abs_diff = abs(val_a - val_b)
        rel_diff = abs_diff / max(abs(val_a), abs(val_b), abs_tol)
        if abs_diff >= max_abs_diff:
            max_abs_diff, max_rel_diff, worst_class = abs_diff, rel_diff, cls

    return equal, worst_class, max_abs_diff, max_rel_diff


# ======================================================================
# RUNTIME VALIDATION
# ======================================================================

def run_runtime_validation(
    engine: InferenceEngine,
    sample_paths: List[Tuple[str, str]],
    edge_cases: Dict[str, str],
    logger: logging.Logger,
) -> Tuple[Dict[str, Any], List[PredictionResult]]:
    checks: Dict[str, bool] = {}
    fatal_errors: List[str] = []
    warnings: List[str] = []

    # -- preprocessing consistency: identical output across repeated runs --
    img1, _ = validate_and_decode_image(sample_paths[0][0], engine.preprocessing_config.channels)
    tensor1, _ = preprocess_image(img1, engine.preprocessing_config)
    img2, _ = validate_and_decode_image(sample_paths[0][0], engine.preprocessing_config.channels)
    tensor2, _ = preprocess_image(img2, engine.preprocessing_config)
    checks["preprocessing_consistency"] = tensor1 is not None and tensor2 is not None and torch.equal(tensor1, tensor2)
    if not checks["preprocessing_consistency"]:
        fatal_errors.append("Preprocessing is not deterministic across repeated runs on the same image.")

    identifiers = [pid for _, pid in sample_paths]
    paths_only = [p for p, _ in sample_paths]
    batch_results = engine.predict_batch(paths_only, identifiers)
    successful = [r for r in batch_results if r.success]

    checks["output_tensor_shape_correct"] = all(len(r.probabilities) == len(engine.class_names) for r in successful)
    if not checks["output_tensor_shape_correct"]:
        fatal_errors.append("One or more predictions do not contain a probability for every class.")

    checks["probability_range_valid"] = all(0.0 <= p <= 1.0 for r in successful for p in r.probabilities.values())
    if not checks["probability_range_valid"]:
        fatal_errors.append("One or more predicted probabilities fall outside [0,1].")

    checks["threshold_registry_compatible"] = all(
        set(r.thresholds_used.keys()) == set(engine.class_names) for r in successful
    )
    if not checks["threshold_registry_compatible"]:
        fatal_errors.append("Thresholds used in predictions do not cover the full class set.")

    checks["class_ordering_consistent"] = all(list(r.probabilities.keys()) == engine.class_names for r in successful)
    if not checks["class_ordering_consistent"]:
        fatal_errors.append("Class ordering in prediction output does not match canonical class ordering.")

    required_fields = [
        "image_identifier", "predicted_diseases", "confidence_scores", "probabilities",
        "thresholds_used", "inference_timestamp_utc", "model_fingerprint_sha256", "model_version",
    ]
    checks["prediction_formatting_valid"] = all(
        all(getattr(r, f, None) is not None for f in required_fields) for r in successful
    )
    if not checks["prediction_formatting_valid"]:
        fatal_errors.append("One or more successful predictions are missing required structured fields.")

    checks["input_ordering_preserved"] = [r.image_identifier for r in batch_results] == identifiers
    if not checks["input_ordering_preserved"]:
        fatal_errors.append("Batch output ordering does not match input ordering.")

    # -- batch consistency: individual predict vs. batched predict must match --
    individual_results = [engine.predict_image(p, pid) for p, pid in sample_paths]
    batch_consistent = True
    worst_overall_abs_diff = 0.0
    worst_overall_rel_diff = 0.0
    worst_overall_class: Optional[str] = None
    worst_overall_image: Optional[str] = None

    for ind, bat in zip(individual_results, batch_results):
        if ind.success != bat.success:
            batch_consistent = False
            continue
        if not ind.success:
            continue
        equal, worst_class, max_abs_diff, max_rel_diff = probabilities_equal(
            ind.probabilities, bat.probabilities, rel_tol=1e-6, abs_tol=1e-8,
        )
        if not equal:
            batch_consistent = False
        if max_abs_diff >= worst_overall_abs_diff:
            worst_overall_abs_diff = max_abs_diff
            worst_overall_rel_diff = max_rel_diff
            worst_overall_class = worst_class
            worst_overall_image = ind.image_identifier

    checks["batch_consistency"] = batch_consistent
    if not batch_consistent:
        fatal_errors.append(
            "Predicting an image individually vs. as part of a batch produced different results "
            f"(max_abs_diff={worst_overall_abs_diff:.3e}, max_rel_diff={worst_overall_rel_diff:.3e}, "
            f"worst_class={worst_overall_class}, worst_image={worst_overall_image})."
        )
    # -- graceful rejection of invalid inputs --
    edge_case_details: Dict[str, Dict[str, Any]] = {}
    edge_case_correct = True
    for expected_error_type, path in edge_cases.items():
        result = engine.predict_image(path, f"edge_{expected_error_type}")
        actual_error_type = result.error.get("error_type") if result.error else None
        edge_case_details[expected_error_type] = {"success": result.success, "error_type": actual_error_type}
        if result.success or actual_error_type != expected_error_type:
            edge_case_correct = False
    checks["graceful_error_handling"] = edge_case_correct
    if not edge_case_correct:
        fatal_errors.append(f"One or more edge cases were not rejected as expected: {edge_case_details}")

    for w in warnings:
        logger.warning("VALIDATION WARNING: %s", w)
    for e in fatal_errors:
        logger.error("VALIDATION FAILURE: %s", e)

    validation_result = {
        "checks": checks,
        "warnings": warnings,
        "fatal_errors": fatal_errors,
        "passed": len(fatal_errors) == 0,
        "edge_case_details": edge_case_details,
    }
    return validation_result, batch_results


# ======================================================================
# ENGINEERING VALIDATION (overall stage gate)
# ======================================================================

def run_stage04_engineering_validation(
    runtime_validation_result: Dict[str, Any],
    model: nn.Module,
    model_registry: Any,
    artifact_registry: Any,
    threshold_registry: Any,
    class_names: List[str],
    preprocessing_config: PreprocessingConfig,
    logger: logging.Logger,
) -> Dict[str, Any]:
    checks: Dict[str, bool] = {}
    fatal_errors: List[str] = []
    warnings: List[str] = []

    checks["artifact_registry_consumed"] = artifact_registry.total_artifacts > 0
    if not checks["artifact_registry_consumed"]:
        fatal_errors.append("ARTIFACT_REGISTRY passed into Stage 4 has zero artifacts recorded.")

    checks["runtime_validation_passed"] = runtime_validation_result["passed"]
    if not checks["runtime_validation_passed"]:
        fatal_errors.append(f"Runtime validation failed: {runtime_validation_result['fatal_errors']}")

    checks["class_count_matches_model"] = len(class_names) == model_registry.num_classes
    if not checks["class_count_matches_model"]:
        fatal_errors.append("Resolved class count does not match MODEL_REGISTRY.num_classes.")

    checks["threshold_count_matches_model"] = threshold_registry.class_count in (0, model_registry.num_classes)
    if not checks["threshold_count_matches_model"]:
        fatal_errors.append("THRESHOLD_REGISTRY class_count does not match MODEL_REGISTRY.num_classes.")

    checks["preprocessing_config_valid"] = (
        preprocessing_config.resize_height > 0 and preprocessing_config.resize_width > 0
        and preprocessing_config.channels > 0
        and len(preprocessing_config.mean) == preprocessing_config.channels
        and len(preprocessing_config.std) == preprocessing_config.channels
    )
    if not checks["preprocessing_config_valid"]:
        fatal_errors.append("Resolved PreprocessingConfig is structurally invalid.")

    checks["model_in_eval_mode"] = not model.training
    if not checks["model_in_eval_mode"]:
        fatal_errors.append("Model is not in eval() mode at inference time.")

    checks["model_fingerprint_recorded"] = bool(model_registry.checkpoint_sha256)
    if not checks["model_fingerprint_recorded"]:
        fatal_errors.append("MODEL_REGISTRY.checkpoint_sha256 is missing; predictions cannot be fingerprinted.")

    for w in warnings:
        logger.warning("VALIDATION WARNING: %s", w)
    for e in fatal_errors:
        logger.error("VALIDATION FAILURE: %s", e)

    return {"checks": checks, "warnings": warnings, "fatal_errors": fatal_errors, "passed": len(fatal_errors) == 0}


# ======================================================================
# MAIN STAGE ENTRY POINT
# ======================================================================

def run_stage04(
    reconstructed_model: nn.Module,
    artifact_registry: Any,
    model_registry: Any,
    metadata_registry: Any,
    threshold_registry: Any,
    config_registry: Any,
    export_registry: Any,
) -> Dict[str, Any]:
    start_time = time.time()

    stage_root = OUTPUT_ROOT / STAGE4_DIR_NAME
    dirs = {"stage_root": stage_root, "logs": stage_root / "logs", "samples": stage_root / "samples"}
    for p in dirs.values():
        p.mkdir(parents=True, exist_ok=True)

    logger = build_logger(dirs["logs"])
    logger.info("START Sprint05-Stage04 Production Inference Runtime")
    log_resources(logger, "START")

    device = torch.device(config_registry.device)
    reconstructed_model.eval()
    reconstructed_model.to(device)
    logger.info("RUNTIME model set to eval() on device=%s", device)

    class_names, thresholds = resolve_class_names_and_thresholds(
        metadata_registry, threshold_registry, model_registry, logger
    )
    preprocessing_config = resolve_preprocessing_config(metadata_registry, export_registry, logger)

    engine = InferenceEngine(
        model=reconstructed_model, device=device, class_names=class_names, thresholds=thresholds,
        preprocessing_config=preprocessing_config, model_registry=model_registry, logger=logger,
    )
    logger.info(
        "RUNTIME InferenceEngine initialized: model_version=%s fingerprint=%s",
        engine.model_version, engine.model_fingerprint,
    )

    sample_paths = generate_synthetic_sample_images(
        dirs["samples"], NUM_SYNTHETIC_SAMPLES,
        preprocessing_config.resize_height, preprocessing_config.resize_width,
        SYNTHETIC_SAMPLE_SEED, logger,
    )
    edge_cases = generate_edge_case_samples(dirs["samples"] / "edge_cases", logger)

    runtime_validation_result, sample_predictions = run_runtime_validation(engine, sample_paths, edge_cases, logger)
    logger.info("RUNTIME VALIDATION passed=%s", runtime_validation_result["passed"])

    engineering_validation = run_stage04_engineering_validation(
        runtime_validation_result=runtime_validation_result, model=reconstructed_model,
        model_registry=model_registry, artifact_registry=artifact_registry,
        threshold_registry=threshold_registry, class_names=class_names,
        preprocessing_config=preprocessing_config, logger=logger,
    )

    if not engineering_validation["passed"]:
        log_resources(logger, "FINISH-FAILED")
        elapsed = time.time() - start_time
        logger.error("FINISH Sprint05-Stage04 status=FAILED elapsed=%.2fs", elapsed)
        raise RuntimeError(
            "Sprint05 Stage04 engineering validation FAILED (fail-fast):\n  - "
            + "\n  - ".join(engineering_validation["fatal_errors"])
        )

    log_resources(logger, "FINISH")
    elapsed = time.time() - start_time

    # ---- Persist outputs ------------------------------------------------
    runtime_configuration = {
        "device": str(device),
        "class_names": class_names,
        "thresholds": thresholds,
        "preprocessing_config": preprocessing_config.to_dict(),
        "model_version": engine.model_version,
        "model_fingerprint_sha256": engine.model_fingerprint,
        "supported_extensions": sorted(SUPPORTED_EXTENSIONS),
    }
    save_json(dirs["stage_root"] / "runtime_configuration.json", runtime_configuration)
    save_json(dirs["stage_root"] / "runtime_validation.json", runtime_validation_result)
    save_json(dirs["stage_root"] / "sample_predictions.json", [p.to_dict() for p in sample_predictions])
    save_json(dirs["stage_root"] / "engineering_validation.json", engineering_validation)

    warnings_count = len(engineering_validation["warnings"]) + len(runtime_validation_result["warnings"])

    runtime_summary = {
        "stage": STAGE4_DIR_NAME,
        "status": "OK",
        "elapsed_seconds": round(elapsed, 3),
        "model_version": engine.model_version,
        "model_fingerprint_sha256": engine.model_fingerprint,
        "num_classes": len(class_names),
        "class_names": class_names,
        "batch_support": True,
        "num_synthetic_samples_validated": len(sample_paths),
        "runtime_validation_passed": runtime_validation_result["passed"],
        "engineering_checks_passed": sum(engineering_validation["checks"].values()),
        "engineering_checks_total": len(engineering_validation["checks"]),
        "warnings_count": warnings_count,
        "output_directory": str(dirs["stage_root"]),
    }
    save_json(dirs["stage_root"] / "runtime_summary.json", runtime_summary)

    stage04_summary = {
        "stage": STAGE4_DIR_NAME,
        "status": "OK",
        "elapsed_seconds": round(elapsed, 3),
        "engineering_checks_passed": sum(engineering_validation["checks"].values()),
        "engineering_checks_total": len(engineering_validation["checks"]),
        "warnings_count": warnings_count,
        "output_directory": str(dirs["stage_root"]),
    }
    save_json(dirs["stage_root"] / "stage04_summary.json", stage04_summary)

    logger.info("FINISH Sprint05-Stage04 status=OK elapsed=%.2fs", elapsed)

    # ---- Console report ---------------------------------------------------
    print("=" * 70)
    print("SPRINT05 — STAGE04")
    print("PRODUCTION INFERENCE RUNTIME")
    print("=" * 70)
    print("Runtime initialized  : OK")
    print(f"Model                : {model_registry.backbone} ({engine.model_version})")
    print("Batch support        : YES (arbitrary batch size)")
    print(f"Threshold registry   : {'LOADED' if threshold_registry.class_count > 0 else 'DEFAULTED (0.5)'} ({len(class_names)} classes)")
    print(f"Validation checks    : {sum(engineering_validation['checks'].values())}/{len(engineering_validation['checks'])}")
    print(f"Warnings             : {warnings_count}")
    print(f"Output directory     : {dirs['stage_root']}")
    print("Stage 4 : OK")

    return {
        "INFERENCE_ENGINE": engine,
        "RUNTIME_CONFIGURATION": runtime_configuration,
        "validation": engineering_validation,
        "runtime_validation": runtime_validation_result,
        "summary": stage04_summary,
    }


if __name__ == "__main__":
    _stage04_result = run_stage04(
        reconstructed_model=RECONSTRUCTED_MODEL,
        artifact_registry=ARTIFACT_REGISTRY,
        model_registry=MODEL_REGISTRY,
        metadata_registry=METADATA_REGISTRY,
        threshold_registry=THRESHOLD_REGISTRY,
        config_registry=CONFIG_REGISTRY,
        export_registry=EXPORT_REGISTRY,
    )
    INFERENCE_ENGINE = _stage04_result["INFERENCE_ENGINE"]
    RUNTIME_CONFIGURATION = _stage04_result["RUNTIME_CONFIGURATION"]

2026-07-08 10:27:32 | INFO     | START Sprint05-Stage04 Production Inference Runtime
2026-07-08 10:27:33 | INFO     | RESOURCES [START] cpu=0.0% ram=7.4%(1.9GB/31.4GB)
2026-07-08 10:27:33 | INFO     | RUNTIME model set to eval() on device=cuda
2026-07-08 10:27:33 | INFO     | RUNTIME class ordering resolved via disease_metadata['diseases'] (14 classes).
2026-07-08 10:27:33 | WARNING  | RUNTIME no explicit normalization mean/std found in training_summary.json; falling back to torchvision ImageNet defaults mean=[0.485, 0.456, 0.406] std=[0.229, 0.224, 0.225]. This is an explicit, logged fallback -- not a silent assumption.
2026-07-08 10:27:33 | INFO     | RUNTIME preprocessing config: resize=(224,224) channels=3 mean_source=fallback_imagenet_default (no normalization recorded in training_summary.json) std_source=fallback_imagenet_default (no normalization recorded in training_summary.json)
2026-07-08 10:27:33 | INFO     | RUNTIME InferenceEngine initialized: model_version=densenet121-74a

/tmp/ipykernel_195/3542953323.py:499: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  img = Image.fromarray(array, mode="RGB")


2026-07-08 10:27:33 | WARNING  | INFERENCE reject identifier=edge_zero_byte_image reason=zero_byte_image: Image file is zero bytes: /kaggle/working/sprint05_deployment/stage04_runtime/samples/edge_cases/edge_zero_byte.png
2026-07-08 10:27:33 | WARNING  | INFERENCE reject identifier=edge_corrupted_image reason=corrupted_image: Failed to decode image: cannot identify image file '/kaggle/working/sprint05_deployment/stage04_runtime/samples/edge_cases/edge_corrupted.png'
2026-07-08 10:27:33 | WARNING  | INFERENCE reject identifier=edge_unsupported_extension reason=unsupported_extension: Unsupported file extension '.txt' for /kaggle/working/sprint05_deployment/stage04_runtime/samples/edge_cases/edge_unsupported.txt
2026-07-08 10:27:33 | WARNING  | INFERENCE reject identifier=edge_missing_file reason=missing_file: Image file does not exist: /kaggle/working/sprint05_deployment/stage04_runtime/samples/edge_cases/does_not_exist.png
2026-07-08 10:27:33 | INFO     | RUNTIME VALIDATION passed=True


In [9]:
"""
VisionServeAI - Sprint 05
Stage 5: Performance & Robustness Validation
======================================================================
Single-responsibility stage: measures DEPLOYMENT quality (not model
quality) of the Stage 4 InferenceEngine -- latency, throughput, memory,
crash-resistance, cross-runtime numerical parity -- and gates on a
fixed set of engineering checks.

STRICT SCOPE:
    - No GradCAM. No explainability. No REST API. No packaging.
    - No report generation beyond the required JSON/CSV artifacts.
    - No metrics recalculation. No threshold optimization. No calibration.
    - No model export (TorchScript/ONNX artifacts are CONSUMED from
      EXPORT_REGISTRY, never re-exported).
    - Stages 1-4 are FROZEN. INFERENCE_ENGINE, RECONSTRUCTED_MODEL,
      ARTIFACT_REGISTRY, MODEL_REGISTRY, METADATA_REGISTRY,
      THRESHOLD_REGISTRY, CONFIG_REGISTRY, EXPORT_REGISTRY are consumed
      as the in-memory objects earlier stages produced -- never reloaded,
      never rediscovered, never rebuilt.
======================================================================
"""

from __future__ import annotations

import csv
import gc
import json
import logging
import random
import sys
import time
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import psutil
import torch
from PIL import Image, ImageFilter

try:
    import onnxruntime as ort
except ImportError:
    ort = None

# ======================================================================
# CONSTANTS
# ======================================================================

OUTPUT_ROOT = Path("/kaggle/working/sprint05_deployment")
STAGE5_DIR_NAME = "stage05_validation"

BATCH_SIZES: List[int] = [1, 2, 4, 8, 16, 32]
WARMUP_ITERATIONS = 3
BENCHMARK_ITERATIONS = 15

STRESS_TEST_CALLS = 120
STRESS_TEST_SEED = 123

NUMERICAL_STABILITY_SAMPLES = 5
NUMERICAL_STABILITY_BATCH_SIZE = 4
MAX_ABS_ERROR_TOLERANCE = 1e-3
MAX_RELATIVE_ERROR_TOLERANCE = 1e-2

MEMORY_LEAK_THRESHOLD_MB = 50.0
GPU_LEAK_THRESHOLD_MB = 20.0

SYNTHETIC_POOL_SEED = 777
VERY_LARGE_IMAGE_SIDE = 3000

try:
    _OOM_ERROR_TYPES: Tuple[type, ...] = (torch.cuda.OutOfMemoryError, RuntimeError)  # type: ignore[attr-defined]
except AttributeError:
    _OOM_ERROR_TYPES = (RuntimeError,)


# ======================================================================
# LOGGING (same format/pattern as Stage 1-4 -- self-contained, distinct
# logger name identifies this stage)
# ======================================================================

def build_logger(log_dir: Path) -> logging.Logger:
    log_dir.mkdir(parents=True, exist_ok=True)
    logger = logging.getLogger("sprint05.stage05")
    logger.setLevel(logging.INFO)
    logger.handlers.clear()
    logger.propagate = False

    fmt = logging.Formatter(
        fmt="%(asctime)s | %(levelname)-8s | %(message)s",
        datefmt="%Y-%m-%d %H:%M:%S",
    )

    file_handler = logging.FileHandler(log_dir / "stage05_validation.log", mode="w")
    file_handler.setFormatter(fmt)
    logger.addHandler(file_handler)

    stream_handler = logging.StreamHandler(sys.stdout)
    stream_handler.setFormatter(fmt)
    logger.addHandler(stream_handler)

    return logger


def get_resource_usage(device: torch.device) -> Dict[str, Any]:
    vm = psutil.virtual_memory()
    usage: Dict[str, Any] = {
        "cpu_percent": psutil.cpu_percent(interval=0.2),
        "ram_percent": vm.percent,
        "ram_used_gb": round(vm.used / (1024 ** 3), 2),
        "ram_total_gb": round(vm.total / (1024 ** 3), 2),
        "process_rss_mb": round(psutil.Process().memory_info().rss / (1024 ** 2), 2),
    }
    if device.type == "cuda":
        usage["gpu_memory_allocated_mb"] = round(torch.cuda.memory_allocated(device) / (1024 ** 2), 2)
        usage["gpu_memory_reserved_mb"] = round(torch.cuda.memory_reserved(device) / (1024 ** 2), 2)
    return usage


def log_resources(logger: logging.Logger, tag: str, device: torch.device) -> Dict[str, Any]:
    usage = get_resource_usage(device)
    logger.info(
        "RESOURCES [%s] cpu=%.1f%% ram=%.1f%% rss=%.1fMB",
        tag, usage["cpu_percent"], usage["ram_percent"], usage["process_rss_mb"],
    )
    return usage


def save_json(path: Path, data: Any) -> None:
    path.write_text(json.dumps(data, indent=2, default=str))


# ======================================================================
# SAFE EXECUTION HARNESS
# ======================================================================

def safe_execute(fn, *args, **kwargs) -> Tuple[str, Any, Optional[str], Optional[str]]:
    """Never lets an exception escape. Returns (status, result, message, exc_type).
    status is 'ok' or 'crash'. Used so Stage 5's own harness never dies while
    probing the frozen Stage 4 runtime with adversarial/out-of-contract inputs."""
    try:
        return "ok", fn(*args, **kwargs), None, None
    except Exception as exc:  # noqa: BLE001 -- intentionally broad: this IS the safety net
        return "crash", None, str(exc), type(exc).__name__


def _forward_no_grad(model: torch.nn.Module, tensor: torch.Tensor) -> torch.Tensor:
    with torch.no_grad():
        return model(tensor)


# ======================================================================
# DATACLASSES
# ======================================================================

@dataclass
class LatencyStats:
    count: int
    mean_ms: float
    min_ms: float
    max_ms: float
    std_ms: float
    p95_ms: float

    def to_dict(self) -> Dict[str, Any]:
        return asdict(self)


def compute_latency_stats(samples_ms: List[float]) -> LatencyStats:
    arr = np.asarray(samples_ms, dtype=np.float64)
    return LatencyStats(
        count=int(arr.size),
        mean_ms=float(np.mean(arr)),
        min_ms=float(np.min(arr)),
        max_ms=float(np.max(arr)),
        std_ms=float(np.std(arr)),
        p95_ms=float(np.percentile(arr, 95)),
    )


@dataclass
class BatchBenchmarkResult:
    batch_size: int
    warm_latency: Optional[LatencyStats]
    cold_start_ms: Optional[float]
    throughput_images_per_sec: Optional[float]
    fps: Optional[float]
    gpu_memory_allocated_mb: Optional[float]
    gpu_memory_reserved_mb: Optional[float]
    cpu_rss_mb: Optional[float]
    peak_gpu_memory_mb: Optional[float]
    result_count_correct: Optional[bool]
    oom: bool
    error: Optional[str]

    def to_dict(self) -> Dict[str, Any]:
        d = asdict(self)
        return d


# ======================================================================
# SECTION 1: PERFORMANCE BENCHMARK
# ======================================================================

def benchmark_batch_size(
    engine: Any, pool_paths: List[str], batch_size: int, device: torch.device, logger: logging.Logger,
) -> BatchBenchmarkResult:
    paths = pool_paths[:batch_size]
    identifiers = [f"bench_bs{batch_size}_{i}" for i in range(batch_size)]

    if device.type == "cuda":
        torch.cuda.reset_peak_memory_stats(device)
        torch.cuda.synchronize()

    try:
        t0 = time.perf_counter()
        _ = engine.predict_batch(paths, identifiers)
        if device.type == "cuda":
            torch.cuda.synchronize()
        cold_ms = (time.perf_counter() - t0) * 1000.0
    except _OOM_ERROR_TYPES as exc:
        if "out of memory" in str(exc).lower():
            if device.type == "cuda":
                torch.cuda.empty_cache()
            gc.collect()
            logger.warning("BENCHMARK batch_size=%d OOM on first call: %s", batch_size, exc)
            return BatchBenchmarkResult(
                batch_size=batch_size, warm_latency=None, cold_start_ms=None,
                throughput_images_per_sec=None, fps=None, gpu_memory_allocated_mb=None,
                gpu_memory_reserved_mb=None, cpu_rss_mb=None, peak_gpu_memory_mb=None,
                result_count_correct=None, oom=True, error=str(exc),
            )
        raise

    for _ in range(WARMUP_ITERATIONS):
        engine.predict_batch(paths, identifiers)
    if device.type == "cuda":
        torch.cuda.synchronize()

    samples: List[float] = []
    results = []
    for _ in range(BENCHMARK_ITERATIONS):
        t0 = time.perf_counter()
        results = engine.predict_batch(paths, identifiers)
        if device.type == "cuda":
            torch.cuda.synchronize()
        samples.append((time.perf_counter() - t0) * 1000.0)

    stats = compute_latency_stats(samples)
    throughput = batch_size / (stats.mean_ms / 1000.0)

    gpu_alloc = gpu_reserved = gpu_peak = None
    if device.type == "cuda":
        gpu_alloc = torch.cuda.memory_allocated(device) / (1024 ** 2)
        gpu_reserved = torch.cuda.memory_reserved(device) / (1024 ** 2)
        gpu_peak = torch.cuda.max_memory_allocated(device) / (1024 ** 2)

    cpu_rss = psutil.Process().memory_info().rss / (1024 ** 2)
    result_count_correct = len(results) == batch_size

    return BatchBenchmarkResult(
        batch_size=batch_size,
        warm_latency=stats,
        cold_start_ms=cold_ms,
        throughput_images_per_sec=throughput,
        fps=throughput,
        gpu_memory_allocated_mb=gpu_alloc,
        gpu_memory_reserved_mb=gpu_reserved,
        cpu_rss_mb=cpu_rss,
        peak_gpu_memory_mb=gpu_peak,
        result_count_correct=result_count_correct,
        oom=False,
        error=None,
    )


# ======================================================================
# SECTION 2: RUNTIME STRESS TESTING
# ======================================================================

def run_stress_test(
    engine: Any, pool_paths: List[str], device: torch.device, logger: logging.Logger,
) -> Dict[str, Any]:
    rng = random.Random(STRESS_TEST_SEED)
    reference_path = pool_paths[0]

    baseline_probs: Optional[Dict[str, float]] = None
    crashes: List[Dict[str, Any]] = []
    mismatches: List[Dict[str, Any]] = []
    memory_samples: List[Dict[str, Any]] = []
    outcomes = {"success": 0, "structured_failure": 0, "runtime_exception": 0}
    schema_keysets = set()

    cpu_before = psutil.Process().memory_info().rss / (1024 ** 2)
    gpu_before = torch.cuda.memory_allocated(device) / (1024 ** 2) if device.type == "cuda" else None

    t_start = time.perf_counter()
    for i in range(STRESS_TEST_CALLS):
        use_ref = (i % 10 == 0)
        path = reference_path if use_ref else rng.choice(pool_paths)
        status, result, err_msg, err_type = safe_execute(engine.predict_image, path, f"stress_{i}")

        if status == "crash":
            outcomes["runtime_exception"] += 1
            crashes.append({"call_index": i, "path": path, "error_type": err_type, "error": err_msg})
            continue

        if result.success:
            outcomes["success"] += 1
            schema_keysets.add(tuple(sorted(result.to_dict().keys())))
            if use_ref:
                if baseline_probs is None:
                    baseline_probs = result.probabilities
                else:
                    equal, worst_class, max_abs, max_rel = probabilities_equal(baseline_probs, result.probabilities)
                    if not equal:
                        mismatches.append({
                            "call_index": i, "max_abs_diff": max_abs,
                            "max_rel_diff": max_rel, "worst_class": worst_class,
                        })
        else:
            outcomes["structured_failure"] += 1

        if i % 20 == 0:
            rss = psutil.Process().memory_info().rss / (1024 ** 2)
            gpu = torch.cuda.memory_allocated(device) / (1024 ** 2) if device.type == "cuda" else None
            memory_samples.append({"call_index": i, "cpu_rss_mb": rss, "gpu_allocated_mb": gpu})

    elapsed = time.perf_counter() - t_start

    gc.collect()
    if device.type == "cuda":
        torch.cuda.empty_cache()
    cpu_after = psutil.Process().memory_info().rss / (1024 ** 2)
    gpu_after = torch.cuda.memory_allocated(device) / (1024 ** 2) if device.type == "cuda" else None

    cpu_growth = cpu_after - cpu_before
    gpu_growth = (gpu_after - gpu_before) if gpu_before is not None else None

    return {
        "total_calls": STRESS_TEST_CALLS,
        "elapsed_seconds": elapsed,
        "calls_per_second": (STRESS_TEST_CALLS / elapsed) if elapsed > 0 else None,
        "outcomes": outcomes,
        "crashes": crashes,
        "determinism_mismatches": mismatches,
        "deterministic": len(mismatches) == 0,
        "memory_samples": memory_samples,
        "cpu_rss_before_mb": cpu_before,
        "cpu_rss_after_mb": cpu_after,
        "cpu_rss_growth_mb": cpu_growth,
        "gpu_allocated_before_mb": gpu_before,
        "gpu_allocated_after_mb": gpu_after,
        "gpu_allocated_growth_mb": gpu_growth,
        "memory_stable": (cpu_growth < MEMORY_LEAK_THRESHOLD_MB)
                          and (gpu_growth is None or gpu_growth < GPU_LEAK_THRESHOLD_MB),
        "no_crashes": len(crashes) == 0,
        "schema_stable": len(schema_keysets) <= 1,
        "observed_schema_keysets": [list(k) for k in schema_keysets],
    }


# ======================================================================
# SECTION 3: ROBUSTNESS VALIDATION (expands Stage 4's edge-case set)
# ======================================================================

def build_robustness_fixtures(out_dir: Path, preprocessing_config: Any, logger: logging.Logger) -> Dict[str, Optional[str]]:
    out_dir.mkdir(parents=True, exist_ok=True)
    h, w = preprocessing_config.resize_height, preprocessing_config.resize_width
    fixtures: Dict[str, Optional[str]] = {}

    p = out_dir / "tiny.png"
    Image.fromarray(np.random.RandomState(1).randint(0, 256, (2, 2, 3), dtype=np.uint8), "RGB").save(p)
    fixtures["tiny_image"] = str(p)

    p = out_dir / "very_large.png"
    big = np.random.RandomState(2).randint(0, 256, (VERY_LARGE_IMAGE_SIDE, VERY_LARGE_IMAGE_SIDE, 3), dtype=np.uint8)
    Image.fromarray(big, "RGB").save(p)
    fixtures["very_large_image"] = str(p)

    p = out_dir / "grayscale.png"
    gray = np.random.RandomState(3).randint(0, 256, (h, w), dtype=np.uint8)
    Image.fromarray(gray, "L").save(p)
    fixtures["grayscale_image"] = str(p)

    p = out_dir / "rgba.png"
    rgba = np.random.RandomState(4).randint(0, 256, (h, w, 4), dtype=np.uint8)
    Image.fromarray(rgba, "RGBA").save(p)
    fixtures["rgba_image"] = str(p)

    p = out_dir / "empty.png"
    p.write_bytes(b"")
    fixtures["empty_image"] = str(p)

    p = out_dir / "random_noise.png"
    noise = np.random.RandomState(5).randint(0, 256, (137, 251, 3), dtype=np.uint8)
    Image.fromarray(noise, "RGB").save(p)
    fixtures["random_noise_image"] = str(p)

    base_arr = np.random.RandomState(6).randint(0, 256, (h, w, 3), dtype=np.uint8)
    base_img = Image.fromarray(base_arr, "RGB")

    p = out_dir / "rotated.png"
    base_img.rotate(45, expand=True).save(p)
    fixtures["rotated_image"] = str(p)

    p = out_dir / "blurred.png"
    base_img.filter(ImageFilter.GaussianBlur(radius=5)).save(p)
    fixtures["blurred_image"] = str(p)

    for fmt, ext in [("JPEG", ".jpg"), ("PNG", ".png"), ("BMP", ".bmp"), ("WEBP", ".webp")]:
        p = out_dir / f"format_{fmt.lower()}{ext}"
        try:
            base_img.save(p, format=fmt)
            fixtures[f"format_{fmt.lower()}"] = str(p)
        except Exception as exc:  # noqa: BLE001
            fixtures[f"format_{fmt.lower()}"] = None
            logger.warning("ROBUSTNESS could not create %s fixture in this environment: %s", fmt, exc)

    return fixtures


def _classify_prediction_outcome(status: str, result: Any, err_msg: Optional[str], err_type: Optional[str]) -> Tuple[str, Optional[str], str]:
    if status == "crash":
        return "runtime_exception", err_type, err_msg or ""
    if result is not None and getattr(result, "success", False):
        return "success", None, "prediction produced successfully"
    if result is not None and not result.success:
        error_type = result.error.get("error_type") if result.error else None
        message = result.error.get("message") if result.error else ""
        return "structured_failure", error_type, message
    return "unknown", None, "no result and no exception"


def run_robustness_validation(
    engine: Any, pool_paths: List[str], dirs: Dict[str, Path], logger: logging.Logger,
) -> Dict[str, Any]:
    fixtures = build_robustness_fixtures(dirs["robustness_images"], engine.preprocessing_config, logger)
    cases: List[Dict[str, Any]] = []

    for name, path in fixtures.items():
        if path is None:
            cases.append({"case": name, "outcome": "skipped", "error_type": None,
                          "detail": "fixture unavailable in this environment"})
            continue
        status, result, err_msg, err_type = safe_execute(engine.predict_image, path, f"robust_{name}")
        outcome, error_type, detail = _classify_prediction_outcome(status, result, err_msg, err_type)
        cases.append({"case": name, "outcome": outcome, "error_type": error_type, "detail": detail})

    mixed_paths = [
        fixtures.get("tiny_image"), fixtures.get("grayscale_image"), fixtures.get("rgba_image"),
        fixtures.get("empty_image"), fixtures.get("format_webp") or fixtures.get("format_png"),
        pool_paths[0],
    ]
    mixed_paths = [p for p in mixed_paths if p]
    mixed_ids = [f"mixed_{i}" for i in range(len(mixed_paths))]
    status, result, err_msg, err_type = safe_execute(engine.predict_batch, mixed_paths, mixed_ids)
    if status == "crash":
        cases.append({"case": "mixed_batch", "outcome": "runtime_exception", "error_type": err_type, "detail": err_msg or ""})
    else:
        ordering_ok = [r.image_identifier for r in result] == mixed_ids
        count_ok = len(result) == len(mixed_paths)
        ok = ordering_ok and count_ok
        cases.append({
            "case": "mixed_batch", "outcome": "success" if ok else "structured_failure",
            "error_type": None if ok else "batch_integrity_violation",
            "detail": f"count_ok={count_ok} ordering_ok={ordering_ok} n={len(mixed_paths)}",
        })

    dup_path = pool_paths[0]
    dup_paths = [dup_path] * 5
    dup_ids = [f"dup_{i}" for i in range(5)]
    status, result, err_msg, err_type = safe_execute(engine.predict_batch, dup_paths, dup_ids)
    if status == "crash":
        cases.append({"case": "duplicate_images", "outcome": "runtime_exception", "error_type": err_type, "detail": err_msg or ""})
    else:
        probs_list = [r.probabilities for r in result if r.success]
        consistent = True
        if len(probs_list) > 1:
            base = probs_list[0]
            for p in probs_list[1:]:
                eq, *_rest = probabilities_equal(base, p)
                if not eq:
                    consistent = False
        cases.append({
            "case": "duplicate_images", "outcome": "success" if consistent else "structured_failure",
            "error_type": None if consistent else "inconsistent_duplicate_predictions",
            "detail": f"{len(probs_list)} successful duplicate predictions compared",
        })

    for bad_name, bad_obj in [
        ("invalid_tensor_string", "not_an_image"),
        ("invalid_tensor_list", [1, 2, 3]),
        ("invalid_tensor_ndarray", np.zeros((10, 10))),
    ]:
        status, result, err_msg, err_type = safe_execute(preprocess_image, bad_obj, engine.preprocessing_config)
        if status == "crash":
            cases.append({"case": bad_name, "outcome": "runtime_exception", "error_type": err_type, "detail": err_msg or ""})
        else:
            tensor, perr = result
            ok = tensor is None and perr is not None
            cases.append({
                "case": bad_name, "outcome": "structured_failure" if ok else "success",
                "error_type": perr.error_type if perr else None,
                "detail": perr.message if perr else "unexpectedly produced a valid tensor from malformed input",
            })

    bad_shapes = [
        (1, 5, engine.preprocessing_config.resize_height, engine.preprocessing_config.resize_width),
        (1, engine.preprocessing_config.channels, 10, 10),
    ]
    for shape in bad_shapes:
        bad_tensor = torch.zeros(shape, dtype=torch.float32, device=engine.device)
        status, result, err_msg, err_type = safe_execute(_forward_no_grad, engine.model, bad_tensor)
        name = f"wrong_dimensions_{'x'.join(map(str, shape))}"
        if status == "crash":
            cases.append({
                "case": name, "outcome": "runtime_exception", "error_type": err_type,
                "detail": (err_msg or "") + " [direct model() call bypasses InferenceEngine's file-path API contract]",
            })
        else:
            cases.append({"case": name, "outcome": "success", "error_type": None,
                          "detail": "model accepted mismatched tensor shape without raising"})

    for name, bad_input in [
        ("none_input", None), ("integer_input", 12345),
        ("list_input", [1, 2, 3]), ("empty_string_input", ""),
    ]:
        status, result, err_msg, err_type = safe_execute(engine.predict_image, bad_input, f"robust_{name}")
        outcome, error_type, detail = _classify_prediction_outcome(status, result, err_msg, err_type)
        cases.append({"case": name, "outcome": outcome, "error_type": error_type, "detail": detail})

    outcome_counts: Dict[str, int] = {}
    for c in cases:
        outcome_counts[c["outcome"]] = outcome_counts.get(c["outcome"], 0) + 1

    return {
        "total_cases": len(cases),
        "outcome_counts": outcome_counts,
        "cases": cases,
        "never_crashed": outcome_counts.get("runtime_exception", 0) == 0,
    }


# ======================================================================
# SECTION 4: NUMERICAL STABILITY (PyTorch vs TorchScript vs ONNX)
# ======================================================================

def run_numerical_stability(
    engine: Any, export_registry: Any, pool_paths: List[str], device: torch.device, logger: logging.Logger,
) -> Dict[str, Any]:
    ts_path = export_registry.torchscript.output_path
    onnx_path = export_registry.onnx.output_path

    ts_load_ok = False
    onnx_load_ok = False
    ts_model = None
    ort_session = None
    errors: List[str] = []

    try:
        ts_model = torch.jit.load(ts_path, map_location=device)
        ts_model.eval()
        ts_load_ok = True
    except Exception as exc:  # noqa: BLE001
        errors.append(f"TorchScript load failed: {exc}")

    if ort is not None:
        try:
            available = ort.get_available_providers()
            providers = ["CUDAExecutionProvider", "CPUExecutionProvider"] if (
                device.type == "cuda" and "CUDAExecutionProvider" in available
            ) else ["CPUExecutionProvider"]
            ort_session = ort.InferenceSession(onnx_path, providers=providers)
            onnx_load_ok = True
        except Exception as exc:  # noqa: BLE001
            errors.append(f"ONNX session load failed: {exc}")
    else:
        errors.append("onnxruntime is not importable in this environment.")

    per_sample: List[Dict[str, Any]] = []
    any_nan = False
    any_inf = False
    shape_ok = True

    engine.model.eval()
    n = max(1, min(NUMERICAL_STABILITY_SAMPLES, len(pool_paths) // max(1, NUMERICAL_STABILITY_BATCH_SIZE)))

    for i in range(n):
        start = i * NUMERICAL_STABILITY_BATCH_SIZE
        paths = pool_paths[start:start + NUMERICAL_STABILITY_BATCH_SIZE] or pool_paths[:1]
        tensors = []
        for p in paths:
            img, derr = validate_and_decode_image(p, engine.preprocessing_config.channels)
            if derr is not None:
                continue
            t, perr = preprocess_image(img, engine.preprocessing_config)
            if perr is not None:
                continue
            tensors.append(t)
        if not tensors:
            continue
        batch = torch.stack(tensors, dim=0).to(device)

        with torch.no_grad():
            pt_out = engine.model(batch)
        any_nan = any_nan or bool(torch.isnan(pt_out).any().item())
        any_inf = any_inf or bool(torch.isinf(pt_out).any().item())

        sample_result: Dict[str, Any] = {"sample_index": i, "batch_size": int(batch.shape[0])}

        if ts_load_ok:
            try:
                with torch.no_grad():
                    ts_out = ts_model(batch)
                any_nan = any_nan or bool(torch.isnan(ts_out).any().item())
                any_inf = any_inf or bool(torch.isinf(ts_out).any().item())
                diff = _compare_tensors(pt_out, ts_out)
                shape_ok = shape_ok and diff["shape_match"]
                sample_result["pytorch_vs_torchscript"] = diff
            except Exception as exc:  # noqa: BLE001
                sample_result["pytorch_vs_torchscript"] = {"error": str(exc)}

        if onnx_load_ok:
            try:
                input_name = export_registry.onnx.input_names[0]
                onnx_out = ort_session.run(None, {input_name: batch.cpu().numpy()})[0]
                onnx_out_t = torch.from_numpy(onnx_out)
                any_nan = any_nan or bool(torch.isnan(onnx_out_t).any().item())
                any_inf = any_inf or bool(torch.isinf(onnx_out_t).any().item())
                diff = _compare_tensors(pt_out, onnx_out_t)
                shape_ok = shape_ok and diff["shape_match"]
                sample_result["pytorch_vs_onnx"] = diff
            except Exception as exc:  # noqa: BLE001
                sample_result["pytorch_vs_onnx"] = {"error": str(exc)}

        per_sample.append(sample_result)

    def _aggregate(key: str) -> Dict[str, Optional[float]]:
        abs_vals = [s[key]["max_abs_error"] for s in per_sample if key in s and "max_abs_error" in s[key]]
        mean_vals = [s[key]["mean_abs_error"] for s in per_sample if key in s and "mean_abs_error" in s[key]]
        rel_vals = [s[key]["max_relative_error"] for s in per_sample if key in s and "max_relative_error" in s[key]]
        return {
            "max_abs_error": max(abs_vals) if abs_vals else None,
            "mean_abs_error": (sum(mean_vals) / len(mean_vals)) if mean_vals else None,
            "max_relative_error": max(rel_vals) if rel_vals else None,
        }

    ts_agg = _aggregate("pytorch_vs_torchscript")
    onnx_agg = _aggregate("pytorch_vs_onnx")

    ts_within_tol = (
        ts_agg["max_abs_error"] is not None
        and ts_agg["max_abs_error"] <= MAX_ABS_ERROR_TOLERANCE
        and ts_agg["max_relative_error"] <= MAX_RELATIVE_ERROR_TOLERANCE
    )
    onnx_within_tol = (
        onnx_agg["max_abs_error"] is not None
        and onnx_agg["max_abs_error"] <= MAX_ABS_ERROR_TOLERANCE
        and onnx_agg["max_relative_error"] <= MAX_RELATIVE_ERROR_TOLERANCE
    )

    return {
        "torchscript_load_success": ts_load_ok,
        "onnx_load_success": onnx_load_ok,
        "samples_evaluated": len(per_sample),
        "per_sample": per_sample,
        "pytorch_vs_torchscript_aggregate": ts_agg,
        "pytorch_vs_onnx_aggregate": onnx_agg,
        "max_abs_error_tolerance": MAX_ABS_ERROR_TOLERANCE,
        "max_relative_error_tolerance": MAX_RELATIVE_ERROR_TOLERANCE,
        "torchscript_within_tolerance": ts_within_tol,
        "onnx_within_tolerance": onnx_within_tol,
        "any_nan": any_nan,
        "any_inf": any_inf,
        "no_shape_mismatch": shape_ok,
        "passed": ts_load_ok and onnx_load_ok and ts_within_tol and onnx_within_tol and not any_nan and not any_inf,
        "errors": errors,
    }


# ======================================================================
# SECTION 5: MEMORY VALIDATION
# ======================================================================

def run_memory_validation(
    baseline: Dict[str, Any], benchmark_results: List[BatchBenchmarkResult], device: torch.device, logger: logging.Logger,
) -> Dict[str, Any]:
    gc.collect()
    if device.type == "cuda":
        torch.cuda.synchronize()
        torch.cuda.empty_cache()

    final_cpu = psutil.Process().memory_info().rss / (1024 ** 2)
    final_gpu_alloc = torch.cuda.memory_allocated(device) / (1024 ** 2) if device.type == "cuda" else None
    final_gpu_reserved = torch.cuda.memory_reserved(device) / (1024 ** 2) if device.type == "cuda" else None

    peak_gpu = None
    if device.type == "cuda":
        peaks = [r.peak_gpu_memory_mb for r in benchmark_results if r.peak_gpu_memory_mb is not None]
        peak_gpu = max(peaks) if peaks else None

    cpu_growth = final_cpu - baseline["cpu_rss_mb"]
    gpu_growth = None
    if device.type == "cuda" and baseline.get("gpu_allocated_mb") is not None:
        gpu_growth = final_gpu_alloc - baseline["gpu_allocated_mb"]

    leak_cpu = cpu_growth > MEMORY_LEAK_THRESHOLD_MB
    leak_gpu = (gpu_growth is not None) and (gpu_growth > GPU_LEAK_THRESHOLD_MB)

    return {
        "baseline": baseline,
        "final": {"cpu_rss_mb": final_cpu, "gpu_allocated_mb": final_gpu_alloc, "gpu_reserved_mb": final_gpu_reserved},
        "peak_gpu_allocated_mb": peak_gpu,
        "cpu_rss_growth_mb": cpu_growth,
        "gpu_allocated_growth_mb": gpu_growth,
        "leak_detected_cpu": leak_cpu,
        "leak_detected_gpu": leak_gpu,
        "leak_detected": leak_cpu or bool(leak_gpu),
        "memory_recovered": (not leak_cpu) and (not leak_gpu),
        "no_cuda_leaks": (device.type != "cuda") or (not leak_gpu),
    }


# ======================================================================
# SECTION 6: ENGINEERING VALIDATION (overall stage gate)
# ======================================================================

def run_stage05_engineering_validation(
    engine: Any,
    model_registry: Any,
    threshold_registry: Any,
    benchmark_results: List[BatchBenchmarkResult],
    stress_result: Dict[str, Any],
    robustness_result: Dict[str, Any],
    numerical_result: Dict[str, Any],
    memory_result: Dict[str, Any],
    logger: logging.Logger,
) -> Dict[str, Any]:
    checks: Dict[str, bool] = {}
    warnings_list: List[str] = []
    fatal_errors: List[str] = []

    checks["inference_runtime_deterministic"] = stress_result["deterministic"]
    if not checks["inference_runtime_deterministic"]:
        fatal_errors.append("Repeated inference on the same input produced non-deterministic results.")

    checks["exports_usable"] = numerical_result["torchscript_load_success"] and numerical_result["onnx_load_success"]
    if not checks["exports_usable"]:
        fatal_errors.append("TorchScript and/or ONNX export could not be loaded for inference.")

    tested = [r for r in benchmark_results if not r.oom and r.error is None]
    checks["batch_support_correct"] = len(tested) > 0 and all(r.result_count_correct for r in tested)
    if not checks["batch_support_correct"]:
        fatal_errors.append("One or more benchmarked batch sizes did not return the correct number of results.")

    checks["no_cuda_leaks"] = memory_result["no_cuda_leaks"]
    if not checks["no_cuda_leaks"]:
        fatal_errors.append("GPU memory growth after cleanup exceeds the configured leak threshold.")

    checks["no_nans"] = not numerical_result.get("any_nan", False)
    if not checks["no_nans"]:
        fatal_errors.append("NaN values detected in model outputs during numerical stability testing.")

    checks["no_infs"] = not numerical_result.get("any_inf", False)
    if not checks["no_infs"]:
        fatal_errors.append("Inf values detected in model outputs during numerical stability testing.")

    checks["no_shape_mismatch"] = numerical_result.get("no_shape_mismatch", True)
    if not checks["no_shape_mismatch"]:
        fatal_errors.append("Output shape mismatch detected between PyTorch, TorchScript, and/or ONNX.")

    checks["threshold_registry_consistent"] = (
        len(engine.thresholds) == len(engine.class_names) == model_registry.num_classes
        and threshold_registry.class_count in (0, model_registry.num_classes)
    )
    if not checks["threshold_registry_consistent"]:
        fatal_errors.append("THRESHOLD_REGISTRY is not consistent with the resolved class set.")

    checks["output_schema_stable"] = stress_result.get("schema_stable", True)
    if not checks["output_schema_stable"]:
        fatal_errors.append("PredictionResult schema keys varied across successful predictions.")

    checks["memory_recovered"] = memory_result["memory_recovered"]
    if not checks["memory_recovered"]:
        fatal_errors.append("Memory did not return close to baseline after gc.collect()/empty_cache().")

    if not numerical_result.get("passed", False) and checks["exports_usable"]:
        warnings_list.append(
            "Re-validated numerical parity (PyTorch vs TorchScript/ONNX) on realistic preprocessed "
            f"inputs exceeded tolerance (abs<={MAX_ABS_ERROR_TOLERANCE}, rel<={MAX_RELATIVE_ERROR_TOLERANCE}). "
            "Stage 3 already gated export parity on synthetic dummy inputs and passed; this is an "
            "additional, non-fatal, real-input cross-check."
        )

    crash_cases = [c for c in robustness_result["cases"] if c["outcome"] == "runtime_exception"]
    if crash_cases:
        warnings_list.append(
            f"{len(crash_cases)} robustness edge-case(s) raised unhandled exceptions in the frozen "
            f"Stage 4 InferenceEngine rather than returning a structured failure: "
            f"{[c['case'] for c in crash_cases]}. These reflect upstream (Stage 4) behavior for inputs "
            "outside its documented file-path API contract; Stage 4 is frozen and out of scope to modify "
            "here. Recorded for visibility in robustness_summary.json."
        )

    if benchmark_results and any(r.oom for r in benchmark_results):
        oom_sizes = [r.batch_size for r in benchmark_results if r.oom]
        warnings_list.append(f"Batch size(s) {oom_sizes} exceeded available memory and were skipped.")

    for w in warnings_list:
        logger.warning("VALIDATION WARNING: %s", w)
    for e in fatal_errors:
        logger.error("VALIDATION FAILURE: %s", e)

    return {"checks": checks, "warnings": warnings_list, "fatal_errors": fatal_errors, "passed": len(fatal_errors) == 0}


# ======================================================================
# MAIN STAGE ENTRY POINT
# ======================================================================

def run_stage05(
    inference_engine: Any,
    reconstructed_model: Any,
    artifact_registry: Any,
    model_registry: Any,
    metadata_registry: Any,
    threshold_registry: Any,
    config_registry: Any,
    export_registry: Any,
) -> Dict[str, Any]:
    start_time = time.time()

    stage_root = OUTPUT_ROOT / STAGE5_DIR_NAME
    dirs = {
        "stage_root": stage_root,
        "logs": stage_root / "logs",
        "benchmark_images": stage_root / "images" / "benchmark",
        "robustness_images": stage_root / "images" / "robustness",
    }
    for p in dirs.values():
        p.mkdir(parents=True, exist_ok=True)

    logger = build_logger(dirs["logs"])
    logger.info("START Sprint05-Stage05 Performance & Robustness Validation")

    engine = inference_engine
    device = torch.device(config_registry.device)
    log_resources(logger, "START", device)

    assert engine.model is reconstructed_model, "INFERENCE_ENGINE.model must be the Stage 2 RECONSTRUCTED_MODEL."

    gc.collect()
    if device.type == "cuda":
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats(device)
    baseline_memory = {
        "cpu_rss_mb": psutil.Process().memory_info().rss / (1024 ** 2),
        "gpu_allocated_mb": torch.cuda.memory_allocated(device) / (1024 ** 2) if device.type == "cuda" else None,
        "gpu_reserved_mb": torch.cuda.memory_reserved(device) / (1024 ** 2) if device.type == "cuda" else None,
    }
    logger.info("MEMORY baseline captured: %s", baseline_memory)

    max_batch = max(BATCH_SIZES)
    pool = generate_synthetic_sample_images(
        dirs["benchmark_images"], max_batch,
        engine.preprocessing_config.resize_height, engine.preprocessing_config.resize_width,
        SYNTHETIC_POOL_SEED, logger,
    )
    pool_paths = [p for p, _ in pool]
    logger.info("BENCHMARK synthetic image pool ready: %d images", len(pool_paths))

    # ---- Section 1: Performance Benchmark ----
    logger.info("START benchmark")
    benchmark_results: List[BatchBenchmarkResult] = []
    skipped_oom: List[int] = []
    for bs in BATCH_SIZES:
        if bs > len(pool_paths):
            logger.warning("BENCHMARK batch_size=%d exceeds pool size %d; skipping.", bs, len(pool_paths))
            continue
        logger.info("BENCHMARK running batch_size=%d", bs)
        r = benchmark_batch_size(engine, pool_paths, bs, device, logger)
        benchmark_results.append(r)
        if r.oom:
            skipped_oom.append(bs)
            logger.warning("BENCHMARK batch_size=%d skipped due to OOM.", bs)
        else:
            logger.info(
                "BENCHMARK batch_size=%d mean=%.2fms p95=%.2fms throughput=%.2f img/s",
                bs, r.warm_latency.mean_ms, r.warm_latency.p95_ms, r.throughput_images_per_sec,
            )
    logger.info("FINISH benchmark")

    successful_benchmarks = [r for r in benchmark_results if not r.oom and r.error is None]
    scaling_analysis = []
    if successful_benchmarks:
        base = successful_benchmarks[0]
        for r in successful_benchmarks:
            scaling_analysis.append({
                "batch_size": r.batch_size,
                "throughput_images_per_sec": r.throughput_images_per_sec,
                "scaling_efficiency_vs_batch1": (
                    (r.throughput_images_per_sec / base.throughput_images_per_sec) / (r.batch_size / base.batch_size)
                    if base.throughput_images_per_sec else None
                ),
            })

    # ---- Section 2: Runtime Stress Testing ----
    logger.info("START stress testing")
    stress_result = run_stress_test(engine, pool_paths, device, logger)
    logger.info(
        "FINISH stress testing calls=%d crashes=%d deterministic=%s memory_stable=%s",
        stress_result["total_calls"], len(stress_result["crashes"]),
        stress_result["deterministic"], stress_result["memory_stable"],
    )

    # ---- Section 3: Robustness Validation ----
    logger.info("START robustness validation")
    robustness_result = run_robustness_validation(engine, pool_paths, dirs, logger)
    logger.info(
        "FINISH robustness validation total=%d outcomes=%s never_crashed=%s",
        robustness_result["total_cases"], robustness_result["outcome_counts"], robustness_result["never_crashed"],
    )

    # ---- Section 4: Numerical Stability ----
    logger.info("START numerical stability")
    numerical_result = run_numerical_stability(engine, export_registry, pool_paths, device, logger)
    logger.info(
        "FINISH numerical stability torchscript_ok=%s onnx_ok=%s passed=%s",
        numerical_result["torchscript_load_success"], numerical_result["onnx_load_success"], numerical_result["passed"],
    )

    # ---- Section 5: Memory Validation ----
    logger.info("START memory validation")
    memory_result = run_memory_validation(baseline_memory, benchmark_results, device, logger)
    logger.info(
        "FINISH memory validation leak_detected=%s memory_recovered=%s",
        memory_result["leak_detected"], memory_result["memory_recovered"],
    )

    # ---- Section 6: Engineering Validation ----
    logger.info("START engineering validation")
    engineering_validation = run_stage05_engineering_validation(
        engine=engine, model_registry=model_registry, threshold_registry=threshold_registry,
        benchmark_results=benchmark_results, stress_result=stress_result, robustness_result=robustness_result,
        numerical_result=numerical_result, memory_result=memory_result, logger=logger,
    )

    if not engineering_validation["passed"]:
        log_resources(logger, "FINISH-FAILED", device)
        elapsed = time.time() - start_time
        logger.error("FINISH Sprint05-Stage05 status=FAILED elapsed=%.2fs", elapsed)
        raise RuntimeError(
            "Sprint05 Stage05 engineering validation FAILED (fail-fast):\n  - "
            + "\n  - ".join(engineering_validation["fatal_errors"])
        )

    log_resources(logger, "FINISH", device)
    elapsed = time.time() - start_time

    # ---- Persist outputs ----
    benchmark_summary = {
        "batch_sizes_requested": BATCH_SIZES,
        "batch_sizes_tested": [r.batch_size for r in successful_benchmarks],
        "batch_sizes_skipped_oom": skipped_oom,
        "warmup_iterations": WARMUP_ITERATIONS,
        "measured_iterations": BENCHMARK_ITERATIONS,
        "results": [r.to_dict() for r in benchmark_results],
        "batch_scaling_analysis": scaling_analysis,
    }
    save_json(dirs["stage_root"] / "benchmark_summary.json", benchmark_summary)

    with open(dirs["stage_root"] / "latency.csv", "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["batch_size", "cold_start_ms", "mean_ms", "min_ms", "max_ms", "std_ms", "p95_ms"])
        for r in benchmark_results:
            if r.warm_latency is None:
                writer.writerow([r.batch_size, "", "", "", "", "", ""])
            else:
                writer.writerow([
                    r.batch_size, round(r.cold_start_ms, 3), round(r.warm_latency.mean_ms, 3),
                    round(r.warm_latency.min_ms, 3), round(r.warm_latency.max_ms, 3),
                    round(r.warm_latency.std_ms, 3), round(r.warm_latency.p95_ms, 3),
                ])

    with open(dirs["stage_root"] / "throughput.csv", "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["batch_size", "throughput_images_per_sec", "fps"])
        for r in benchmark_results:
            if r.throughput_images_per_sec is None:
                writer.writerow([r.batch_size, "", ""])
            else:
                writer.writerow([r.batch_size, round(r.throughput_images_per_sec, 3), round(r.fps, 3)])

    save_json(dirs["stage_root"] / "stress_test.json", stress_result)
    save_json(dirs["stage_root"] / "robustness_summary.json", robustness_result)
    save_json(dirs["stage_root"] / "memory_profile.json", memory_result)

    runtime_validation_summary = {
        "numerical_stability": numerical_result,
        "stress_test_deterministic": stress_result["deterministic"],
        "batch_support_verified_batch_sizes": [r.batch_size for r in benchmark_results if r.result_count_correct],
    }
    save_json(dirs["stage_root"] / "runtime_validation.json", runtime_validation_summary)
    save_json(dirs["stage_root"] / "engineering_validation.json", engineering_validation)

    stage05_summary = {
        "stage": STAGE5_DIR_NAME,
        "status": "OK",
        "elapsed_seconds": round(elapsed, 3),
        "batch_sizes_tested": [r.batch_size for r in successful_benchmarks],
        "batch_sizes_skipped_oom": skipped_oom,
        "stress_test_calls": stress_result["total_calls"],
        "stress_test_crashes": len(stress_result["crashes"]),
        "robustness_cases_tested": robustness_result["total_cases"],
        "robustness_outcome_counts": robustness_result["outcome_counts"],
        "numerical_stability_passed": numerical_result["passed"],
        "memory_leak_detected": memory_result["leak_detected"],
        "engineering_checks_passed": sum(engineering_validation["checks"].values()),
        "engineering_checks_total": len(engineering_validation["checks"]),
        "warnings_count": len(engineering_validation["warnings"]),
        "output_directory": str(dirs["stage_root"]),
    }
    save_json(dirs["stage_root"] / "stage05_summary.json", stage05_summary)

    logger.info("FINISH Sprint05-Stage05 status=OK elapsed=%.2fs", elapsed)

    # ---- Console report ----
    def _fmt_ms(v: Optional[float]) -> str:
        return "N/A" if v is None else f"{v:.2f} ms"

    bs1 = next((r for r in benchmark_results if r.batch_size == 1 and not r.oom), None)

    print("=" * 70)
    print("SPRINT05 — STAGE05")
    print("PERFORMANCE & ROBUSTNESS VALIDATION")
    print("=" * 70)
    if bs1 is not None:
        print(f"Latency              : bs=1 mean={_fmt_ms(bs1.warm_latency.mean_ms)} p95={_fmt_ms(bs1.warm_latency.p95_ms)} cold_start={_fmt_ms(bs1.cold_start_ms)}")
        print(f"Throughput           : bs=1 {bs1.throughput_images_per_sec:.2f} img/s (FPS={bs1.fps:.2f})")
    else:
        print("Latency              : N/A")
        print("Throughput           : N/A")
    if memory_result["final"]["gpu_allocated_mb"] is not None:
        print(f"GPU Memory           : allocated={memory_result['final']['gpu_allocated_mb']:.2f}MB reserved={memory_result['final']['gpu_reserved_mb']:.2f}MB")
    else:
        print("GPU Memory           : N/A (CPU-only device)")
    print(f"CPU Memory           : {memory_result['final']['cpu_rss_mb']:.2f} MB (growth={memory_result['cpu_rss_growth_mb']:.2f}MB)")
    print(f"Stress Test          : {stress_result['total_calls']} calls, {len(stress_result['crashes'])} crashes, deterministic={stress_result['deterministic']}")
    print(f"Robustness           : {robustness_result['total_cases']} cases -> {robustness_result['outcome_counts']}")
    print(f"Engineering Checks   : {sum(engineering_validation['checks'].values())}/{len(engineering_validation['checks'])} passed")
    print(f"Warnings             : {len(engineering_validation['warnings'])}")
    print(f"Output Directory     : {dirs['stage_root']}")
    print("Stage 5 : OK")

    return {
        "benchmark_summary": benchmark_summary,
        "stress_test": stress_result,
        "robustness_summary": robustness_result,
        "numerical_stability": numerical_result,
        "memory_profile": memory_result,
        "validation": engineering_validation,
        "summary": stage05_summary,
    }


if __name__ == "__main__":
    _stage05_result = run_stage05(
        inference_engine=INFERENCE_ENGINE,
        reconstructed_model=RECONSTRUCTED_MODEL,
        artifact_registry=ARTIFACT_REGISTRY,
        model_registry=MODEL_REGISTRY,
        metadata_registry=METADATA_REGISTRY,
        threshold_registry=THRESHOLD_REGISTRY,
        config_registry=CONFIG_REGISTRY,
        export_registry=EXPORT_REGISTRY,
    )

2026-07-08 10:31:47 | INFO     | START Sprint05-Stage05 Performance & Robustness Validation
2026-07-08 10:31:47 | INFO     | RESOURCES [START] cpu=0.0% ram=7.9% rss=1574.2MB
2026-07-08 10:31:47 | INFO     | MEMORY baseline captured: {'cpu_rss_mb': 1574.23046875, 'gpu_allocated_mb': 36.12890625, 'gpu_reserved_mb': 52.0}


/tmp/ipykernel_195/3542953323.py:499: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  img = Image.fromarray(array, mode="RGB")


2026-07-08 10:31:47 | INFO     | RUNTIME generated 32 synthetic sample image(s) (seed=777) for validation.
2026-07-08 10:31:47 | INFO     | BENCHMARK synthetic image pool ready: 32 images
2026-07-08 10:31:47 | INFO     | START benchmark
2026-07-08 10:31:47 | INFO     | BENCHMARK running batch_size=1
2026-07-08 10:31:48 | INFO     | BENCHMARK batch_size=1 mean=16.81ms p95=17.41ms throughput=59.48 img/s
2026-07-08 10:31:48 | INFO     | BENCHMARK running batch_size=2
2026-07-08 10:31:48 | INFO     | BENCHMARK batch_size=2 mean=19.72ms p95=20.49ms throughput=101.40 img/s
2026-07-08 10:31:48 | INFO     | BENCHMARK running batch_size=4
2026-07-08 10:31:48 | INFO     | BENCHMARK batch_size=4 mean=23.02ms p95=23.69ms throughput=173.73 img/s
2026-07-08 10:31:48 | INFO     | BENCHMARK running batch_size=8
2026-07-08 10:31:49 | INFO     | BENCHMARK batch_size=8 mean=40.59ms p95=41.18ms throughput=197.07 img/s
2026-07-08 10:31:49 | INFO     | BENCHMARK running batch_size=16
2026-07-08 10:31:51 | I

/tmp/ipykernel_195/4184024320.py:374: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  Image.fromarray(np.random.RandomState(1).randint(0, 256, (2, 2, 3), dtype=np.uint8), "RGB").save(p)
/tmp/ipykernel_195/4184024320.py:379: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  Image.fromarray(big, "RGB").save(p)
/tmp/ipykernel_195/4184024320.py:384: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  Image.fromarray(gray, "L").save(p)
/tmp/ipykernel_195/4184024320.py:389: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  Image.fromarray(rgba, "RGBA").save(p)
/tmp/ipykernel_195/4184024320.py:398: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  Image.fromarray(noise, "RGB").save(p)
/tmp/ipykernel_195/4184024320.py:402: DeprecationWarning: 'mode' par

2026-07-08 10:31:58 | WARNING  | INFERENCE reject identifier=robust_empty_image reason=zero_byte_image: Image file is zero bytes: /kaggle/working/sprint05_deployment/stage05_validation/images/robustness/empty.png
2026-07-08 10:31:58 | WARNING  | INFERENCE reject identifier=robust_format_webp reason=unsupported_extension: Unsupported file extension '.webp' for /kaggle/working/sprint05_deployment/stage05_validation/images/robustness/format_webp.webp
2026-07-08 10:31:58 | WARNING  | INFERENCE reject identifier=mixed_3 reason=zero_byte_image: Image file is zero bytes: /kaggle/working/sprint05_deployment/stage05_validation/images/robustness/empty.png
2026-07-08 10:31:58 | WARNING  | INFERENCE reject identifier=mixed_4 reason=unsupported_extension: Unsupported file extension '.webp' for /kaggle/working/sprint05_deployment/stage05_validation/images/robustness/format_webp.webp
2026-07-08 10:31:59 | WARNING  | INFERENCE reject identifier=robust_empty_string_input reason=missing_file: Image file

In [11]:
"""
VisionServeAI - Sprint 05
Stage 6: Explainability Runtime
======================================================================
Single-responsibility stage: builds a reusable, object-oriented
ExplainabilityEngine exposing GradCAM-family, gradient-based, and
perturbation-based explainability methods on top of the already
reconstructed/validated production model.

STRICT SCOPE:
    - No evaluation. No benchmarking. No model export. No metrics.
    - No dataset traversal. Stages 1-5 are FROZEN. RECONSTRUCTED_MODEL,
      INFERENCE_ENGINE, MODEL_REGISTRY, METADATA_REGISTRY,
      THRESHOLD_REGISTRY, CONFIG_REGISTRY, ARTIFACT_REGISTRY,
      EXPORT_REGISTRY are consumed as in-memory objects only -- never
      reloaded, never rediscovered, never rebuilt.
======================================================================
"""

from __future__ import annotations

import json
import logging
import time
from dataclasses import dataclass, field, asdict
from pathlib import Path
from typing import Any, Callable, Dict, List, Optional, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image

try:
    import cv2
    _CV2_AVAILABLE = True
except ImportError:
    _CV2_AVAILABLE = False

try:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.cm as _mpl_cm
    _MATPLOTLIB_AVAILABLE = True
except ImportError:
    _MATPLOTLIB_AVAILABLE = False

# ======================================================================
# CONSTANTS
# ======================================================================

OUTPUT_ROOT = Path("/kaggle/working/sprint05_deployment")
STAGE6_DIR_NAME = "stage06_explainability"

NUM_SYNTHETIC_SAMPLES = 4
SYNTHETIC_SAMPLE_SEED = 4242

IG_STEPS = 16
OCCLUSION_PATCH_SIZE = 32
OCCLUSION_STRIDE = 16
SCORECAM_MAX_CHANNELS = 32

METHOD_NAMES: List[str] = [
    "gradcam", "gradcam_plus", "scorecam", "eigencam",
    "guided_backprop", "integrated_gradients", "occlusion",
]


# ======================================================================
# LOGGING (self-contained, same format/pattern as Stage 1-5)
# ======================================================================

def build_logger(log_dir: Path) -> logging.Logger:
    log_dir.mkdir(parents=True, exist_ok=True)
    logger = logging.getLogger("sprint05.stage06")
    logger.setLevel(logging.INFO)
    logger.handlers.clear()
    logger.propagate = False

    fmt = logging.Formatter(
        fmt="%(asctime)s | %(levelname)-8s | %(message)s",
        datefmt="%Y-%m-%d %H:%M:%S",
    )

    file_handler = logging.FileHandler(log_dir / "stage06_explainability.log", mode="w")
    file_handler.setFormatter(fmt)
    logger.addHandler(file_handler)

    stream_handler = logging.StreamHandler()
    stream_handler.setFormatter(fmt)
    logger.addHandler(stream_handler)

    return logger


def save_json(path: Path, data: Any) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w") as f:
        json.dump(data, f, indent=2, default=str)


def make_stage_dirs(stage_root: Path) -> Dict[str, Path]:
    dirs = {
        "stage_root": stage_root,
        "logs": stage_root / "logs",
        "gradcam": stage_root / "gradcam",
        "gradcam_plus": stage_root / "gradcam_plus",
        "scorecam": stage_root / "scorecam",
        "eigencam": stage_root / "eigencam",
        "integrated_gradients": stage_root / "integrated_gradients",
        "guided_backprop": stage_root / "guided_backprop",
        "occlusion": stage_root / "occlusion",
        "overlays": stage_root / "overlays",
        "metadata": stage_root / "metadata",
        "inputs": stage_root / "images" / "synthetic_inputs",
    }
    for d in dirs.values():
        d.mkdir(parents=True, exist_ok=True)
    return dirs


# ======================================================================
# DATA CLASSES
# ======================================================================

@dataclass
class MethodAvailability:
    method: str
    available: bool
    reason: str

    def to_dict(self) -> Dict[str, Any]:
        return asdict(self)


@dataclass
class MethodResult:
    method: str
    sample_id: str
    success: bool
    execution_time_ms: float = 0.0
    predicted_class: Optional[str] = None
    predicted_class_index: Optional[int] = None
    target_class: Optional[str] = None
    target_class_index: Optional[int] = None
    confidence: Optional[float] = None
    heatmap_path: Optional[str] = None
    overlay_path: Optional[str] = None
    raw_attribution_path: Optional[str] = None
    heatmap_shape: Optional[List[int]] = None
    no_nan: Optional[bool] = None
    no_inf: Optional[bool] = None
    error: Optional[str] = None

    def to_dict(self) -> Dict[str, Any]:
        return asdict(self)


# ======================================================================
# HOOK UTILITIES
# ======================================================================

class ActivationsAndGradients:
    """Registers forward+backward hooks on a single target layer and
    captures its activations and gradients for the last forward/backward
    pass. Always removed via .remove() in a finally block by callers."""

    def __init__(self, target_layer: nn.Module) -> None:
        self.activations: Optional[torch.Tensor] = None
        self.gradients: Optional[torch.Tensor] = None
        self._fwd_handle = target_layer.register_forward_hook(self._forward_hook)
        try:
            self._bwd_handle = target_layer.register_full_backward_hook(self._backward_hook)
        except AttributeError:
            self._bwd_handle = target_layer.register_backward_hook(self._backward_hook)

    def _forward_hook(self, module, inputs, output):
        self.activations = output

    def _backward_hook(self, module, grad_input, grad_output):
        self.gradients = grad_output[0]

    def remove(self) -> None:
        self._fwd_handle.remove()
        self._bwd_handle.remove()


def discover_target_layer(model: nn.Module, logger: logging.Logger) -> Tuple[str, nn.Module, int]:
    """Automatic layer discovery: selects the LAST nn.Conv2d module found
    via named_modules() traversal. Works identically across DenseNet121,
    ResNet, EfficientNet, VGG, and any future torchvision-style CNN
    backbone without ever hardcoding a layer name."""
    last_name: Optional[str] = None
    last_module: Optional[nn.Module] = None
    for name, module in model.named_modules():
        if isinstance(module, nn.Conv2d):
            last_name, last_module = name, module

    if last_module is None:
        raise RuntimeError(
            "Automatic layer discovery failed: no nn.Conv2d layer found anywhere in the "
            "model. CAM-based methods require at least one convolutional layer."
        )

    feature_dim = last_module.out_channels
    logger.info(
        "LAYER DISCOVERY selected target layer='%s' type=%s feature_dim=%d",
        last_name, type(last_module).__name__, feature_dim,
    )
    return last_name, last_module, feature_dim


# ======================================================================
# EXPLAINABILITY ENGINE
# ======================================================================

class ExplainabilityEngine:
    """Production explainability engine.

    Consumes ONLY the already-existing in-memory objects produced by
    Stages 2-4 (INFERENCE_ENGINE, RECONSTRUCTED_MODEL, MODEL_REGISTRY,
    METADATA_REGISTRY, THRESHOLD_REGISTRY, CONFIG_REGISTRY,
    ARTIFACT_REGISTRY, EXPORT_REGISTRY). Never reopens files, never
    rediscovers datasets, never reloads checkpoints.

    Exposes a clean, reusable, object-oriented API (generate_gradcam,
    generate_gradcam_plus, generate_scorecam, generate_eigencam,
    generate_integrated_gradients, generate_guided_backprop,
    generate_occlusion, batch_generate, save_visualization,
    overlay_heatmap, export_results) that any deployment surface
    (REST API, FastAPI, Streamlit, Gradio, desktop, future
    microservices) can call directly without rewriting any GradCAM
    logic.
    """

    def __init__(
        self,
        inference_engine: Any,
        reconstructed_model: nn.Module,
        model_registry: Any,
        metadata_registry: Any,
        threshold_registry: Any,
        config_registry: Any,
        artifact_registry: Any,
        export_registry: Any,
        output_dir: Path,
        logger: logging.Logger,
    ) -> None:
        self.engine = inference_engine
        self.model = reconstructed_model
        self.model.eval()
        self.model_registry = model_registry
        self.metadata_registry = metadata_registry
        self.threshold_registry = threshold_registry
        self.config_registry = config_registry
        self.artifact_registry = artifact_registry
        self.export_registry = export_registry
        self.logger = logger

        self.device = inference_engine.device
        self.class_names: List[str] = list(inference_engine.class_names)
        self.thresholds: Dict[str, float] = dict(inference_engine.thresholds)
        self.preprocessing_config = inference_engine.preprocessing_config

        self.dirs = make_stage_dirs(output_dir)

        # ---- Automatic layer discovery ----
        self.layer_name, self.target_layer, self.feature_dim = discover_target_layer(self.model, logger)
        self.activation_size: Optional[List[int]] = None
        self._probe_activation_size()

        # ---- Colorization backend detection (logged once) ----
        if _CV2_AVAILABLE:
            self.logger.info("COLORMAP backend=cv2 (COLORMAP_JET)")
        elif _MATPLOTLIB_AVAILABLE:
            self.logger.warning("COLORMAP cv2 unavailable; falling back to matplotlib 'jet' colormap.")
        else:
            self.logger.warning("COLORMAP cv2 and matplotlib unavailable; using manual jet-style fallback.")

        # ---- Method availability detection (never silently fail) ----
        self.method_availability: Dict[str, MethodAvailability] = self._detect_method_availability()

        self.results: List[MethodResult] = []

        self._method_dispatch: Dict[str, Callable[..., MethodResult]] = {
            "gradcam": self.generate_gradcam,
            "gradcam_plus": self.generate_gradcam_plus,
            "scorecam": self.generate_scorecam,
            "eigencam": self.generate_eigencam,
            "guided_backprop": self.generate_guided_backprop,
            "integrated_gradients": self.generate_integrated_gradients,
            "occlusion": self.generate_occlusion,
        }

    # ------------------------------------------------------------------
    # Setup helpers
    # ------------------------------------------------------------------

    def _probe_activation_size(self) -> None:
        cfg = self.preprocessing_config
        dummy = torch.zeros(1, cfg.channels, cfg.resize_height, cfg.resize_width, device=self.device)
        holder: Dict[str, torch.Tensor] = {}

        def _hook(module, inputs, output):
            holder["a"] = output

        handle = self.target_layer.register_forward_hook(_hook)
        try:
            with torch.no_grad():
                self.model(dummy)
        finally:
            handle.remove()

        if "a" in holder:
            shape = list(holder["a"].shape[1:])  # [C, H, W]
            self.activation_size = shape
            self.logger.info(
                "LAYER DISCOVERY activation size for layer='%s': %s (forward hook verified)",
                self.layer_name, shape,
            )
        else:
            raise RuntimeError(f"Forward hook on layer '{self.layer_name}' failed to capture an activation.")

    def _detect_method_availability(self) -> Dict[str, MethodAvailability]:
        avail: Dict[str, MethodAvailability] = {}

        for m in ("gradcam", "gradcam_plus", "scorecam", "eigencam"):
            avail[m] = MethodAvailability(m, True, f"target conv layer '{self.layer_name}' available")

        relu_count = sum(1 for mod in self.model.modules() if isinstance(mod, nn.ReLU))
        if relu_count > 0:
            avail["guided_backprop"] = MethodAvailability(
                "guided_backprop", True, f"{relu_count} nn.ReLU module(s) found for hook-based gradient override"
            )
        else:
            avail["guided_backprop"] = MethodAvailability(
                "guided_backprop", False, "no nn.ReLU modules found in model; method disabled"
            )
            self.logger.warning("METHOD guided_backprop disabled: no nn.ReLU modules found in model.")

        avail["integrated_gradients"] = MethodAvailability(
            "integrated_gradients", True, "pure input-gradient method; always available"
        )
        avail["occlusion"] = MethodAvailability(
            "occlusion", True, "pure forward-pass perturbation method; always available"
        )
        return avail

    # ------------------------------------------------------------------
    # Preprocessing / display helpers (self-contained, no reopening of
    # Stage 4 internals -- mirrors the same documented preprocessing
    # contract carried on INFERENCE_ENGINE.preprocessing_config)
    # ------------------------------------------------------------------

    def _to_input_tensor(self, image: Image.Image) -> torch.Tensor:
        cfg = self.preprocessing_config
        resized = image.convert("RGB").resize((cfg.resize_width, cfg.resize_height), Image.BILINEAR)
        array = np.asarray(resized, dtype=np.float32) / 255.0
        tensor = torch.from_numpy(array).permute(2, 0, 1).contiguous()
        mean = torch.tensor(cfg.mean, dtype=torch.float32).view(-1, 1, 1)
        std = torch.tensor(cfg.std, dtype=torch.float32).view(-1, 1, 1)
        return (tensor - mean) / std

    def _to_display_rgb(self, tensor: torch.Tensor) -> np.ndarray:
        cfg = self.preprocessing_config
        mean = torch.tensor(cfg.mean, dtype=torch.float32).view(-1, 1, 1)
        std = torch.tensor(cfg.std, dtype=torch.float32).view(-1, 1, 1)
        denorm = (tensor.detach().cpu() * std + mean).clamp(0, 1)
        array = (denorm.permute(1, 2, 0).numpy() * 255.0).astype(np.uint8)
        return array

    def _predicted_class(self, probs: torch.Tensor) -> Tuple[int, str, float]:
        idx = int(torch.argmax(probs).item())
        return idx, self.class_names[idx], float(probs[idx].item())

    def _resolve_target(
        self, probe_probs: torch.Tensor, target_class: Optional[Any]
    ) -> Tuple[int, str, int, str, float]:
        pred_idx, pred_name, pred_conf = self._predicted_class(probe_probs)
        if target_class is None:
            return pred_idx, pred_name, pred_idx, pred_name, pred_conf
        if isinstance(target_class, str):
            if target_class not in self.class_names:
                raise ValueError(f"Unknown target_class '{target_class}'.")
            target_idx = self.class_names.index(target_class)
        else:
            target_idx = int(target_class)
        return pred_idx, pred_name, target_idx, self.class_names[target_idx], pred_conf

    @staticmethod
    def _normalize_map(x: np.ndarray) -> np.ndarray:
        x = np.nan_to_num(x.astype(np.float32), nan=0.0, posinf=0.0, neginf=0.0)
        x_min, x_max = float(x.min()), float(x.max())
        if (x_max - x_min) < 1e-12:
            return np.zeros_like(x, dtype=np.float32)
        return ((x - x_min) / (x_max - x_min)).astype(np.float32)

    # ------------------------------------------------------------------
    # Colorization / overlay / saving (shared reusable primitives)
    # ------------------------------------------------------------------

    def _colorize(self, heatmap_2d: np.ndarray) -> np.ndarray:
        h_u8 = (np.clip(heatmap_2d, 0.0, 1.0) * 255.0).astype(np.uint8)
        if _CV2_AVAILABLE:
            colored = cv2.applyColorMap(h_u8, cv2.COLORMAP_JET)
            return cv2.cvtColor(colored, cv2.COLOR_BGR2RGB)
        if _MATPLOTLIB_AVAILABLE:
            cmap = _mpl_cm.get_cmap("jet")
            colored = cmap(h_u8.astype(np.float32) / 255.0)[..., :3]
            return (colored * 255.0).astype(np.uint8)
        # Manual 3-stop blue -> green -> red fallback (no external deps).
        n = h_u8.astype(np.float32) / 255.0
        r = np.clip(1.5 - np.abs(2.0 * n - 1.5) * 2.0, 0.0, 1.0)
        g = np.clip(1.5 - np.abs(2.0 * n - 1.0) * 2.0, 0.0, 1.0)
        b = np.clip(1.5 - np.abs(2.0 * n - 0.5) * 2.0, 0.0, 1.0)
        return (np.stack([r, g, b], axis=-1) * 255.0).astype(np.uint8)

    def overlay_heatmap(self, base_rgb_uint8: np.ndarray, heatmap_2d: np.ndarray, alpha: float = 0.45) -> np.ndarray:
        """Public reusable primitive: alpha-blends a colorized heatmap onto
        the base RGB display image, resizing the heatmap if necessary."""
        height, width = base_rgb_uint8.shape[:2]
        if heatmap_2d.shape != (height, width):
            resized = Image.fromarray((np.clip(heatmap_2d, 0, 1) * 255).astype(np.uint8)).resize(
                (width, height), Image.BILINEAR
            )
            heatmap_2d = np.asarray(resized, dtype=np.float32) / 255.0
        colored = self._colorize(heatmap_2d)
        overlay = alpha * colored.astype(np.float32) + (1 - alpha) * base_rgb_uint8.astype(np.float32)
        return np.clip(overlay, 0, 255).astype(np.uint8)

    @staticmethod
    def save_visualization(array_uint8: np.ndarray, path: Path) -> None:
        """Public reusable primitive: saves an RGB uint8 array to disk."""
        path.parent.mkdir(parents=True, exist_ok=True)
        Image.fromarray(array_uint8).save(path)

    # ------------------------------------------------------------------
    # Core per-method computations. Each returns (attribution_2d, probs).
    # ------------------------------------------------------------------

    def _compute_gradcam(self, tensor: torch.Tensor, target_idx: int) -> Tuple[np.ndarray, np.ndarray]:
        hooks = ActivationsAndGradients(self.target_layer)
        try:
            inp = tensor.unsqueeze(0).to(self.device).clone().requires_grad_(True)
            logits = self.model(inp)
            probs = torch.sigmoid(logits)[0]
            self.model.zero_grad(set_to_none=True)
            logits[0, target_idx].backward()
            activations = hooks.activations[0]
            gradients = hooks.gradients[0]
            weights = gradients.mean(dim=(1, 2))
            cam = torch.relu((weights.view(-1, 1, 1) * activations).sum(dim=0))
            return self._normalize_map(cam.detach().cpu().numpy()), probs.detach().cpu().numpy()
        finally:
            hooks.remove()

    def _compute_gradcam_plus(self, tensor: torch.Tensor, target_idx: int) -> Tuple[np.ndarray, np.ndarray]:
        hooks = ActivationsAndGradients(self.target_layer)
        try:
            inp = tensor.unsqueeze(0).to(self.device).clone().requires_grad_(True)
            logits = self.model(inp)
            probs = torch.sigmoid(logits)[0]
            self.model.zero_grad(set_to_none=True)
            logits[0, target_idx].backward()
            activations = hooks.activations[0]
            grads = hooks.gradients[0]
            grads_sq = grads.pow(2)
            grads_cube = grads.pow(3)
            sum_act_grad3 = (activations * grads_cube).sum(dim=(1, 2), keepdim=True)
            denom = 2.0 * grads_sq + sum_act_grad3
            denom = torch.where(denom.abs() > 1e-8, denom, torch.ones_like(denom))
            alpha = grads_sq / denom
            weights = (alpha * torch.relu(grads)).sum(dim=(1, 2))
            cam = torch.relu((weights.view(-1, 1, 1) * activations).sum(dim=0))
            return self._normalize_map(cam.detach().cpu().numpy()), probs.detach().cpu().numpy()
        finally:
            hooks.remove()

    def _compute_scorecam(self, tensor: torch.Tensor, target_idx: int) -> Tuple[np.ndarray, np.ndarray]:
        holder: Dict[str, torch.Tensor] = {}

        def _hook(module, inputs, output):
            holder["a"] = output

        handle = self.target_layer.register_forward_hook(_hook)
        try:
            with torch.no_grad():
                inp = tensor.unsqueeze(0).to(self.device)
                base_logits = self.model(inp)
                base_probs = torch.sigmoid(base_logits)[0].cpu().numpy()
                activations = holder["a"][0]  # C, H, W
                in_h, in_w = inp.shape[2], inp.shape[3]

                num_channels = activations.shape[0]
                if num_channels > SCORECAM_MAX_CHANNELS:
                    magnitude = activations.abs().mean(dim=(1, 2))
                    top_idx = torch.topk(magnitude, SCORECAM_MAX_CHANNELS).indices.tolist()
                    self.logger.info(
                        "SCORECAM capped %d -> %d channels by activation magnitude for performance.",
                        num_channels, SCORECAM_MAX_CHANNELS,
                    )
                else:
                    top_idx = list(range(num_channels))

                scores: List[float] = []
                masks: List[torch.Tensor] = []
                for c in top_idx:
                    am = activations[c:c + 1].unsqueeze(0)
                    am_up = F.interpolate(am, size=(in_h, in_w), mode="bilinear", align_corners=False)[0, 0]
                    amin, amax = am_up.min(), am_up.max()
                    if float(amax - amin) < 1e-8:
                        continue
                    norm_mask = (am_up - amin) / (amax - amin)
                    masked_input = inp * norm_mask.unsqueeze(0).unsqueeze(0)
                    out = self.model(masked_input)
                    scores.append(torch.sigmoid(out)[0, target_idx].item())
                    masks.append(norm_mask)

                if not masks:
                    raise RuntimeError("ScoreCAM produced no valid channel masks (degenerate activations).")

                weights = torch.softmax(torch.tensor(scores, device=self.device), dim=0)
                cam = torch.zeros(in_h, in_w, device=self.device)
                for w, m in zip(weights, masks):
                    cam += w * m
                cam = torch.relu(cam).cpu().numpy()
                return self._normalize_map(cam), base_probs
        finally:
            handle.remove()

    def _compute_eigencam(self, tensor: torch.Tensor, target_idx: int) -> Tuple[np.ndarray, np.ndarray]:
        holder: Dict[str, torch.Tensor] = {}

        def _hook(module, inputs, output):
            holder["a"] = output

        handle = self.target_layer.register_forward_hook(_hook)
        try:
            with torch.no_grad():
                inp = tensor.unsqueeze(0).to(self.device)
                logits = self.model(inp)
                probs = torch.sigmoid(logits)[0].cpu().numpy()
                activations = holder["a"][0]  # C, H, W
                c, h, w = activations.shape
                flat = activations.reshape(c, h * w).cpu().numpy()
                flat_centered = flat - flat.mean(axis=0, keepdims=True)
                _, _, vt = np.linalg.svd(flat_centered, full_matrices=False)
                principal = vt[0].reshape(h, w)
                if principal.sum() < 0:
                    principal = -principal
                return self._normalize_map(principal), probs
        finally:
            handle.remove()

    def _compute_guided_backprop(self, tensor: torch.Tensor, target_idx: int) -> Tuple[np.ndarray, np.ndarray]:
        relu_modules = [m for m in self.model.modules() if isinstance(m, nn.ReLU)]
        # register_full_backward_hook conflicts with inplace=True ReLUs (view+inplace
        # autograd error). Temporarily disable inplace, always restored in finally so
        # the frozen RECONSTRUCTED_MODEL is left exactly as Stage 2 built it.
        original_inplace = [m.inplace for m in relu_modules]
        handles = []

        def _clamp_hook(module, grad_input, grad_output):
            g = grad_input[0]
            if g is None:
                return grad_input
            return (torch.clamp(g, min=0.0).clone(),)

        for m in relu_modules:
            m.inplace = False
            try:
                handles.append(m.register_full_backward_hook(_clamp_hook))
            except AttributeError:
                handles.append(m.register_backward_hook(_clamp_hook))
        try:
            inp = tensor.unsqueeze(0).to(self.device).clone().requires_grad_(True)
            logits = self.model(inp)
            probs = torch.sigmoid(logits)[0]
            self.model.zero_grad(set_to_none=True)
            logits[0, target_idx].backward()
            grad = inp.grad[0].detach().cpu().numpy()
            saliency = np.abs(grad).max(axis=0)
            return self._normalize_map(saliency), probs.detach().cpu().numpy()
        finally:
            for h in handles:
                h.remove()
            for m, orig in zip(relu_modules, original_inplace):
                m.inplace = orig

    def _compute_integrated_gradients(
        self, tensor: torch.Tensor, target_idx: int, steps: int = IG_STEPS
    ) -> Tuple[np.ndarray, np.ndarray]:
        input_tensor = tensor.to(self.device)
        baseline = torch.zeros_like(input_tensor)
        grads = []
        for i in range(steps + 1):
            alpha = float(i) / steps
            scaled = (baseline + alpha * (input_tensor - baseline)).unsqueeze(0).clone().requires_grad_(True)
            logits = self.model(scaled)
            self.model.zero_grad(set_to_none=True)
            logits[0, target_idx].backward()
            grads.append(scaled.grad[0].detach())
        grads_stack = torch.stack(grads, dim=0)
        avg_grads = (grads_stack[:-1] + grads_stack[1:]) / 2.0
        avg_grad = avg_grads.mean(dim=0)
        attribution = ((input_tensor - baseline) * avg_grad).sum(dim=0).cpu().numpy()
        with torch.no_grad():
            probs = torch.sigmoid(self.model(input_tensor.unsqueeze(0)))[0].cpu().numpy()
        return self._normalize_map(np.abs(attribution)), probs

    def _compute_occlusion(
        self,
        tensor: torch.Tensor,
        target_idx: int,
        patch_size: int = OCCLUSION_PATCH_SIZE,
        stride: int = OCCLUSION_STRIDE,
    ) -> Tuple[np.ndarray, np.ndarray]:
        with torch.no_grad():
            input_tensor = tensor.unsqueeze(0).to(self.device)
            base_logits = self.model(input_tensor)
            base_prob = torch.sigmoid(base_logits)[0, target_idx].item()
            probs = torch.sigmoid(base_logits)[0].cpu().numpy()

            _, h, w = tensor.shape
            heatmap = np.zeros((h, w), dtype=np.float32)
            counts = np.zeros((h, w), dtype=np.float32)

            for y in range(0, h, stride):
                for x in range(0, w, stride):
                    y2, x2 = min(y + patch_size, h), min(x + patch_size, w)
                    occluded = input_tensor.clone()
                    occluded[:, :, y:y2, x:x2] = 0.0
                    out = self.model(occluded)
                    prob = torch.sigmoid(out)[0, target_idx].item()
                    drop = base_prob - prob
                    heatmap[y:y2, x:x2] += drop
                    counts[y:y2, x:x2] += 1.0

            counts[counts == 0] = 1.0
            heatmap = heatmap / counts
            return self._normalize_map(heatmap), probs

    # ------------------------------------------------------------------
    # Generic run wrapper: EVERY method goes through here so that a
    # failure in one method is fully isolated and never aborts the rest.
    # ------------------------------------------------------------------

    def _run_method(
        self,
        method_name: str,
        image: Image.Image,
        target_class: Optional[Any],
        sample_id: str,
        compute_fn: Callable[..., Tuple[np.ndarray, np.ndarray]],
        extra_params: Optional[Dict[str, Any]] = None,
    ) -> MethodResult:
        start = time.time()
        availability = self.method_availability[method_name]

        if not availability.available:
            msg = f"method disabled: {availability.reason}"
            self.logger.warning("METHOD %s skipped sample=%s reason=%s", method_name, sample_id, msg)
            return MethodResult(method=method_name, sample_id=sample_id, success=False, error=msg)

        try:
            self.logger.info("START method=%s sample=%s", method_name, sample_id)
            tensor = self._to_input_tensor(image)

            with torch.no_grad():
                probe_logits = self.model(tensor.unsqueeze(0).to(self.device))
                probe_probs = torch.sigmoid(probe_logits)[0]

            pred_idx, pred_name, target_idx, target_name, pred_conf = self._resolve_target(probe_probs, target_class)

            attribution, _ = compute_fn(tensor, target_idx, **(extra_params or {}))

            no_nan = not bool(np.isnan(attribution).any())
            no_inf = not bool(np.isinf(attribution).any())
            if not no_nan or not no_inf:
                raise RuntimeError(f"{method_name} produced NaN/Inf values in the attribution map.")

            base_rgb = self._to_display_rgb(tensor)
            overlay = self.overlay_heatmap(base_rgb, attribution)
            colorized = self._colorize(attribution)

            heatmap_path = self.dirs[method_name] / f"{sample_id}_heatmap.png"
            overlay_path = self.dirs["overlays"] / f"{sample_id}_{method_name}_overlay.png"
            raw_path = self.dirs["metadata"] / f"{sample_id}_{method_name}_raw.npy"

            self.save_visualization(colorized, heatmap_path)
            self.save_visualization(overlay, overlay_path)
            np.save(raw_path, attribution.astype(np.float32))

            elapsed_ms = (time.time() - start) * 1000.0
            result = MethodResult(
                method=method_name,
                sample_id=sample_id,
                success=True,
                execution_time_ms=round(elapsed_ms, 3),
                predicted_class=pred_name,
                predicted_class_index=pred_idx,
                target_class=target_name,
                target_class_index=target_idx,
                confidence=round(pred_conf, 6),
                heatmap_path=str(heatmap_path),
                overlay_path=str(overlay_path),
                raw_attribution_path=str(raw_path),
                heatmap_shape=list(attribution.shape),
                no_nan=no_nan,
                no_inf=no_inf,
                error=None,
            )
            self.logger.info("FINISH method=%s sample=%s time_ms=%.2f", method_name, sample_id, elapsed_ms)
            return result
        except Exception as exc:  # noqa: BLE001 -- isolate failure, never propagate
            elapsed_ms = (time.time() - start) * 1000.0
            self.logger.error("METHOD %s FAILED sample=%s error=%s", method_name, sample_id, exc)
            return MethodResult(
                method=method_name, sample_id=sample_id, success=False,
                execution_time_ms=round(elapsed_ms, 3), error=str(exc),
            )
        finally:
            self.model.zero_grad(set_to_none=True)

    # ------------------------------------------------------------------
    # Public reusable API
    # ------------------------------------------------------------------

    def generate_gradcam(self, image: Image.Image, target_class: Optional[Any] = None, sample_id: str = "sample") -> MethodResult:
        return self._run_method("gradcam", image, target_class, sample_id, self._compute_gradcam)

    def generate_gradcam_plus(self, image: Image.Image, target_class: Optional[Any] = None, sample_id: str = "sample") -> MethodResult:
        return self._run_method("gradcam_plus", image, target_class, sample_id, self._compute_gradcam_plus)

    def generate_scorecam(self, image: Image.Image, target_class: Optional[Any] = None, sample_id: str = "sample") -> MethodResult:
        return self._run_method("scorecam", image, target_class, sample_id, self._compute_scorecam)

    def generate_eigencam(self, image: Image.Image, target_class: Optional[Any] = None, sample_id: str = "sample") -> MethodResult:
        return self._run_method("eigencam", image, target_class, sample_id, self._compute_eigencam)

    def generate_guided_backprop(self, image: Image.Image, target_class: Optional[Any] = None, sample_id: str = "sample") -> MethodResult:
        return self._run_method("guided_backprop", image, target_class, sample_id, self._compute_guided_backprop)

    def generate_integrated_gradients(
        self, image: Image.Image, target_class: Optional[Any] = None, sample_id: str = "sample", steps: int = IG_STEPS
    ) -> MethodResult:
        return self._run_method(
            "integrated_gradients", image, target_class, sample_id, self._compute_integrated_gradients,
            extra_params={"steps": steps},
        )

    def generate_occlusion(
        self, image: Image.Image, target_class: Optional[Any] = None, sample_id: str = "sample",
        patch_size: int = OCCLUSION_PATCH_SIZE, stride: int = OCCLUSION_STRIDE,
    ) -> MethodResult:
        return self._run_method(
            "occlusion", image, target_class, sample_id, self._compute_occlusion,
            extra_params={"patch_size": patch_size, "stride": stride},
        )

    def batch_generate(
        self,
        images: List[Tuple[str, Image.Image]],
        methods: Optional[List[str]] = None,
        target_class: Optional[Any] = None,
    ) -> List[MethodResult]:
        """Runs the requested (default: all implemented) methods across a
        batch of (sample_id, PIL.Image) pairs. Every method/image
        combination is isolated -- one failure never stops the rest."""
        methods = methods or METHOD_NAMES
        batch_results: List[MethodResult] = []
        for sample_id, image in images:
            for method_name in methods:
                fn = self._method_dispatch.get(method_name)
                if fn is None:
                    self.logger.warning("BATCH unknown method '%s' requested; skipping.", method_name)
                    continue
                result = fn(image, target_class=target_class, sample_id=sample_id)
                batch_results.append(result)
        self.results.extend(batch_results)
        return batch_results

    def export_results(self, engineering_validation: Optional[Dict[str, Any]] = None) -> Dict[str, Any]:
        """Writes all required JSON artifacts to the metadata directory
        and returns the top-level explainability_summary dict."""
        metadata_dir = self.dirs["metadata"]

        summary = {
            "methods_implemented": METHOD_NAMES,
            "methods_available": {k: v.available for k, v in self.method_availability.items()},
            "methods_disabled": {k: v.reason for k, v in self.method_availability.items() if not v.available},
            "layer_name": self.layer_name,
            "layer_type": type(self.target_layer).__name__,
            "feature_dim": self.feature_dim,
            "activation_size": self.activation_size,
            "total_results": len(self.results),
            "successful_results": sum(1 for r in self.results if r.success),
            "failed_results": sum(1 for r in self.results if not r.success),
            "class_names": self.class_names,
            "num_classes": len(self.class_names),
            "model_version": getattr(self.engine, "model_version", None),
            "model_fingerprint_sha256": getattr(self.engine, "model_fingerprint", None),
            "backbone": self.model_registry.backbone,
        }
        save_json(metadata_dir / "explainability_summary.json", summary)

        gradcam_results = [r.to_dict() for r in self.results if r.method == "gradcam"]
        save_json(metadata_dir / "gradcam_metadata.json", {
            "layer_name": self.layer_name,
            "feature_dim": self.feature_dim,
            "activation_size": self.activation_size,
            "results": gradcam_results,
        })

        save_json(metadata_dir / "layer_registry.json", {
            "selected_layer": self.layer_name,
            "layer_type": type(self.target_layer).__name__,
            "feature_dim": self.feature_dim,
            "activation_size": self.activation_size,
            "backbone": self.model_registry.backbone,
            "discovery_method": "last_nn.Conv2d_via_named_modules",
        })

        save_json(metadata_dir / "method_registry.json", {
            k: v.to_dict() for k, v in self.method_availability.items()
        })

        perf: Dict[str, Any] = {}
        for method_name in METHOD_NAMES:
            times = [r.execution_time_ms for r in self.results if r.method == method_name and r.success]
            perf[method_name] = {
                "count": len(times),
                "mean_ms": float(np.mean(times)) if times else None,
                "min_ms": float(np.min(times)) if times else None,
                "max_ms": float(np.max(times)) if times else None,
            }
        save_json(metadata_dir / "performance_summary.json", perf)

        if engineering_validation is not None:
            save_json(metadata_dir / "engineering_validation.json", engineering_validation)

        return summary


# ======================================================================
# ENGINEERING VALIDATION
# ======================================================================

def run_stage06_engineering_validation(
    engine: ExplainabilityEngine, results: List[MethodResult], logger: logging.Logger,
) -> Dict[str, Any]:
    checks: Dict[str, bool] = {}
    warnings: List[str] = []

    successful = [r for r in results if r.success]
    failed = [r for r in results if not r.success]

    checks["layer_found"] = engine.target_layer is not None
    checks["forward_hook_works"] = engine.activation_size is not None
    checks["backward_hook_works"] = any(r.success for r in results if r.method in ("gradcam", "gradcam_plus"))
    checks["heatmap_dimensions_correct"] = all(r.heatmap_shape is not None and len(r.heatmap_shape) == 2 for r in successful)
    checks["no_nan"] = all(r.no_nan for r in successful) if successful else True
    checks["no_inf"] = all(r.no_inf for r in successful) if successful else True

    overlay_size_ok = True
    if successful:
        try:
            sample = successful[0]
            with Image.open(sample.overlay_path) as im:
                overlay_size_ok = im.size == (engine.preprocessing_config.resize_width, engine.preprocessing_config.resize_height)
        except Exception as exc:  # noqa: BLE001
            overlay_size_ok = False
            warnings.append(f"Could not verify overlay size: {exc}")
    checks["overlay_size_correct"] = overlay_size_ok

    checks["color_map_valid"] = True  # cv2 -> matplotlib -> manual fallback chain always yields a valid RGB map
    checks["execution_time_measured"] = all(r.execution_time_ms is not None and r.execution_time_ms >= 0 for r in results)
    checks["every_image_saved"] = all(
        Path(r.heatmap_path).exists() and Path(r.overlay_path).exists() for r in successful
    ) if successful else False

    disabled = [k for k, v in engine.method_availability.items() if not v.available]
    checks["failures_isolated"] = True  # by construction: every method call is wrapped in try/except

    if failed:
        warnings.append(f"{len(failed)} method execution(s) failed but were isolated; pipeline continued.")
    if disabled:
        warnings.append(f"Method(s) disabled due to unavailable capability: {disabled}")
    if not _CV2_AVAILABLE:
        warnings.append("cv2 unavailable; colorization used matplotlib or manual fallback instead.")

    fatal_errors: List[str] = []
    if not successful:
        fatal_errors.append("No method execution succeeded across any sample.")

    for check_name, ok in checks.items():
        if not ok:
            logger.warning("ENGINEERING CHECK FAILED: %s", check_name)

    passed = len(fatal_errors) == 0 and all(checks.values())
    return {"checks": checks, "warnings": warnings, "fatal_errors": fatal_errors, "passed": passed}


# ======================================================================
# SYNTHETIC VALIDATION IMAGES
# ======================================================================

def generate_stage06_synthetic_images(
    output_dir: Path, count: int, height: int, width: int, seed: int, logger: logging.Logger,
) -> List[Tuple[str, str]]:
    """Deterministic synthetic RGB images used ONLY to exercise the
    explainability pipeline end-to-end. No NIH dataset traversal."""
    output_dir.mkdir(parents=True, exist_ok=True)
    rng = np.random.RandomState(seed)
    paths: List[Tuple[str, str]] = []

    generators: List[Tuple[str, Callable[[], np.ndarray]]] = [
        ("noise", lambda: rng.randint(0, 256, size=(height, width, 3), dtype=np.uint8)),
        ("checkerboard", lambda: _checkerboard(height, width)),
        ("radial_gradient", lambda: _radial_gradient(height, width)),
        ("stripes", lambda: _stripes(height, width)),
    ]

    for i in range(count):
        name, gen = generators[i % len(generators)]
        array = gen()
        img = Image.fromarray(array, mode="RGB")
        identifier = f"stage06_synthetic_{i:02d}_{name}"
        path = output_dir / f"{identifier}.png"
        img.save(path)
        paths.append((identifier, str(path)))

    logger.info("VALIDATION IMAGES generated %d synthetic samples in %s", len(paths), output_dir)
    return paths


def _checkerboard(h: int, w: int, block: int = 16) -> np.ndarray:
    yy, xx = np.indices((h, w))
    pattern = (((yy // block) + (xx // block)) % 2).astype(np.uint8) * 255
    return np.stack([pattern, 255 - pattern, pattern], axis=-1).astype(np.uint8)


def _radial_gradient(h: int, w: int) -> np.ndarray:
    yy, xx = np.indices((h, w)).astype(np.float32)
    cy, cx = h / 2.0, w / 2.0
    dist = np.sqrt((yy - cy) ** 2 + (xx - cx) ** 2)
    dist = (dist / dist.max() * 255.0).astype(np.uint8)
    return np.stack([dist, 255 - dist, np.full_like(dist, 128)], axis=-1)


def _stripes(h: int, w: int, stripe_width: int = 12) -> np.ndarray:
    xx = np.indices((h, w))[1]
    pattern = (((xx // stripe_width) % 2) * 255).astype(np.uint8)
    return np.stack([pattern, pattern, 255 - pattern], axis=-1)


# ======================================================================
# STAGE ORCHESTRATION
# ======================================================================

def run_stage06(
    inference_engine: Any,
    reconstructed_model: nn.Module,
    model_registry: Any,
    metadata_registry: Any,
    threshold_registry: Any,
    config_registry: Any,
    artifact_registry: Any,
    export_registry: Any,
) -> Dict[str, Any]:
    start_time = time.time()
    stage_root = OUTPUT_ROOT / STAGE6_DIR_NAME
    dirs = make_stage_dirs(stage_root)
    logger = build_logger(dirs["logs"])

    logger.info("START Sprint05-Stage06 status=RUNNING")

    engine = ExplainabilityEngine(
        inference_engine=inference_engine,
        reconstructed_model=reconstructed_model,
        model_registry=model_registry,
        metadata_registry=metadata_registry,
        threshold_registry=threshold_registry,
        config_registry=config_registry,
        artifact_registry=artifact_registry,
        export_registry=export_registry,
        output_dir=stage_root,
        logger=logger,
    )

    cfg = inference_engine.preprocessing_config
    sample_paths = generate_stage06_synthetic_images(
        dirs["inputs"], NUM_SYNTHETIC_SAMPLES, cfg.resize_height, cfg.resize_width, SYNTHETIC_SAMPLE_SEED, logger,
    )

    identifiers = [pid for pid, _ in sample_paths]
    raw_paths = [p for _, p in sample_paths]
    predictions = inference_engine.predict_batch(raw_paths, identifiers)

    successful_samples = [
        (identifiers[i], raw_paths[i]) for i, pred in enumerate(predictions) if pred.success
    ]
    logger.info(
        "RUNTIME SAMPLES %d/%d synthetic images passed InferenceEngine validation successfully.",
        len(successful_samples), len(sample_paths),
    )
    if not successful_samples:
        raise RuntimeError(
            "No successful runtime samples available for explainability generation "
            "(all synthetic images failed InferenceEngine validation)."
        )

    images_for_explain = [(sid, Image.open(p).convert("RGB")) for sid, p in successful_samples]
    all_results = engine.batch_generate(images_for_explain, methods=METHOD_NAMES, target_class=None)

    engineering_validation = run_stage06_engineering_validation(engine, all_results, logger)
    summary = engine.export_results(engineering_validation)

    if not engineering_validation["passed"]:
        logger.error("FINISH Sprint05-Stage06 status=FAILED (engineering validation did not pass)")
    else:
        logger.info("FINISH Sprint05-Stage06 status=OK")

    elapsed = time.time() - start_time
    methods_passed = sorted({r.method for r in all_results if r.success})

    stage06_summary = {
        "stage": STAGE6_DIR_NAME,
        "status": "OK" if engineering_validation["passed"] else "FAILED",
        "elapsed_seconds": round(elapsed, 3),
        "methods_implemented": METHOD_NAMES,
        "methods_passed": methods_passed,
        "layers_registered": 1,
        "layer_name": engine.layer_name,
        "images_generated": sum(1 for r in all_results if r.success) * 2,  # heatmap + overlay
        "samples_used": len(successful_samples),
        "artifacts_saved": 6,
        "engineering_checks_passed": sum(engineering_validation["checks"].values()),
        "engineering_checks_total": len(engineering_validation["checks"]),
        "warnings_count": len(engineering_validation["warnings"]),
        "output_directory": str(dirs["stage_root"]),
    }
    save_json(dirs["stage_root"] / "stage06_summary.json", stage06_summary)

    print("=" * 70)
    print("SPRINT05 — STAGE06")
    print("EXPLAINABILITY RUNTIME")
    print("=" * 70)
    print(f"Methods Implemented : {len(METHOD_NAMES)} ({', '.join(METHOD_NAMES)})")
    print(f"Methods Passed       : {len(methods_passed)}/{len(METHOD_NAMES)}")
    print(f"Layers Registered    : 1 ({engine.layer_name})")
    print(f"Images Generated     : {stage06_summary['images_generated']}")
    print(f"Artifacts Saved      : {stage06_summary['artifacts_saved']} JSON files")
    print(f"Engineering Checks   : {stage06_summary['engineering_checks_passed']}/{stage06_summary['engineering_checks_total']}")
    print(f"Warnings             : {stage06_summary['warnings_count']}")
    print(f"Output Directory     : {dirs['stage_root']}")
    print()
    print(f"Stage 6 : {stage06_summary['status']}")

    return {
        "EXPLAINABILITY_ENGINE": engine,
        "results": all_results,
        "engineering_validation": engineering_validation,
        "summary": stage06_summary,
    }


if __name__ == "__main__":
    _stage06_result = run_stage06(
        inference_engine=INFERENCE_ENGINE,
        reconstructed_model=RECONSTRUCTED_MODEL,
        model_registry=MODEL_REGISTRY,
        metadata_registry=METADATA_REGISTRY,
        threshold_registry=THRESHOLD_REGISTRY,
        config_registry=CONFIG_REGISTRY,
        artifact_registry=ARTIFACT_REGISTRY,
        export_registry=EXPORT_REGISTRY,
    )
    EXPLAINABILITY_ENGINE = _stage06_result["EXPLAINABILITY_ENGINE"]

2026-07-08 10:33:38 | INFO     | START Sprint05-Stage06 status=RUNNING
2026-07-08 10:33:38 | INFO     | LAYER DISCOVERY selected target layer='backbone.features.denseblock4.denselayer16.conv2' type=Conv2d feature_dim=32
2026-07-08 10:33:38 | INFO     | LAYER DISCOVERY activation size for layer='backbone.features.denseblock4.denselayer16.conv2': [32, 7, 7] (forward hook verified)
2026-07-08 10:33:38 | INFO     | COLORMAP backend=cv2 (COLORMAP_JET)
/tmp/ipykernel_195/882028464.py:928: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  img = Image.fromarray(array, mode="RGB")
2026-07-08 10:33:38 | INFO     | VALIDATION IMAGES generated 4 synthetic samples in /kaggle/working/sprint05_deployment/stage06_explainability/images/synthetic_inputs
2026-07-08 10:33:38 | INFO     | RUNTIME SAMPLES 4/4 synthetic images passed InferenceEngine validation successfully.
2026-07-08 10:33:38 | INFO     | START method=gradcam sample=stage06_synthetic_00_noise

SPRINT05 — STAGE06
EXPLAINABILITY RUNTIME
Methods Implemented : 7 (gradcam, gradcam_plus, scorecam, eigencam, guided_backprop, integrated_gradients, occlusion)
Methods Passed       : 7/7
Layers Registered    : 1 (backbone.features.denseblock4.denselayer16.conv2)
Images Generated     : 56
Artifacts Saved      : 6 JSON files
Engineering Checks   : 11/11
Warnings             : 0
Output Directory     : /kaggle/working/sprint05_deployment/stage06_explainability

Stage 6 : OK


In [12]:
"""
VisionServeAI - Sprint 05
Stage 7: Deployment Packaging & Production Readiness
======================================================================
Single-responsibility stage: transforms the outputs of Stages 1-6 into
a deployment-ready package with complete metadata, validation,
manifests, and reproducibility information.

STRICT SCOPE:
    - No retraining. No inference. No re-export. No evaluation metrics.
    - No explainability regeneration. No dataset rediscovery.
    - Stages 1-6 are FROZEN. CONFIG_REGISTRY, ARTIFACT_REGISTRY,
      MODEL_REGISTRY, METADATA_REGISTRY, THRESHOLD_REGISTRY,
      EXPORT_REGISTRY, INFERENCE_ENGINE, RECONSTRUCTED_MODEL are
      consumed as the in-memory Python objects earlier stages produced.

      Building a package MANIFEST inherently requires reading the bytes
      of already-produced artifact files (to checksum/size/copy them)
      exactly as Stage 2's ArtifactRegistry did for Stage 1's discovery
      output -- this is packaging of frozen outputs, never rediscovery
      of raw datasets and never recomputation of upstream results. The
      single exception is a read-only reference to Stage 6's already
      -generated 'explainability_summary.json' (to surface which
      explainability methods are available in deployment metadata);
      that file is never rewritten or regenerated by this stage.
======================================================================
"""

from __future__ import annotations

import hashlib
import json
import logging
import shutil
import subprocess
import sys
import time
from dataclasses import dataclass, field, asdict
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import torch
import torchvision

# ======================================================================
# CONSTANTS
# ======================================================================

OUTPUT_ROOT = Path("/kaggle/working/sprint05_deployment")

STAGE1_DIR = OUTPUT_ROOT / "stage01_environment"
STAGE2_DIR = OUTPUT_ROOT / "stage02_registry"
STAGE3_DIR = OUTPUT_ROOT / "stage03_export"
STAGE4_DIR = OUTPUT_ROOT / "stage04_runtime"
STAGE5_DIR = OUTPUT_ROOT / "stage05_validation"
STAGE6_DIR = OUTPUT_ROOT / "stage06_explainability"

STAGE7_DIR_NAME = "stage07_deployment_package"

PACKAGE_VERSION = "1.0.0"

# Runtimes this pipeline can plausibly serve from -- pruned down to only
# those whose Stage 3 export actually succeeded when the matrix is built.
CANDIDATE_RUNTIMES = ["pytorch", "torchscript", "onnxruntime"]

# Documented assumptions (explicit, logged fallbacks -- never silent),
# mirroring the "documented default" convention Stage 3 used for input
# resolution. These are the practical support windows for the pinned
# framework stack, not values discovered from any registry.
SUPPORTED_PYTHON_MIN = (3, 9)
SUPPORTED_PYTHON_MAX_EXCLUSIVE = (3, 13)
SUPPORTED_TORCH_MAJOR_MIN = 2

# Batch-size grid mirroring Stage 5's own BATCH_SIZES contract. Stage 5 is
# frozen and out of scope to re-invoke here, so this documents (not
# re-benchmarks) which batch sizes the deployment target was validated
# against upstream.
DOCUMENTED_VALIDATED_BATCH_SIZES = [1, 2, 4, 8, 16, 32]

# Explainability methods implemented by Stage 6's ExplainabilityEngine
# (frozen contract, METHOD_NAMES). Used only as a labeling fallback if
# Stage 6's own explainability_summary.json cannot be read.
EXPLAINABILITY_METHODS_CONTRACT = [
    "gradcam", "gradcam_plus", "scorecam", "eigencam",
    "guided_backprop", "integrated_gradients", "occlusion",
]


# ======================================================================
# LOGGING (self-contained, same format/pattern as Stages 1-6)
# ======================================================================

def build_logger(log_dir: Path) -> logging.Logger:
    log_dir.mkdir(parents=True, exist_ok=True)
    logger = logging.getLogger("sprint05.stage07")
    logger.setLevel(logging.INFO)
    logger.handlers.clear()
    logger.propagate = False

    fmt = logging.Formatter(
        fmt="%(asctime)s | %(levelname)-8s | %(message)s",
        datefmt="%Y-%m-%d %H:%M:%S",
    )

    file_handler = logging.FileHandler(log_dir / "stage07_deployment_package.log", mode="w")
    file_handler.setFormatter(fmt)
    logger.addHandler(file_handler)

    stream_handler = logging.StreamHandler(sys.stdout)
    stream_handler.setFormatter(fmt)
    logger.addHandler(stream_handler)

    return logger


def get_resource_usage() -> Dict[str, Any]:
    import psutil
    vm = psutil.virtual_memory()
    usage: Dict[str, Any] = {
        "cpu_percent": psutil.cpu_percent(interval=0.2),
        "ram_percent": vm.percent,
        "ram_used_gb": round(vm.used / (1024 ** 3), 2),
        "ram_total_gb": round(vm.total / (1024 ** 3), 2),
    }
    if torch.cuda.is_available():
        usage["gpu_memory_allocated_gb"] = round(torch.cuda.memory_allocated() / (1024 ** 3), 3)
        usage["gpu_memory_reserved_gb"] = round(torch.cuda.memory_reserved() / (1024 ** 3), 3)
    return usage


def log_resources(logger: logging.Logger, tag: str) -> Dict[str, Any]:
    usage = get_resource_usage()
    logger.info(
        "RESOURCES [%s] cpu=%.1f%% ram=%.1f%%(%.1fGB/%.1fGB)",
        tag, usage["cpu_percent"], usage["ram_percent"], usage["ram_used_gb"], usage["ram_total_gb"],
    )
    return usage


def save_json(path: Path, data: Any) -> None:
    path.write_text(json.dumps(data, indent=2, default=str))


def load_json(path: Path) -> Any:
    return json.loads(path.read_text())


def sha256_of_file(path: Path, chunk_size: int = 1 << 20) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            h.update(chunk)
    return h.hexdigest()


def make_stage_dirs(stage_root: Path) -> Dict[str, Path]:
    dirs = {
        "stage_root": stage_root,
        "logs": stage_root / "logs",
        "manifests": stage_root / "manifests",
        "metadata": stage_root / "metadata",
        "integrity": stage_root / "integrity",
        "compatibility": stage_root / "compatibility",
        "reports": stage_root / "reports",
        "package": stage_root / "package",
    }
    for d in dirs.values():
        d.mkdir(parents=True, exist_ok=True)
    return dirs


# ======================================================================
# DATACLASSES
# ======================================================================

@dataclass
class ManifestEntry:
    artifact_id: str
    description: str
    file_path: str
    stage_of_origin: str
    is_critical: bool
    exists: bool = False
    size_bytes: Optional[int] = None
    sha256: Optional[str] = None
    created_utc: Optional[str] = None
    source_path: Optional[str] = None
    errors: List[str] = field(default_factory=list)

    def to_dict(self) -> Dict[str, Any]:
        return asdict(self)


@dataclass
class DeploymentManifest:
    package_version: str
    generated_utc: str
    entries: Dict[str, ManifestEntry]
    total_artifacts: int
    total_size_bytes: int
    missing_critical: List[str]

    def to_dict(self) -> Dict[str, Any]:
        return {
            "package_version": self.package_version,
            "generated_utc": self.generated_utc,
            "total_artifacts": self.total_artifacts,
            "total_size_bytes": self.total_size_bytes,
            "missing_critical": self.missing_critical,
            "entries": {k: v.to_dict() for k, v in self.entries.items()},
        }


# ======================================================================
# HELPERS -- LOCATING FROZEN UPSTREAM ARTIFACTS (no rediscovery, just
# reading paths already recorded by earlier, frozen stages)
# ======================================================================

def find_artifact_record(artifact_registry: Any, category: str, filename: str) -> Optional[Any]:
    """Look up a Stage 2 ArtifactRecord by its frozen artifact_id convention
    ('{category}::{filename}'). Read-only lookup -- no rediscovery."""
    return artifact_registry.records.get(f"{category}::{filename}")


def load_explainability_summary(logger: logging.Logger) -> Optional[Dict[str, Any]]:
    """Read-only reference to Stage 6's already-generated, frozen
    explainability_summary.json. Never regenerated, never rewritten --
    only surfaced here so deployment metadata can report which
    explainability methods are available."""
    path = STAGE6_DIR / "metadata" / "explainability_summary.json"
    if not path.exists():
        logger.warning("EXPLAINABILITY explainability_summary.json not found at %s; methods_available will be omitted.", path)
        return None
    try:
        return load_json(path)
    except (json.JSONDecodeError, OSError) as exc:
        logger.warning("EXPLAINABILITY failed to read explainability_summary.json: %s", exc)
        return None


def get_git_commit(logger: logging.Logger) -> Optional[str]:
    """Best-effort build identifier. Returns None (logged, not fatal) if
    this is not a git checkout or git is unavailable -- common in a
    Kaggle kernel."""
    try:
        result = subprocess.run(
            ["git", "rev-parse", "HEAD"], capture_output=True, text=True, timeout=5,
        )
        if result.returncode == 0:
            return result.stdout.strip()
        logger.info("REPRODUCIBILITY no git commit identifier available (not a git checkout).")
        return None
    except (OSError, subprocess.SubprocessError) as exc:
        logger.info("REPRODUCIBILITY git identifier unavailable: %s", exc)
        return None


def copy_into_package(src: Optional[Path], dest: Path, logger: logging.Logger) -> List[str]:
    """Best-effort byte-identical copy of a frozen upstream artifact into
    the self-contained package/ directory. Errors are returned (never
    raised) so manifest building can continue and report them."""
    errors: List[str] = []
    if src is None:
        errors.append("Source path not available (not discovered/recorded upstream).")
        return errors
    if not src.exists() or not src.is_file():
        errors.append(f"Source path recorded upstream no longer exists on disk: {src}")
        return errors
    try:
        dest.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(str(src), str(dest))
    except OSError as exc:
        errors.append(f"Failed to copy '{src}' -> '{dest}': {exc}")
    return errors


# ======================================================================
# MANIFEST GENERATION
# ======================================================================

def build_manifest_entry(
    artifact_id: str, description: str, path: Path, stage_of_origin: str,
    is_critical: bool, logger: logging.Logger, source_path: Optional[Path] = None,
    extra_errors: Optional[List[str]] = None,
) -> ManifestEntry:
    entry = ManifestEntry(
        artifact_id=artifact_id, description=description, file_path=str(path),
        stage_of_origin=stage_of_origin, is_critical=is_critical,
        source_path=str(source_path) if source_path is not None else None,
        errors=list(extra_errors) if extra_errors else [],
    )

    if not path.exists() or not path.is_file():
        entry.errors.append(f"Artifact file does not exist: {path}")
        if is_critical:
            logger.error("MANIFEST[%s] CRITICAL artifact missing: %s", artifact_id, path)
        else:
            logger.warning("MANIFEST[%s] optional artifact missing: %s", artifact_id, path)
        return entry

    stat = path.stat()
    entry.exists = True
    entry.size_bytes = stat.st_size
    # st_ctime on Linux is inode-change time, the closest available proxy
    # for a creation timestamp (POSIX has no true birth-time guarantee).
    entry.created_utc = datetime.fromtimestamp(stat.st_ctime, tz=timezone.utc).isoformat()
    try:
        entry.sha256 = sha256_of_file(path)
    except OSError as exc:
        entry.errors.append(f"Failed to hash file: {exc}")

    logger.info(
        "MANIFEST[%s] path=%s size=%d sha256=%s critical=%s",
        artifact_id, path, entry.size_bytes, entry.sha256, is_critical,
    )
    return entry


def build_deployment_manifest(
    dirs: Dict[str, Path],
    config_registry: Any, artifact_registry: Any, model_registry: Any,
    metadata_registry: Any, threshold_registry: Any, export_registry: Any,
    inference_engine: Any, logger: logging.Logger,
) -> DeploymentManifest:
    entries: Dict[str, ManifestEntry] = {}
    package_dir = dirs["package"]

    # ---- 1. TorchScript model (Stage 3 export, copied into package/) ----
    ts_src = Path(export_registry.torchscript.output_path)
    ts_dest = package_dir / "model.ts"
    ts_copy_errors = copy_into_package(ts_src, ts_dest, logger)
    entries["torchscript_model"] = build_manifest_entry(
        "torchscript_model", "TorchScript-exported production model (Stage 3).",
        ts_dest, "stage03_export", True, logger, source_path=ts_src, extra_errors=ts_copy_errors,
    )

    # ---- 2. ONNX model (Stage 3 export, copied into package/) ----
    onnx_src = Path(export_registry.onnx.output_path)
    onnx_dest = package_dir / "model.onnx"
    onnx_copy_errors = copy_into_package(onnx_src, onnx_dest, logger)
    entries["onnx_model"] = build_manifest_entry(
        "onnx_model", "ONNX-exported production model (Stage 3).",
        onnx_dest, "stage03_export", True, logger, source_path=onnx_src, extra_errors=onnx_copy_errors,
    )

    # ---- 3. Threshold registry (Stage 2, byte-identical copy of source) ----
    threshold_src = Path(threshold_registry.source_path) if threshold_registry.source_path else None
    threshold_dest = package_dir / "optimal_thresholds.json"
    threshold_copy_errors = copy_into_package(threshold_src, threshold_dest, logger)
    entries["threshold_registry"] = build_manifest_entry(
        "threshold_registry", "Per-class decision thresholds (Stage 2, sourced from sprint04_evaluation).",
        threshold_dest, "stage02_registry", True, logger, source_path=threshold_src,
        extra_errors=threshold_copy_errors,
    )

    # ---- 4. Disease registry (Stage 2, byte-identical copy of source) ----
    disease_record = find_artifact_record(artifact_registry, "sprint03", "disease_registry.json")
    disease_src = Path(disease_record.path) if disease_record and disease_record.path else None
    disease_dest = package_dir / "disease_registry.json"
    disease_copy_errors = copy_into_package(disease_src, disease_dest, logger)
    entries["disease_registry"] = build_manifest_entry(
        "disease_registry", "Canonical disease/class-name ordering (Stage 2, sourced from sprint03).",
        disease_dest, "stage02_registry", True, logger, source_path=disease_src,
        extra_errors=disease_copy_errors,
    )

    # ---- 5. Configuration (Stage 1's frozen deployment_config.json) ----
    config_src = STAGE1_DIR / "configs" / "deployment_config.json"
    config_dest = package_dir / "deployment_config.json"
    config_copy_errors = copy_into_package(config_src, config_dest, logger)
    entries["configuration"] = build_manifest_entry(
        "configuration", "Frozen deployment configuration (Stage 1).",
        config_dest, "stage01_environment", True, logger, source_path=config_src,
        extra_errors=config_copy_errors,
    )

    # ---- 6. Export metadata (serialized fresh from in-memory EXPORT_REGISTRY) ----
    export_metadata_dest = package_dir / "export_metadata.json"
    save_json(export_metadata_dest, export_registry.to_dict())
    entries["export_metadata"] = build_manifest_entry(
        "export_metadata", "TorchScript/ONNX export + numerical-validation results (Stage 3).",
        export_metadata_dest, "stage03_export", True, logger,
    )

    # ---- 7. Runtime metadata (serialized fresh from in-memory INFERENCE_ENGINE) ----
    runtime_metadata = {
        "model_version": inference_engine.model_version,
        "model_fingerprint_sha256": inference_engine.model_fingerprint,
        "device": str(inference_engine.device),
        "class_names": inference_engine.class_names,
        "thresholds": inference_engine.thresholds,
        "preprocessing_config": inference_engine.preprocessing_config.to_dict(),
    }
    runtime_metadata_dest = package_dir / "runtime_metadata.json"
    save_json(runtime_metadata_dest, runtime_metadata)
    entries["runtime_metadata"] = build_manifest_entry(
        "runtime_metadata", "Production InferenceEngine configuration (Stage 4).",
        runtime_metadata_dest, "stage04_runtime", True, logger,
    )

    # ---- 8. Explainability metadata (Stage 6 output, referenced read-only,
    #          never regenerated/rewritten per stage constraints) ----
    explainability_src = STAGE6_DIR / "metadata" / "explainability_summary.json"
    entries["explainability_metadata"] = build_manifest_entry(
        "explainability_metadata", "Explainability method availability & results summary (Stage 6, referenced in-place).",
        explainability_src, "stage06_explainability", False, logger,
    )

    generated_utc = datetime.now(timezone.utc).isoformat()
    total_size = sum(e.size_bytes or 0 for e in entries.values())
    missing_critical = [aid for aid, e in entries.items() if e.is_critical and (not e.exists or e.errors)]

    for aid in missing_critical:
        logger.error("MANIFEST critical artifact unavailable: %s", aid)

    return DeploymentManifest(
        package_version=PACKAGE_VERSION, generated_utc=generated_utc, entries=entries,
        total_artifacts=len(entries), total_size_bytes=total_size, missing_critical=missing_critical,
    )


# ======================================================================
# DEPLOYMENT METADATA GENERATION
# ======================================================================

def resolve_supported_runtimes(export_registry: Any) -> List[str]:
    runtimes = ["pytorch"]  # the reconstructed model itself always counts
    if export_registry.torchscript.success:
        runtimes.append("torchscript")
    if export_registry.onnx.success and export_registry.onnx.checker_passed:
        runtimes.append("onnxruntime")
    return runtimes


def resolve_supported_devices() -> List[str]:
    devices = ["cpu"]
    if torch.cuda.is_available():
        devices.append("cuda")
    return devices


def build_deployment_metadata(
    config_registry: Any, model_registry: Any, metadata_registry: Any,
    threshold_registry: Any, export_registry: Any, inference_engine: Any,
    explainability_summary: Optional[Dict[str, Any]], logger: logging.Logger,
) -> Dict[str, Any]:
    preprocessing = inference_engine.preprocessing_config.to_dict()

    if explainability_summary is not None:
        methods_available = explainability_summary.get("methods_available", {})
    else:
        methods_available = {name: None for name in EXPLAINABILITY_METHODS_CONTRACT}
        logger.warning("METADATA explainability method availability unknown (Stage 6 summary unreadable); recording as null.")

    metadata: Dict[str, Any] = {
        "package_version": PACKAGE_VERSION,
        "build_timestamp_utc": datetime.now(timezone.utc).isoformat(),
        "model_version": inference_engine.model_version,
        "model_fingerprint_sha256": model_registry.checkpoint_sha256,
        "backbone": model_registry.backbone,
        "num_classes": model_registry.num_classes,
        "class_names": inference_engine.class_names,
        "preprocessing_configuration": preprocessing,
        "normalization": {"mean": preprocessing["mean"], "std": preprocessing["std"]},
        "thresholds": threshold_registry.thresholds,
        "export_versions": {
            "onnx_opset_version": export_registry.onnx.opset_version,
            "torchscript_export_method": export_registry.torchscript.export_method,
        },
        "supported_runtimes": resolve_supported_runtimes(export_registry),
        "supported_devices": resolve_supported_devices(),
        "supported_batch_sizes": {
            "dynamic_batching": export_registry.model_signature.batch_support == "dynamic",
            "documented_validated_sizes": DOCUMENTED_VALIDATED_BATCH_SIZES,
        },
        "explainability_methods_available": methods_available,
        "device_configured": config_registry.device,
        "dtype": config_registry.dtype,
    }
    logger.info(
        "METADATA deployment_metadata built: model_version=%s backbone=%s classes=%d runtimes=%s",
        metadata["model_version"], metadata["backbone"], metadata["num_classes"], metadata["supported_runtimes"],
    )
    return metadata


# ======================================================================
# COMPATIBILITY REPORT
# ======================================================================

def onnxruntime_providers() -> Optional[List[str]]:
    try:
        import onnxruntime as ort
        return ort.get_available_providers()
    except ImportError:
        return None


def build_compatibility_report(
    config_registry: Any, export_registry: Any, model_registry: Any, logger: logging.Logger,
) -> Dict[str, Any]:
    checks: Dict[str, bool] = {}
    warnings: List[str] = []

    python_version = tuple(int(p) for p in sys.version.split()[0].split(".")[:2])
    python_compatible = SUPPORTED_PYTHON_MIN <= python_version < SUPPORTED_PYTHON_MAX_EXCLUSIVE
    checks["python_version_supported"] = python_compatible
    if not python_compatible:
        warnings.append(
            f"Python {sys.version.split()[0]} is outside the documented supported range "
            f"{SUPPORTED_PYTHON_MIN}-{SUPPORTED_PYTHON_MAX_EXCLUSIVE} (exclusive upper bound)."
        )

    torch_major = int(torch.__version__.split(".")[0])
    torch_compatible = torch_major >= SUPPORTED_TORCH_MAJOR_MIN
    checks["torch_version_supported"] = torch_compatible
    if not torch_compatible:
        warnings.append(f"Torch {torch.__version__} is below the documented minimum major version {SUPPORTED_TORCH_MAJOR_MIN}.")

    checks["torchscript_export_valid"] = export_registry.torchscript.success
    checks["onnx_export_valid"] = export_registry.onnx.success and export_registry.onnx.checker_passed

    cuda_available = torch.cuda.is_available()
    checks["cuda_runtime_consistent"] = (not cuda_available) or (torch.version.cuda is not None)
    checks["cpu_supported"] = True  # CPU deployment is always a valid baseline

    providers = onnxruntime_providers()
    checks["onnxruntime_importable"] = providers is not None
    if providers is None:
        warnings.append("onnxruntime is not importable in this environment; ONNX serving cannot be validated here.")

    if not cuda_available:
        warnings.append("No GPU detected in this environment; CUDA row of the support matrix is informational only.")

    support_matrix = {
        "pytorch": {"cpu": True, "cuda": cuda_available},
        "torchscript": {
            "cpu": export_registry.torchscript.success,
            "cuda": export_registry.torchscript.success and cuda_available,
        },
        "onnxruntime": {
            "cpu": export_registry.onnx.success and (providers is None or "CPUExecutionProvider" in providers),
            "cuda": export_registry.onnx.success and providers is not None and "CUDAExecutionProvider" in providers,
        },
    }

    for w in warnings:
        logger.warning("COMPATIBILITY WARNING: %s", w)

    return {
        "generated_utc": datetime.now(timezone.utc).isoformat(),
        "python": {
            "version": sys.version.split()[0],
            "supported_range": f">={'.'.join(map(str, SUPPORTED_PYTHON_MIN))},<{'.'.join(map(str, SUPPORTED_PYTHON_MAX_EXCLUSIVE))}",
            "compatible": python_compatible,
        },
        "torch": {"version": torch.__version__, "minimum_supported_major": SUPPORTED_TORCH_MAJOR_MIN, "compatible": torch_compatible},
        "torchvision": {"version": torchvision.__version__},
        "torchscript": {
            "available": hasattr(torch, "jit"),
            "export_method": export_registry.torchscript.export_method,
            "export_success": export_registry.torchscript.success,
            "compatible": checks["torchscript_export_valid"],
        },
        "onnx": {
            "opset_version": export_registry.onnx.opset_version,
            "checker_passed": export_registry.onnx.checker_passed,
            "onnxruntime_available": providers is not None,
            "onnxruntime_providers": providers,
            "compatible": checks["onnx_export_valid"],
        },
        "cuda": {
            "available": cuda_available,
            "version": torch.version.cuda,
            "device_configured": config_registry.device,
            "compatible": checks["cuda_runtime_consistent"],
        },
        "cpu": {"supported": True, "compatible": True},
        "support_matrix": support_matrix,
        "checks": checks,
        "warnings": warnings,
        "passed": all(checks[k] for k in ("torchscript_export_valid", "onnx_export_valid", "cuda_runtime_consistent", "cpu_supported")),
    }


# ======================================================================
# REPRODUCIBILITY MANIFEST
# ======================================================================

def build_reproducibility_manifest(
    config_registry: Any, model_registry: Any, export_registry: Any, logger: logging.Logger,
) -> Dict[str, Any]:
    stage1_cfg = config_registry.to_dict()["stage1_deployment_config"]
    runtime_cfg = stage1_cfg.get("runtime", {})

    manifest = {
        "generated_utc": datetime.now(timezone.utc).isoformat(),
        "random_seed": runtime_cfg.get("seed"),
        "deterministic_mode": runtime_cfg.get("deterministic"),
        "framework_versions": {
            "python": sys.version.split()[0],
            "torch": torch.__version__,
            "torchvision": torchvision.__version__,
        },
        "cuda_version": torch.version.cuda,
        "cudnn_version": torch.backends.cudnn.version() if torch.cuda.is_available() else None,
        "onnx_opset_version": export_registry.onnx.opset_version,
        "model_checkpoint_sha256": model_registry.checkpoint_sha256,
        "model_backbone": model_registry.backbone,
        "torchscript_graph_hash": export_registry.torchscript.graph_hash,
        "git_commit": get_git_commit(logger),
        "package_version": PACKAGE_VERSION,
    }
    logger.info(
        "REPRODUCIBILITY manifest built: seed=%s torch=%s cuda=%s opset=%d",
        manifest["random_seed"], manifest["framework_versions"]["torch"],
        manifest["cuda_version"], manifest["onnx_opset_version"],
    )
    return manifest


# ======================================================================
# INTEGRITY VALIDATION
# ======================================================================

def run_integrity_validation(
    manifest: DeploymentManifest, artifact_registry: Any, model_registry: Any,
    metadata_registry: Any, threshold_registry: Any, export_registry: Any,
    inference_engine: Any, compatibility: Dict[str, Any], logger: logging.Logger,
) -> Dict[str, Any]:
    checks: Dict[str, bool] = {}
    warnings: List[str] = []
    fatal_errors: List[str] = []

    # 1. All required (critical) artifacts exist in the manifest.
    checks["all_critical_artifacts_exist"] = len(manifest.missing_critical) == 0
    for aid in manifest.missing_critical:
        fatal_errors.append(f"Critical deployment artifact unavailable: {aid}")

    # 2. Checksum drift: has any upstream artifact changed since Stage 2
    #    registered it? (recomputed now vs. the sha256 Stage 2 recorded)
    checksum_mismatches: List[str] = []
    for artifact_id, record in artifact_registry.records.items():
        if not (record.is_critical and record.exists and record.sha256):
            continue
        path = Path(record.path)
        if not path.exists():
            checksum_mismatches.append(f"{artifact_id}: file no longer exists at {path}")
            continue
        current_hash = sha256_of_file(path)
        if current_hash != record.sha256:
            checksum_mismatches.append(f"{artifact_id}: sha256 drift detected (recorded={record.sha256[:12]}..., now={current_hash[:12]}...)")
    checks["checksums_match"] = len(checksum_mismatches) == 0
    for m in checksum_mismatches:
        fatal_errors.append(f"Checksum drift: {m}")

    # 3. Metadata consistency -- required upstream metadata dictionaries
    #    are non-empty.
    metadata_dict = metadata_registry.to_dict()
    checks["disease_metadata_present"] = bool(metadata_dict.get("disease_metadata"))
    checks["training_metadata_present"] = bool(metadata_dict.get("training_metadata"))
    checks["evaluation_metadata_present"] = bool(metadata_dict.get("evaluation_metadata"))
    if not checks["evaluation_metadata_present"]:
        warnings.append("evaluation_metadata is empty; deployment_metadata will lack evaluation provenance.")
    if not (checks["disease_metadata_present"] and checks["training_metadata_present"]):
        fatal_errors.append("Core metadata (disease/training) is missing -- cannot certify a deployment package.")

    # 4. Class-count consistency across every registry that carries a class count.
    class_counts = {
        "model_registry.num_classes": model_registry.num_classes,
        "inference_engine.class_names": len(inference_engine.class_names),
    }
    if threshold_registry.class_count > 0:
        class_counts["threshold_registry.class_count"] = threshold_registry.class_count
    unique_counts = set(class_counts.values())
    checks["class_count_consistent"] = len(unique_counts) == 1
    if not checks["class_count_consistent"]:
        fatal_errors.append(f"Class-count mismatch across registries: {class_counts}")

    # 5. Threshold-count consistency (re-verified at packaging time).
    checks["threshold_count_consistent"] = (
        threshold_registry.class_count == 0 or threshold_registry.class_count == model_registry.num_classes
    )
    checks["threshold_registry_valid"] = len(threshold_registry.validation_errors) == 0
    if not checks["threshold_count_consistent"]:
        fatal_errors.append(
            f"Threshold class_count ({threshold_registry.class_count}) != model num_classes ({model_registry.num_classes})."
        )
    if not checks["threshold_registry_valid"]:
        warnings.append(f"THRESHOLD_REGISTRY carries {len(threshold_registry.validation_errors)} validation warning(s) from Stage 2.")

    # 6. Export consistency -- export lineage must trace back to the same
    #    checkpoint hash the model was reconstructed from.
    lineage_consistent = (
        export_registry.torchscript.model_hash == model_registry.checkpoint_sha256
        and export_registry.onnx.model_hash == model_registry.checkpoint_sha256
    )
    checks["export_lineage_consistent"] = lineage_consistent
    if not lineage_consistent:
        fatal_errors.append("Exported TorchScript/ONNX artifacts do not trace back to the reconstructed model's checkpoint hash.")
    checks["export_numerical_validation_passed"] = export_registry.numerical_validation.passed
    if not checks["export_numerical_validation_passed"]:
        fatal_errors.append("Stage 3 numerical validation (PyTorch vs TorchScript/ONNX) did not pass.")

    # 7. Runtime compatibility -- devices agree across registries.
    checks["runtime_device_consistent"] = str(inference_engine.device) == model_registry.device
    if not checks["runtime_device_consistent"]:
        warnings.append(
            f"InferenceEngine device ({inference_engine.device}) differs from MODEL_REGISTRY device ({model_registry.device})."
        )
    checks["compatibility_report_passed"] = bool(compatibility.get("passed"))
    if not checks["compatibility_report_passed"]:
        fatal_errors.append("Compatibility report did not pass (see compatibility_matrix.json).")

    for w in warnings:
        logger.warning("INTEGRITY WARNING: %s", w)
    for e in fatal_errors:
        logger.error("INTEGRITY FAILURE: %s", e)

    return {
        "generated_utc": datetime.now(timezone.utc).isoformat(),
        "checks": checks,
        "checksum_mismatches": checksum_mismatches,
        "class_counts": class_counts,
        "warnings": warnings,
        "fatal_errors": fatal_errors,
        "passed": len(fatal_errors) == 0,
    }


# ======================================================================
# ENGINEERING VALIDATION (standard Sprint05 stage gate)
# ======================================================================

def run_stage07_engineering_validation(
    manifest: DeploymentManifest, integrity: Dict[str, Any], compatibility: Dict[str, Any],
    logger: logging.Logger,
) -> Dict[str, Any]:
    checks: Dict[str, bool] = {}
    warnings: List[str] = list(integrity["warnings"]) + list(compatibility["warnings"])
    fatal_errors: List[str] = []

    # Fail-fast ONLY for missing critical deployment artifacts, per Stage 7's
    # explicit constraint -- everything else surfaces as a warning here and
    # is scored (not gated) in the readiness report.
    checks["no_missing_critical_artifacts"] = len(manifest.missing_critical) == 0
    if manifest.missing_critical:
        fatal_errors.append(f"Missing critical deployment artifacts: {manifest.missing_critical}")

    checks["integrity_validation_passed"] = integrity["passed"]
    if not integrity["passed"]:
        # Only escalate to fail-fast the subset of integrity failures that
        # are themselves about missing/corrupted critical artifacts; other
        # integrity failures (e.g. a soft metadata gap) are recorded as
        # warnings above rather than blocking packaging outright.
        for fe in integrity["fatal_errors"]:
            if "Critical deployment artifact unavailable" in fe or "Checksum drift" in fe:
                fatal_errors.append(fe)
            else:
                warnings.append(fe)

    checks["manifest_generated"] = manifest.total_artifacts > 0
    checks["compatibility_report_generated"] = bool(compatibility)

    for e in fatal_errors:
        logger.error("ENGINEERING VALIDATION FAILURE: %s", e)

    return {"checks": checks, "warnings": warnings, "fatal_errors": fatal_errors, "passed": len(fatal_errors) == 0}


# ======================================================================
# DEPLOYMENT READINESS SCORE
# ======================================================================

def compute_deployment_readiness(
    manifest: DeploymentManifest, deployment_metadata: Dict[str, Any],
    compatibility: Dict[str, Any], integrity: Dict[str, Any],
    model_registry: Any, export_registry: Any, inference_engine: Any,
    explainability_summary: Optional[Dict[str, Any]], logger: logging.Logger,
) -> Dict[str, Any]:
    checks: Dict[str, bool] = {}

    checks["model_reconstructed"] = (
        model_registry.validation.architecture_instantiated
        and model_registry.validation.checkpoint_loaded
        and model_registry.validation.state_dict_strict_match
    )
    checks["torchscript_export_validated"] = export_registry.torchscript.success
    checks["onnx_export_validated"] = export_registry.onnx.success and export_registry.onnx.checker_passed
    checks["numerical_validation_passed"] = export_registry.numerical_validation.passed
    checks["runtime_validated"] = bool(inference_engine.class_names)
    checks["explainability_available"] = (
        explainability_summary is not None
        and any(bool(v) for v in explainability_summary.get("methods_available", {}).values())
    )
    checks["metadata_complete"] = bool(
        deployment_metadata.get("class_names") and deployment_metadata.get("thresholds")
    )
    checks["manifests_generated"] = manifest.total_artifacts > 0 and len(manifest.missing_critical) == 0
    checks["checksums_verified"] = len(integrity.get("checksum_mismatches", [])) == 0
    checks["compatibility_passed"] = bool(compatibility.get("passed"))
    checks["integrity_passed"] = bool(integrity.get("passed"))

    passed_count = sum(1 for v in checks.values() if v)
    total_count = len(checks)
    score = round(100.0 * passed_count / total_count, 2) if total_count else 0.0

    blocking_issues = [name for name, ok in checks.items() if not ok and name in (
        "model_reconstructed", "torchscript_export_validated", "onnx_export_validated",
        "numerical_validation_passed", "manifests_generated", "checksums_verified", "integrity_passed",
    )]
    warnings = [name for name, ok in checks.items() if not ok and name not in blocking_issues]

    if blocking_issues:
        readiness_level = "NOT_READY"
    elif warnings:
        readiness_level = "READY_WITH_WARNINGS"
    else:
        readiness_level = "READY"

    for issue in blocking_issues:
        logger.error("READINESS BLOCKING ISSUE: %s", issue)
    for w in warnings:
        logger.warning("READINESS WARNING: %s", w)

    logger.info(
        "READINESS score=%.2f level=%s passed=%d/%d blocking=%d",
        score, readiness_level, passed_count, total_count, len(blocking_issues),
    )

    return {
        "generated_utc": datetime.now(timezone.utc).isoformat(),
        "overall_score": score,
        "readiness_level": readiness_level,
        "checks": checks,
        "passed_checks": passed_count,
        "total_checks": total_count,
        "warnings": warnings,
        "blocking_issues": blocking_issues,
    }


# ======================================================================
# MAIN STAGE ENTRY POINT
# ======================================================================

def run_stage07(
    config_registry: Any,
    artifact_registry: Any,
    model_registry: Any,
    metadata_registry: Any,
    threshold_registry: Any,
    export_registry: Any,
    inference_engine: Any,
    reconstructed_model: Any,
) -> Dict[str, Any]:
    start_time = time.time()

    stage_root = OUTPUT_ROOT / STAGE7_DIR_NAME
    dirs = make_stage_dirs(stage_root)
    logger = build_logger(dirs["logs"])

    logger.info("START Sprint05-Stage07 Deployment Packaging & Production Readiness")
    log_resources(logger, "START")

    # ---- MANIFEST GENERATION ------------------------------------------
    logger.info("MANIFEST generation starting")
    manifest = build_deployment_manifest(
        dirs=dirs, config_registry=config_registry, artifact_registry=artifact_registry,
        model_registry=model_registry, metadata_registry=metadata_registry,
        threshold_registry=threshold_registry, export_registry=export_registry,
        inference_engine=inference_engine, logger=logger,
    )
    logger.info(
        "MANIFEST generation complete: %d artifacts, %d bytes total, %d critical-missing",
        manifest.total_artifacts, manifest.total_size_bytes, len(manifest.missing_critical),
    )

    # ---- METADATA GENERATION ------------------------------------------
    logger.info("METADATA generation starting")
    explainability_summary = load_explainability_summary(logger)
    deployment_metadata = build_deployment_metadata(
        config_registry=config_registry, model_registry=model_registry, metadata_registry=metadata_registry,
        threshold_registry=threshold_registry, export_registry=export_registry, inference_engine=inference_engine,
        explainability_summary=explainability_summary, logger=logger,
    )
    logger.info("METADATA generation complete")

    # ---- COMPATIBILITY VALIDATION --------------------------------------
    logger.info("COMPATIBILITY validation starting")
    compatibility = build_compatibility_report(
        config_registry=config_registry, export_registry=export_registry, model_registry=model_registry, logger=logger,
    )
    logger.info(
        "COMPATIBILITY validation complete: %d/%d checks passed",
        sum(compatibility["checks"].values()), len(compatibility["checks"]),
    )

    # ---- REPRODUCIBILITY MANIFEST --------------------------------------
    reproducibility = build_reproducibility_manifest(
        config_registry=config_registry, model_registry=model_registry, export_registry=export_registry, logger=logger,
    )

    # ---- INTEGRITY VALIDATION -------------------------------------------
    logger.info("INTEGRITY validation starting")
    integrity = run_integrity_validation(
        manifest=manifest, artifact_registry=artifact_registry, model_registry=model_registry,
        metadata_registry=metadata_registry, threshold_registry=threshold_registry,
        export_registry=export_registry, inference_engine=inference_engine,
        compatibility=compatibility, logger=logger,
    )
    logger.info(
        "INTEGRITY validation complete: %d/%d checks passed, passed=%s",
        sum(integrity["checks"].values()), len(integrity["checks"]), integrity["passed"],
    )

    # ---- ENGINEERING VALIDATION (fail-fast gate) ------------------------
    engineering_validation = run_stage07_engineering_validation(
        manifest=manifest, integrity=integrity, compatibility=compatibility, logger=logger,
    )

    if not engineering_validation["passed"]:
        log_resources(logger, "FINISH-FAILED")
        elapsed = time.time() - start_time
        logger.error("FINISH Sprint05-Stage07 status=FAILED elapsed=%.2fs", elapsed)
        raise RuntimeError(
            "Sprint05 Stage07 engineering validation FAILED (fail-fast):\n  - "
            + "\n  - ".join(engineering_validation["fatal_errors"])
        )

    # ---- READINESS SCORING ----------------------------------------------
    logger.info("READINESS scoring starting")
    readiness = compute_deployment_readiness(
        manifest=manifest, deployment_metadata=deployment_metadata, compatibility=compatibility,
        integrity=integrity, model_registry=model_registry, export_registry=export_registry,
        inference_engine=inference_engine, explainability_summary=explainability_summary, logger=logger,
    )
    logger.info("READINESS scoring complete: score=%.2f level=%s", readiness["overall_score"], readiness["readiness_level"])

    log_resources(logger, "FINISH")
    elapsed = time.time() - start_time

    # ---- ARTIFACT WRITING -------------------------------------------------
    logger.info("ARTIFACT WRITING starting")
    save_json(dirs["manifests"] / "deployment_manifest.json", manifest.to_dict())
    save_json(dirs["metadata"] / "deployment_metadata.json", deployment_metadata)
    save_json(dirs["compatibility"] / "compatibility_matrix.json", compatibility)
    save_json(dirs["integrity"] / "integrity_report.json", integrity)
    save_json(dirs["metadata"] / "reproducibility_manifest.json", reproducibility)
    save_json(dirs["reports"] / "deployment_readiness.json", readiness)
    save_json(dirs["integrity"] / "engineering_validation.json", engineering_validation)

    package_summary = {
        "package_version": PACKAGE_VERSION,
        "build_timestamp_utc": deployment_metadata["build_timestamp_utc"],
        "model_version": deployment_metadata["model_version"],
        "backbone": deployment_metadata["backbone"],
        "num_classes": deployment_metadata["num_classes"],
        "supported_runtimes": deployment_metadata["supported_runtimes"],
        "supported_devices": deployment_metadata["supported_devices"],
        "artifacts_packaged": manifest.total_artifacts,
        "total_package_size_bytes": manifest.total_size_bytes,
        "readiness_level": readiness["readiness_level"],
        "readiness_score": readiness["overall_score"],
        "package_directory": str(dirs["package"]),
        "output_directory": str(dirs["stage_root"]),
    }
    save_json(dirs["reports"] / "package_summary.json", package_summary)

    stage07_summary = {
        "stage": STAGE7_DIR_NAME,
        "status": "OK",
        "elapsed_seconds": round(elapsed, 3),
        "artifacts_packaged": manifest.total_artifacts,
        "checksums_verified": len(integrity["checksum_mismatches"]) == 0,
        "compatibility_checks_passed": sum(compatibility["checks"].values()),
        "compatibility_checks_total": len(compatibility["checks"]),
        "integrity_checks_passed": sum(integrity["checks"].values()),
        "integrity_checks_total": len(integrity["checks"]),
        "deployment_readiness_score": readiness["overall_score"],
        "readiness_level": readiness["readiness_level"],
        "engineering_checks_passed": sum(engineering_validation["checks"].values()),
        "engineering_checks_total": len(engineering_validation["checks"]),
        "warnings_count": len(engineering_validation["warnings"]),
        "output_directory": str(dirs["stage_root"]),
    }
    save_json(dirs["stage_root"] / "stage07_summary.json", stage07_summary)

    logger.info("FINISH Sprint05-Stage07 status=OK elapsed=%.2fs", elapsed)

    # ---- CONSOLE REPORT ----------------------------------------------------
    print("=" * 70)
    print("SPRINT05 — STAGE07")
    print("DEPLOYMENT PACKAGE & PRODUCTION READINESS")
    print("=" * 70)
    print(f"Artifacts Packaged   : {manifest.total_artifacts}")
    print(f"Checksums Verified   : {'YES' if len(integrity['checksum_mismatches']) == 0 else 'NO (' + str(len(integrity['checksum_mismatches'])) + ' mismatch(es))'}")
    print(f"Compatibility Checks : {sum(compatibility['checks'].values())}/{len(compatibility['checks'])}")
    print(f"Integrity Checks     : {sum(integrity['checks'].values())}/{len(integrity['checks'])}")
    print(f"Deployment Score     : {readiness['overall_score']:.2f} ({readiness['readiness_level']})")
    print(f"Engineering Checks   : {sum(engineering_validation['checks'].values())}/{len(engineering_validation['checks'])}")
    print(f"Warnings             : {len(engineering_validation['warnings'])}")
    print(f"Output Directory     : {dirs['stage_root']}")
    print("Stage 7 : OK")

    return {
        "MANIFEST": manifest,
        "DEPLOYMENT_METADATA": deployment_metadata,
        "COMPATIBILITY_REPORT": compatibility,
        "REPRODUCIBILITY_MANIFEST": reproducibility,
        "INTEGRITY_REPORT": integrity,
        "ENGINEERING_VALIDATION": engineering_validation,
        "READINESS": readiness,
        "PACKAGE_SUMMARY": package_summary,
        "summary": stage07_summary,
    }


if __name__ == "__main__":
    _stage07_result = run_stage07(
        config_registry=CONFIG_REGISTRY,
        artifact_registry=ARTIFACT_REGISTRY,
        model_registry=MODEL_REGISTRY,
        metadata_registry=METADATA_REGISTRY,
        threshold_registry=THRESHOLD_REGISTRY,
        export_registry=EXPORT_REGISTRY,
        inference_engine=INFERENCE_ENGINE,
        reconstructed_model=RECONSTRUCTED_MODEL,
    )
    DEPLOYMENT_MANIFEST = _stage07_result["MANIFEST"]
    DEPLOYMENT_METADATA = _stage07_result["DEPLOYMENT_METADATA"]
    COMPATIBILITY_REPORT = _stage07_result["COMPATIBILITY_REPORT"]
    REPRODUCIBILITY_MANIFEST = _stage07_result["REPRODUCIBILITY_MANIFEST"]
    INTEGRITY_REPORT = _stage07_result["INTEGRITY_REPORT"]
    DEPLOYMENT_READINESS = _stage07_result["READINESS"]

2026-07-08 10:34:11 | INFO     | START Sprint05-Stage07 Deployment Packaging & Production Readiness
2026-07-08 10:34:11 | INFO     | RESOURCES [START] cpu=2.5% ram=8.3%(2.1GB/31.4GB)
2026-07-08 10:34:11 | INFO     | MANIFEST generation starting
2026-07-08 10:34:11 | INFO     | MANIFEST[torchscript_model] path=/kaggle/working/sprint05_deployment/stage07_deployment_package/package/model.ts size=28617249 sha256=e6da75f9224ca58a15c18b0b0b1ee65f9360115ff756986fa95cd62eda2c059a critical=True
2026-07-08 10:34:11 | INFO     | MANIFEST[onnx_model] path=/kaggle/working/sprint05_deployment/stage07_deployment_package/package/model.onnx size=1062588 sha256=87392e11168de04aa3962a83cedaa4ba813f58c41730c156370dabde5888ec21 critical=True
2026-07-08 10:34:12 | INFO     | MANIFEST[threshold_registry] path=/kaggle/working/sprint05_deployment/stage07_deployment_package/package/optimal_thresholds.json size=5109 sha256=2651e5d3b0dba2a53a836b244552e1e3c2ea30a4b817475a9537e1dea618807c critical=True
2026-07-08 

In [25]:
"""
VisionServeAI - Sprint 05
Stage 8: Deployment Documentation & Release Engineering
======================================================================
Single-responsibility stage: consumes the frozen outputs of Stages 01-07
(registries, reports, and manifests already resident in memory) and
produces the complete production release: executive summary, full
deployment report, HuggingFace-style model card, deployment guide, API
reference, release notes, deployment validation report, release
manifest, and the final Sprint 05 report.

No training. No evaluation. No inference. No GradCAM. No TorchScript /
ONNX export. No benchmarking. No dataset loading. No artifact
discovery. This stage only reads already-computed objects and already-
written JSON summaries from Stages 01-07, transforms them into
documentation, and writes new files under stage08_release/.
"""

import sys
import time
import json
import hashlib
import logging
import importlib.metadata as _ilmd
from pathlib import Path
from datetime import datetime, timezone
from dataclasses import dataclass, field, asdict, is_dataclass
from typing import Any, Dict, List, Optional

import torch
import torchvision

# ======================================================================
# CONSTANTS
# ======================================================================
OUTPUT_ROOT = Path("/kaggle/working/sprint05_deployment")
STAGE8_DIR_NAME = "stage08_release"

GENERATOR_NAME = "VisionServeAI-Sprint05-Stage08"
STAGE_NAME = "Sprint05-Stage08-DeploymentDocumentation"
PROJECT_NAME = "VisionServeAI"
MODEL_DISPLAY_NAME = "VisionServeAI Chest X-Ray Multi-Label Classifier"

STAGE1_DIR = OUTPUT_ROOT / "stage01_environment"
STAGE2_DIR = OUTPUT_ROOT / "stage02_registry"
STAGE3_DIR = OUTPUT_ROOT / "stage03_export"
STAGE4_DIR = OUTPUT_ROOT / "stage04_runtime"
STAGE5_DIR = OUTPUT_ROOT / "stage05_validation"
STAGE6_DIR = OUTPUT_ROOT / "stage06_explainability"
STAGE7_DIR = OUTPUT_ROOT / "stage07_deployment_package"

STAGE_SUMMARY_FILES: Dict[str, Path] = {
    "stage01": STAGE1_DIR / "stage01_summary.json",
    "stage02": STAGE2_DIR / "stage02_summary.json",
    "stage03": STAGE3_DIR / "stage03_summary.json",
    "stage04": STAGE4_DIR / "stage04_summary.json",
    "stage05": STAGE5_DIR / "stage05_summary.json",
    "stage06": STAGE6_DIR / "stage06_summary.json",
    "stage07": STAGE7_DIR / "stage07_summary.json",
}

STAGE_PIPELINE: List[Dict[str, Any]] = [
    {"stage": "Stage 01", "name": "Deployment Environment",
     "outputs": ["CONFIG_REGISTRY", "DeploymentConfig", "environment.json", "artifact discovery", "engineering validation"]},
    {"stage": "Stage 02", "name": "Artifact Registry & Model Reconstruction",
     "outputs": ["ARTIFACT_REGISTRY", "MODEL_REGISTRY", "METADATA_REGISTRY", "THRESHOLD_REGISTRY",
                 "RECONSTRUCTED_MODEL", "Checkpoint reconstruction", "Strict validation"]},
    {"stage": "Stage 03", "name": "TorchScript + ONNX Export",
     "outputs": ["TorchScript", "ONNX", "Numerical validation", "Dynamic batching", "Export registry", "Export metadata"]},
    {"stage": "Stage 04", "name": "Production Inference Runtime",
     "outputs": ["InferenceEngine", "Preprocessing pipeline", "Threshold loading", "Input validation",
                 "Error handling", "Batch inference", "Runtime metadata"]},
    {"stage": "Stage 05", "name": "Performance & Robustness Validation",
     "outputs": ["Performance benchmark", "Latency", "Throughput", "Stress testing", "Memory validation",
                 "Robustness testing", "Engineering validation"]},
    {"stage": "Stage 06", "name": "Explainability Runtime",
     "outputs": ["GradCAM", "GradCAM++", "ScoreCAM", "EigenCAM", "Guided Backprop", "Integrated Gradients",
                 "Occlusion", "Explainability metadata", "Images", "JSON summaries"]},
    {"stage": "Stage 07", "name": "Deployment Packaging & Production Readiness",
     "outputs": ["Deployment package", "Deployment manifest", "Compatibility report", "Integrity report",
                 "Reproducibility manifest", "Deployment metadata", "Readiness score", "Checksums",
                 "SHA256", "Package summary"]},
    {"stage": "Stage 08", "name": "Deployment Documentation, Release Engineering & Final Deployment Report",
     "outputs": ["Executive summary", "Deployment report", "Model card", "Deployment guide", "API reference",
                 "Release notes", "Deployment validation report", "Release manifest", "Final sprint report"]},
]

KNOWN_INFERENCE_ERROR_TYPES = [
    "missing_file", "unsupported_extension", "corrupted_image",
    "zero_byte_image", "wrong_channel_count", "invalid_tensor_dimensions",
]

FUTURE_ROADMAP = [
    "Sprint 06: containerized serving (FastAPI + Triton/TorchServe) with autoscaling.",
    "Sprint 06: continuous model-quality monitoring in production (drift, calibration decay).",
    "Sprint 06: multi-view (frontal + lateral) chest X-ray fusion.",
    "Sprint 07: clinical validation study under IRB-approved protocol prior to any clinical use.",
    "Sprint 07: automated CI/CD pipeline for checkpoint promotion and canary rollout.",
]


# ======================================================================
# LOGGING / RESOURCE HELPERS (identical pattern to Stages 01-07)
# ======================================================================
def build_logger(log_dir: Path) -> logging.Logger:
    log_dir.mkdir(parents=True, exist_ok=True)
    logger = logging.getLogger("sprint05.stage08")
    logger.setLevel(logging.INFO)
    logger.handlers.clear()
    logger.propagate = False

    fmt = logging.Formatter(
        fmt="%(asctime)s | %(levelname)-8s | %(message)s",
        datefmt="%Y-%m-%d %H:%M:%S",
    )

    file_handler = logging.FileHandler(log_dir / "stage08_release_engineering.log", mode="w")
    file_handler.setFormatter(fmt)
    logger.addHandler(file_handler)

    stream_handler = logging.StreamHandler(sys.stdout)
    stream_handler.setFormatter(fmt)
    logger.addHandler(stream_handler)

    return logger


def get_resource_usage() -> Dict[str, Any]:
    import psutil
    vm = psutil.virtual_memory()
    usage: Dict[str, Any] = {
        "cpu_percent": psutil.cpu_percent(interval=0.2),
        "ram_percent": vm.percent,
        "ram_used_gb": round(vm.used / (1024 ** 3), 2),
        "ram_total_gb": round(vm.total / (1024 ** 3), 2),
    }
    if torch.cuda.is_available():
        usage["gpu_memory_allocated_gb"] = round(torch.cuda.memory_allocated() / (1024 ** 3), 3)
        usage["gpu_memory_reserved_gb"] = round(torch.cuda.memory_reserved() / (1024 ** 3), 3)
    return usage


def log_resources(logger: logging.Logger, tag: str) -> Dict[str, Any]:
    usage = get_resource_usage()
    logger.info(
        "RESOURCES [%s] cpu=%.1f%% ram=%.1f%%(%.1fGB/%.1fGB)",
        tag, usage["cpu_percent"], usage["ram_percent"], usage["ram_used_gb"], usage["ram_total_gb"],
    )
    return usage


def _utc_now() -> str:
    return datetime.now(timezone.utc).isoformat()


def sha256_of_file(path: Path, chunk_size: int = 1 << 20) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            h.update(chunk)
    return h.hexdigest()


def _format_bytes(num_bytes: Optional[int]) -> str:
    if not num_bytes:
        return "0.00 MB"
    return f"{num_bytes / (1024 ** 2):.2f} MB"


def make_stage_dirs(stage_root: Path) -> Dict[str, Path]:
    dirs = {
        "stage_root": stage_root,
        "logs": stage_root / "logs",
        "reports": stage_root / "reports",
        "documentation": stage_root / "documentation",
        "model_card": stage_root / "model_card",
        "deployment_guides": stage_root / "deployment_guides",
        "api": stage_root / "api",
        "release": stage_root / "release",
        "validation": stage_root / "validation",
    }
    for d in dirs.values():
        d.mkdir(parents=True, exist_ok=True)
    return dirs


# ======================================================================
# JSON SAFETY
# ======================================================================
def _json_safe(obj: Any) -> Any:
    """Recursively coerce obj into pure JSON-serializable primitives.
    No NaN, no numpy/tensor types, no Path objects, no dataclass objects."""
    if obj is None or isinstance(obj, (str, bool, int)):
        return obj
    if isinstance(obj, float):
        if obj != obj or obj in (float("inf"), float("-inf")):
            return None
        return obj
    if isinstance(obj, Path):
        return str(obj)
    if is_dataclass(obj) and not isinstance(obj, type):
        return _json_safe(asdict(obj))
    if hasattr(obj, "to_dict") and callable(getattr(obj, "to_dict")):
        return _json_safe(obj.to_dict())
    if isinstance(obj, dict):
        return {str(k): _json_safe(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple, set, frozenset)):
        return [_json_safe(v) for v in obj]
    if hasattr(obj, "item") and callable(getattr(obj, "item")):
        try:
            return obj.item()
        except Exception:
            return str(obj)
    return str(obj)


# ======================================================================
# DATACLASSES
# ======================================================================
@dataclass
class ReleaseArtifactEntry:
    artifact_id: str
    category: str
    file_path: str
    size_bytes: int
    sha256: str
    generated_utc: str


@dataclass
class ModuleExecutionRecord:
    module_id: str
    module_name: str
    status: str
    files_generated: List[str] = field(default_factory=list)
    elapsed_seconds: float = 0.0
    errors: List[str] = field(default_factory=list)


# ======================================================================
# FILE-WRITING HELPERS (track every artifact Stage 08 produces)
# ======================================================================
def _register_file(
    path: Path, artifact_id: str, category: str,
    generated_files: List[ReleaseArtifactEntry], logger: logging.Logger,
) -> None:
    size = path.stat().st_size
    digest = sha256_of_file(path)
    generated_files.append(ReleaseArtifactEntry(
        artifact_id=artifact_id, category=category, file_path=str(path),
        size_bytes=size, sha256=digest, generated_utc=_utc_now(),
    ))
    logger.info("ARTIFACT WRITTEN id=%s path=%s size=%dB sha256=%s...", artifact_id, path, size, digest[:12])


def _write_json(
    path: Path, data: Any, artifact_id: str, category: str,
    generated_files: List[ReleaseArtifactEntry], logger: logging.Logger,
) -> None:
    payload = {
        "generator": GENERATOR_NAME,
        "stage": STAGE_NAME,
        "artifact_id": artifact_id,
        "version": "1.0.0",
        "generated_utc": _utc_now(),
        "data": _json_safe(data),
    }
    path.write_text(json.dumps(payload, indent=2, sort_keys=False))
    _register_file(path, artifact_id, category, generated_files, logger)


def _write_text(
    path: Path, text: str, artifact_id: str, category: str,
    generated_files: List[ReleaseArtifactEntry], logger: logging.Logger,
) -> None:
    path.write_text(text)
    _register_file(path, artifact_id, category, generated_files, logger)


def _load_optional_json(path: Path, logger: logging.Logger, label: str) -> Optional[Dict[str, Any]]:
    if not path.exists():
        logger.warning("OPTIONAL INPUT missing: %s (%s) -- proceeding without it.", label, path)
        return None
    try:
        return json.loads(path.read_text())
    except Exception as exc:
        logger.warning("OPTIONAL INPUT unreadable: %s (%s): %s -- proceeding without it.", label, path, exc)
        return None


def _lib_version(name: str) -> str:
    try:
        return _ilmd.version(name)
    except Exception:
        return "not_installed"


# ======================================================================
# DATA EXTRACTION HELPERS (read-only views over the frozen Stage 01-07
# registries -- no recomputation, no rediscovery)
# ======================================================================
def _class_names(threshold_registry: Any, model_registry: Any, inference_engine: Any) -> List[str]:
    if getattr(threshold_registry, "class_names", None):
        return list(threshold_registry.class_names)
    return list(getattr(inference_engine, "class_names", []))


def _stage_summaries(logger: logging.Logger) -> Dict[str, Optional[Dict[str, Any]]]:
    return {
        key: _load_optional_json(path, logger, key)
        for key, path in STAGE_SUMMARY_FILES.items()
    }


# ======================================================================
# MODULE 1 -- EXECUTIVE DEPLOYMENT SUMMARY
# ======================================================================
def build_module1_executive_summary(
    dirs: Dict[str, Path], logger: logging.Logger, generated_files: List[ReleaseArtifactEntry],
    config_registry: Any, model_registry: Any, export_registry: Any, threshold_registry: Any,
    deployment_readiness: Dict[str, Any], package_summary: Dict[str, Any],
    deployment_manifest: Any, reproducibility_manifest: Dict[str, Any],
) -> Dict[str, Any]:
    logger.info("MODULE START module=1 name=ExecutiveDeploymentSummary")
    t0 = time.time()

    onnx_export = export_registry.onnx
    torchscript_export = export_registry.torchscript
    fw = reproducibility_manifest.get("framework_versions", {})

    summary = {
        "model_name": MODEL_DISPLAY_NAME,
        "version": package_summary.get("package_version"),
        "backbone": model_registry.backbone,
        "checkpoint_sha256": model_registry.checkpoint_sha256,
        "num_classes": model_registry.num_classes,
        "class_names": _class_names(threshold_registry, model_registry, None) or None,
        "supported_runtimes": package_summary.get("supported_runtimes"),
        "torch_version": fw.get("torch", torch.__version__),
        "torchvision_version": fw.get("torchvision", torchvision.__version__),
        "cuda_version": reproducibility_manifest.get("cuda_version"),
        "onnx_opset_version": onnx_export.opset_version,
        "onnx_package_version": _lib_version("onnx"),
        "onnxruntime_package_version": _lib_version("onnxruntime"),
        "torchscript_export_method": torchscript_export.export_method,
        "torchscript_export_success": torchscript_export.success,
        "onnx_export_success": onnx_export.success and onnx_export.checker_passed,
        "device_configured": config_registry.device,
        "dtype": config_registry.dtype,
        "readiness_level": deployment_readiness.get("readiness_level"),
        "deployment_score": deployment_readiness.get("overall_score"),
        "package_version": package_summary.get("package_version"),
        "package_size_bytes": package_summary.get("total_package_size_bytes"),
        "package_size_human": _format_bytes(package_summary.get("total_package_size_bytes")),
        "artifacts_packaged": deployment_manifest.total_artifacts,
        "package_directory": package_summary.get("package_directory"),
        "stage07_output_directory": package_summary.get("output_directory"),
        "generation_time_utc": _utc_now(),
    }

    _write_json(
        dirs["reports"] / "deployment_summary.json", summary,
        "deployment_summary", "reports", generated_files, logger,
    )

    md = [
        f"# {MODEL_DISPLAY_NAME} -- Executive Deployment Summary",
        "",
        f"**Generated (UTC):** {summary['generation_time_utc']}  ",
        f"**Package Version:** {summary['package_version']}  ",
        f"**Readiness:** {summary['readiness_level']} ({summary['deployment_score']:.2f}/100)",
        "",
        "| Field | Value |",
        "|---|---|",
        f"| Model Name | {summary['model_name']} |",
        f"| Backbone | {summary['backbone']} |",
        f"| Checkpoint SHA256 | `{summary['checkpoint_sha256']}` |",
        f"| Classes | {summary['num_classes']} |",
        f"| Runtime(s) | {', '.join(summary['supported_runtimes'] or [])} |",
        f"| Torch Version | {summary['torch_version']} |",
        f"| Torchvision Version | {summary['torchvision_version']} |",
        f"| CUDA Version | {summary['cuda_version']} |",
        f"| ONNX Opset | {summary['onnx_opset_version']} |",
        f"| TorchScript Export | {summary['torchscript_export_method']} (success={summary['torchscript_export_success']}) |",
        f"| ONNX Export | success={summary['onnx_export_success']} |",
        f"| Device Configured | {summary['device_configured']} |",
        f"| Dtype | {summary['dtype']} |",
        f"| Package Version | {summary['package_version']} |",
        f"| Package Size | {summary['package_size_human']} |",
        f"| Artifacts Packaged | {summary['artifacts_packaged']} |",
        f"| Package Directory | `{summary['package_directory']}` |",
        f"| Stage 07 Output Directory | `{summary['stage07_output_directory']}` |",
        "",
    ]
    _write_text(
        dirs["reports"] / "deployment_summary.md", "\n".join(md),
        "deployment_summary_md", "reports", generated_files, logger,
    )

    elapsed = time.time() - t0
    logger.info("MODULE FINISH module=1 name=ExecutiveDeploymentSummary elapsed=%.3fs", elapsed)
    return summary


# ======================================================================
# MODULE 2 -- COMPLETE DEPLOYMENT REPORT
# ======================================================================
def build_module2_deployment_report(
    dirs: Dict[str, Path], logger: logging.Logger, generated_files: List[ReleaseArtifactEntry],
    config_registry: Any, artifact_registry: Any, model_registry: Any, metadata_registry: Any,
    threshold_registry: Any, export_registry: Any, inference_engine: Any, deployment_manifest: Any,
    deployment_metadata: Dict[str, Any], compatibility_report: Dict[str, Any],
    integrity_report: Dict[str, Any], deployment_readiness: Dict[str, Any],
    package_summary: Dict[str, Any], stage_summaries: Dict[str, Optional[Dict[str, Any]]],
) -> Dict[str, Any]:
    logger.info("MODULE START module=2 name=CompleteDeploymentReport")
    t0 = time.time()

    class_names = _class_names(threshold_registry, model_registry, inference_engine)
    manifest_dict = deployment_manifest.to_dict()

    entries_by_stage: Dict[str, int] = {}
    for entry in manifest_dict["entries"].values():
        origin = entry.get("stage_of_origin", "unknown")
        entries_by_stage[origin] = entries_by_stage.get(origin, 0) + 1

    perf = stage_summaries.get("stage05") or {}
    explain = stage_summaries.get("stage06") or {}

    report = {
        "project_overview": (
            f"{PROJECT_NAME} is a production-grade chest X-ray disease classification system "
            f"that predicts {model_registry.num_classes} NIH ChestXray14 disease labels from a "
            f"single frontal-view radiograph. The system was built stage-by-stage across Sprint 05, "
            f"moving a trained checkpoint through reconstruction, export, runtime, validation, "
            f"explainability, and packaging into a versioned, auditable release."
        ),
        "pipeline_architecture": STAGE_PIPELINE,
        "sprint_overview": {
            "sprint": "Sprint 05",
            "objective": "Take a trained checkpoint from Sprint 04 to a fully documented, "
                         "production-ready deployment release.",
            "total_stages": len(STAGE_PIPELINE),
            "current_stage": "Stage 08 (final)",
        },
        "completed_stages": [
            {
                **stage_info,
                "summary": stage_summaries.get(f"stage{str(i + 1).zfill(2)}"),
            }
            for i, stage_info in enumerate(STAGE_PIPELINE[:7])
        ],
        "deployment_architecture": {
            "reconstruction": "Checkpoint -> architecture rebuild -> strict state_dict match (Stage 02).",
            "export_targets": ["TorchScript", "ONNX"],
            "serving_runtimes": package_summary.get("supported_runtimes"),
            "inference_engine": "validate -> decode -> preprocess -> forward pass -> threshold -> structured JSON",
            "packaging": "Deployment manifest + compatibility + integrity + reproducibility + readiness score.",
        },
        "deployment_workflow": [
            "1. Client submits one or more image paths/identifiers.",
            "2. InferenceEngine.validate_and_decode_image() rejects unreadable/invalid files.",
            "3. preprocess_image() resizes, normalizes, and tensorizes accepted images.",
            "4. Model forward pass under torch.no_grad() produces logits -> sigmoid probabilities.",
            "5. Per-class thresholds are applied to derive predicted_diseases and confidence_scores.",
            "6. A structured PredictionResult is returned per image (success or error).",
        ],
        "model_details": {
            "backbone": model_registry.backbone,
            "num_classes": model_registry.num_classes,
            "class_names": class_names,
            "total_parameters": model_registry.total_parameters,
            "trainable_parameters": model_registry.trainable_parameters,
            "device": model_registry.device,
            "dtype": model_registry.dtype,
            "checkpoint_path": model_registry.checkpoint_path,
            "checkpoint_sha256": model_registry.checkpoint_sha256,
            "validation": _json_safe(model_registry.validation),
        },
        "inference_runtime": {
            "model_version": inference_engine.model_version,
            "model_fingerprint_sha256": inference_engine.model_fingerprint,
            "device": str(inference_engine.device),
            "preprocessing_config": inference_engine.preprocessing_config.to_dict(),
            "num_classes_served": len(inference_engine.class_names),
        },
        "export_pipeline": export_registry.to_dict(),
        "performance": perf if perf else {"available": False, "note": "Stage 05 summary not found on disk."},
        "explainability": {
            "methods_available": deployment_metadata.get("explainability_methods_available"),
            "stage06_summary": explain if explain else None,
        },
        "compatibility": compatibility_report,
        "integrity": integrity_report,
        "readiness": deployment_readiness,
        "package_contents": {
            "total_artifacts": manifest_dict["total_artifacts"],
            "total_size_bytes": manifest_dict["total_size_bytes"],
            "total_size_human": _format_bytes(manifest_dict["total_size_bytes"]),
            "missing_critical": manifest_dict["missing_critical"],
            "artifacts_by_stage_of_origin": entries_by_stage,
        },
        "engineering_validation": {
            "per_stage": {k: (v or {}).get("engineering_checks_passed") for k, v in stage_summaries.items()},
            "per_stage_total": {k: (v or {}).get("engineering_checks_total") for k, v in stage_summaries.items()},
            "integrity_passed": integrity_report.get("passed"),
            "compatibility_passed": compatibility_report.get("passed"),
        },
        "future_work": FUTURE_ROADMAP,
        "conclusion": (
            f"Sprint 05 delivers a checksum-verified, dual-runtime (TorchScript + ONNX) deployment "
            f"package for the {model_registry.backbone} chest X-ray classifier, scoring "
            f"{deployment_readiness.get('overall_score')}/100 on the automated readiness gate "
            f"({deployment_readiness.get('readiness_level')}). The package is ready for integration "
            f"into a serving layer, subject to the clinical and research-use disclaimers in the model card."
        ),
        "generated_utc": _utc_now(),
    }

    _write_json(
        dirs["documentation"] / "deployment_report.json", report,
        "deployment_report", "documentation", generated_files, logger,
    )

    md_lines: List[str] = [
        f"# {PROJECT_NAME} -- Sprint 05 Complete Deployment Report", "",
        "## 1. Project Overview", report["project_overview"], "",
        "## 2. Pipeline Architecture",
    ]
    for stage in STAGE_PIPELINE:
        md_lines.append(f"- **{stage['stage']}: {stage['name']}** -- {', '.join(stage['outputs'])}")
    md_lines += [
        "", "## 3. Sprint Overview",
        f"- Sprint: {report['sprint_overview']['sprint']}",
        f"- Objective: {report['sprint_overview']['objective']}",
        f"- Stages: {report['sprint_overview']['total_stages']} ({report['sprint_overview']['current_stage']})",
        "", "## 4. Completed Stages",
    ]
    for cs in report["completed_stages"]:
        status = (cs["summary"] or {}).get("status", "UNKNOWN") if cs["summary"] else "SUMMARY_UNAVAILABLE"
        md_lines.append(f"- **{cs['stage']}: {cs['name']}** -- status={status}")
    md_lines += [
        "", "## 5. Deployment Architecture",
    ]
    for k, v in report["deployment_architecture"].items():
        md_lines.append(f"- **{k}**: {v}")
    md_lines += ["", "## 6. Deployment Workflow"]
    md_lines += report["deployment_workflow"]
    md_lines += [
        "", "## 7. Model Details",
        f"- Backbone: {report['model_details']['backbone']}",
        f"- Classes: {report['model_details']['num_classes']}",
        f"- Total Parameters: {report['model_details']['total_parameters']:,}",
        f"- Checkpoint SHA256: `{report['model_details']['checkpoint_sha256']}`",
        "", "## 8. Inference Runtime",
        f"- Model Version: {report['inference_runtime']['model_version']}",
        f"- Device: {report['inference_runtime']['device']}",
        "", "## 9. Export Pipeline",
        f"- TorchScript: method={export_registry.torchscript.export_method} success={export_registry.torchscript.success}",
        f"- ONNX: opset={export_registry.onnx.opset_version} success={export_registry.onnx.success}",
        f"- Numerical Validation Passed: {export_registry.numerical_validation.passed}",
        "", "## 10. Performance",
    ]
    if perf:
        md_lines.append(f"```json\n{json.dumps(_json_safe(perf), indent=2)}\n```")
    else:
        md_lines.append("_Stage 05 summary not found on disk; performance figures unavailable._")
    md_lines += [
        "", "## 11. Explainability",
        f"- Methods Available: {report['explainability']['methods_available']}",
        "", "## 12. Compatibility",
        f"- Passed: {compatibility_report.get('passed')}",
        "", "## 13. Integrity",
        f"- Passed: {integrity_report.get('passed')}",
        "", "## 14. Readiness",
        f"- Score: {deployment_readiness.get('overall_score')} ({deployment_readiness.get('readiness_level')})",
        "", "## 15. Package Contents",
        f"- Total Artifacts: {report['package_contents']['total_artifacts']}",
        f"- Total Size: {report['package_contents']['total_size_human']}",
        "", "## 16. Engineering Validation",
        f"- Integrity Passed: {report['engineering_validation']['integrity_passed']}",
        f"- Compatibility Passed: {report['engineering_validation']['compatibility_passed']}",
        "", "## 17. Future Work",
    ]
    md_lines += [f"- {item}" for item in FUTURE_ROADMAP]
    md_lines += ["", "## 18. Conclusion", report["conclusion"], ""]

    _write_text(
        dirs["documentation"] / "deployment_report.md", "\n".join(md_lines),
        "deployment_report_md", "documentation", generated_files, logger,
    )

    elapsed = time.time() - t0
    logger.info("MODULE FINISH module=2 name=CompleteDeploymentReport elapsed=%.3fs", elapsed)
    return report


# ======================================================================
# MODULE 3 -- MODEL CARD (HuggingFace style)
# ======================================================================
def build_module3_model_card(
    dirs: Dict[str, Path], logger: logging.Logger, generated_files: List[ReleaseArtifactEntry],
    model_registry: Any, metadata_registry: Any, threshold_registry: Any, export_registry: Any,
    inference_engine: Any, compatibility_report: Dict[str, Any], integrity_report: Dict[str, Any],
    package_summary: Dict[str, Any],
) -> Dict[str, Any]:
    logger.info("MODULE START module=3 name=ModelCard")
    t0 = time.time()

    class_names = _class_names(threshold_registry, model_registry, inference_engine)
    preprocessing = inference_engine.preprocessing_config.to_dict()
    metadata_dict = metadata_registry.to_dict()

    limitations: List[str] = [
        "Single frontal-view chest radiograph input only; no lateral-view or multi-view fusion.",
        "Multi-label thresholds are calibrated on the Sprint 04 evaluation split and may not "
        "transfer directly to a different patient population, scanner, or acquisition protocol.",
        "The model has not undergone prospective clinical validation.",
    ]
    limitations.extend(f"Threshold registry warning: {w}" for w in threshold_registry.validation_errors)
    limitations.extend(f"Compatibility warning: {w}" for w in compatibility_report.get("warnings", []))
    limitations.extend(f"Integrity warning: {w}" for w in integrity_report.get("warnings", []))

    card = {
        "model_name": MODEL_DISPLAY_NAME,
        "version": package_summary.get("package_version"),
        "description": (
            f"A {model_registry.backbone}-based multi-label image classifier that estimates the "
            f"probability of {model_registry.num_classes} thoracic disease findings from a single "
            f"chest X-ray image, following the NIH ChestX-ray14 label taxonomy."
        ),
        "purpose": "Research-oriented decision-support signal for thoracic disease screening on chest radiographs.",
        "architecture": {
            "backbone": model_registry.backbone,
            "total_parameters": model_registry.total_parameters,
            "trainable_parameters": model_registry.trainable_parameters,
            "output_head": f"{model_registry.num_classes}-way sigmoid multi-label classifier",
        },
        "training_dataset": {
            "name": "NIH Chest X-ray14",
            "training_metadata_available": bool(metadata_dict.get("training_metadata")),
            "training_metadata_keys": sorted((metadata_dict.get("training_metadata") or {}).keys()),
            "evaluation_metadata_available": bool(metadata_dict.get("evaluation_metadata")),
            "evaluation_metadata_keys": sorted((metadata_dict.get("evaluation_metadata") or {}).keys()),
        },
        "input_resolution": {
            "height": preprocessing["resize_height"],
            "width": preprocessing["resize_width"],
            "channels": preprocessing["channels"],
            "normalization_mean": preprocessing["mean"],
            "normalization_std": preprocessing["std"],
        },
        "output_classes": class_names,
        "thresholding": {
            "per_class_thresholds": threshold_registry.thresholds,
            "source": threshold_registry.source_path,
            "has_extended_metadata": bool(threshold_registry.threshold_metadata),
        },
        "inference_pipeline": "decode -> resize -> normalize -> forward pass (sigmoid) -> per-class threshold",
        "deployment_formats": {
            "torchscript": {
                "method": export_registry.torchscript.export_method,
                "success": export_registry.torchscript.success,
                "path": export_registry.torchscript.output_path,
            },
            "onnx": {
                "opset_version": export_registry.onnx.opset_version,
                "success": export_registry.onnx.success,
                "checker_passed": export_registry.onnx.checker_passed,
                "path": export_registry.onnx.output_path,
            },
        },
        "supported_hardware": compatibility_report.get("support_matrix"),
        "expected_inputs": "RGB image file (png/jpg/jpeg/bmp/tiff), any resolution -- resized internally.",
        "expected_outputs": (
            "JSON PredictionResult: predicted_diseases, confidence_scores, probabilities, "
            "thresholds_used, model_version, model_fingerprint_sha256, inference_timestamp_utc."
        ),
        "known_limitations": limitations,
        "failure_cases": KNOWN_INFERENCE_ERROR_TYPES,
        "ethical_considerations": (
            "Predictions reflect patterns in historical, retrospectively-labeled data and may "
            "encode dataset-specific biases (acquisition site, patient demographics, label noise "
            "inherited from the original NLP-derived NIH labels). Outputs must not be used as the "
            "sole basis for a clinical decision."
        ),
        "clinical_disclaimer": (
            "This model is NOT approved for clinical diagnosis, triage, or treatment decisions. "
            "It has not been cleared by any regulatory body (e.g., FDA, CE). Any clinical use "
            "requires independent validation, regulatory clearance, and qualified human oversight."
        ),
        "research_use_disclaimer": (
            "Provided for research and engineering demonstration purposes only. Users are "
            "responsible for independently validating performance on their own data before any "
            "downstream use."
        ),
        "version_info": package_summary.get("package_version"),
        "authors": f"{PROJECT_NAME} ML Platform Engineering Team",
        "license": "Research use only -- not licensed for clinical deployment.",
        "generated_utc": _utc_now(),
    }

    _write_json(
        dirs["model_card"] / "model_card.json", card,
        "model_card", "model_card", generated_files, logger,
    )

    md = [
        "---",
        f"model_name: {card['model_name']}",
        f"version: {card['version']}",
        "tags: [chest-xray, multi-label-classification, medical-imaging, pytorch, onnx, torchscript]",
        "---",
        f"# {card['model_name']}", "",
        "## Model Description", card["description"], "",
        f"**Purpose:** {card['purpose']}", "",
        "## Architecture",
        f"- Backbone: {card['architecture']['backbone']}",
        f"- Total parameters: {card['architecture']['total_parameters']:,}",
        f"- Output head: {card['architecture']['output_head']}", "",
        "## Training Dataset",
        f"- Dataset: {card['training_dataset']['name']}",
        f"- Training metadata available: {card['training_dataset']['training_metadata_available']}",
        f"- Evaluation metadata available: {card['training_dataset']['evaluation_metadata_available']}", "",
        "## Input Resolution",
        f"- {card['input_resolution']['height']} x {card['input_resolution']['width']} x "
        f"{card['input_resolution']['channels']}", "",
        "## Output Classes",
    ]
    md += [f"- {c}" for c in class_names]
    md += [
        "", "## Thresholding",
        f"- Source: `{card['thresholding']['source']}`",
        "```json",
        json.dumps(card["thresholding"]["per_class_thresholds"], indent=2),
        "```",
        "", "## Inference Pipeline", card["inference_pipeline"], "",
        "## Deployment Formats",
        f"- TorchScript ({card['deployment_formats']['torchscript']['method']}): "
        f"success={card['deployment_formats']['torchscript']['success']}",
        f"- ONNX (opset {card['deployment_formats']['onnx']['opset_version']}): "
        f"success={card['deployment_formats']['onnx']['success']}", "",
        "## Supported Hardware",
        "```json",
        json.dumps(card["supported_hardware"], indent=2),
        "```",
        "", "## Expected Inputs", card["expected_inputs"], "",
        "## Expected Outputs", card["expected_outputs"], "",
        "## Known Limitations",
    ]
    md += [f"- {l}" for l in card["known_limitations"]]
    md += ["", "## Failure Cases (structured error types)"]
    md += [f"- `{e}`" for e in card["failure_cases"]]
    md += [
        "", "## Ethical Considerations", card["ethical_considerations"], "",
        "## Clinical Disclaimer", card["clinical_disclaimer"], "",
        "## Research Use Disclaimer", card["research_use_disclaimer"], "",
        f"**Version:** {card['version_info']}  ",
        f"**Authors:** {card['authors']}  ",
        f"**License:** {card['license']}", "",
    ]
    _write_text(
        dirs["model_card"] / "model_card.md", "\n".join(md),
        "model_card_md", "model_card", generated_files, logger,
    )

    elapsed = time.time() - t0
    logger.info("MODULE FINISH module=3 name=ModelCard elapsed=%.3fs", elapsed)
    return card


# ======================================================================
# MODULE 4 -- DEPLOYMENT GUIDE
# ======================================================================
def _package_tree(package_dir: Optional[str], max_entries: int = 200) -> List[str]:
    if not package_dir:
        return ["(package directory not recorded)"]
    root = Path(package_dir)
    if not root.exists():
        return [f"(package directory not found on disk: {root})"]
    lines: List[str] = [f"{root.name}/"]
    count = 0
    for p in sorted(root.rglob("*")):
        if count >= max_entries:
            lines.append("... (truncated)")
            break
        depth = len(p.relative_to(root).parts) - 1
        prefix = "  " * (depth + 1)
        suffix = "/" if p.is_dir() else ""
        lines.append(f"{prefix}{p.name}{suffix}")
        count += 1
    return lines


def build_module4_deployment_guide(
    dirs: Dict[str, Path], logger: logging.Logger, generated_files: List[ReleaseArtifactEntry],
    model_registry: Any, threshold_registry: Any, export_registry: Any, inference_engine: Any,
    deployment_readiness: Dict[str, Any], compatibility_report: Dict[str, Any],
    package_summary: Dict[str, Any],
) -> str:
    logger.info("MODULE START module=4 name=DeploymentGuide")
    t0 = time.time()

    preprocessing = inference_engine.preprocessing_config.to_dict()
    ts_path = export_registry.torchscript.output_path
    onnx_path = export_registry.onnx.output_path
    checkpoint_path = model_registry.checkpoint_path

    checklist = [
        f"- [{'x' if ok else ' '}] {name}"
        for name, ok in deployment_readiness.get("checks", {}).items()
    ]

    # Precompute every dynamic value as a plain variable *before* building the
    # f-string. Python's f-string grammar (pre-3.12 / PEP 701) does not allow
    # the same quote character used to delimit an f-string to reappear inside
    # its {...} expression parts -- so no nested "..." or f"..." is used
    # inside the f\"\"\"...\"\"\" block below. This keeps the cell portable
    # across Python 3.8-3.13+ regardless of the kernel's interpreter version.
    package_tree_text = "\n".join(_package_tree(package_summary.get("package_directory")))
    python_version = compatibility_report.get("python", {}).get("version")
    python_range = compatibility_report.get("python", {}).get("supported_range")
    torch_compat_version = compatibility_report.get("torch", {}).get("version")
    torch_min_major = compatibility_report.get("torch", {}).get("minimum_supported_major")
    torchvision_compat_version = compatibility_report.get("torchvision", {}).get("version")
    onnx_opset_compat = compatibility_report.get("onnx", {}).get("opset_version")
    cuda_available = compatibility_report.get("cuda", {}).get("available")
    cuda_device_configured = compatibility_report.get("cuda", {}).get("device_configured")
    thresholds_json = json.dumps(threshold_registry.thresholds, indent=2)
    error_types_md = "\n".join(f"- `{e}`" for e in KNOWN_INFERENCE_ERROR_TYPES)
    checklist_md = "\n".join(checklist)

    md = f"""# {MODEL_DISPLAY_NAME} -- Deployment Guide

## 1. Package Folder Structure

## 2. Environment Setup

- Python: {python_version} (supported range {python_range})
- Torch: {torch_compat_version} (minimum major version {torch_min_major})
- Torchvision: {torchvision_compat_version}
- ONNX opset: {onnx_opset_compat}
- CUDA available: {cuda_available} (configured device: {cuda_device_configured})

```bash
pip install torch torchvision onnx onnxruntime pillow numpy
```

## 3. Loading TorchScript

```python
import torch
model = torch.jit.load("{ts_path}", map_location="cpu")
model.eval()
```

## 4. Loading ONNX

```python
import onnxruntime as ort
session = ort.InferenceSession("{onnx_path}", providers=["CPUExecutionProvider"])
```

## 5. Loading Native PyTorch Checkpoint

```python
import torch
checkpoint = torch.load("{checkpoint_path}", map_location="cpu")
# Reconstruct the {model_registry.backbone} architecture (see Stage 02), then:
# model.load_state_dict(state_dict, strict=True); model.eval()
```

## 6. Preprocessing

- Resize to: {preprocessing['resize_height']} x {preprocessing['resize_width']} \
({preprocessing['resize_source']})
- Channels: {preprocessing['channels']}
- Normalization mean: {preprocessing['mean']} ({preprocessing['mean_source']})
- Normalization std: {preprocessing['std']} ({preprocessing['std_source']})

```python
from PIL import Image
import numpy as np
import torch

img = Image.open(path).convert("RGB").resize(
    ({preprocessing['resize_width']}, {preprocessing['resize_height']}), Image.BILINEAR
)
array = np.asarray(img, dtype=np.float32) / 255.0
tensor = torch.from_numpy(array).permute(2, 0, 1)
mean = torch.tensor({preprocessing['mean']}).view(-1, 1, 1)
std = torch.tensor({preprocessing['std']}).view(-1, 1, 1)
tensor = (tensor - mean) / std
```

## 7. Thresholds

Per-class deployment thresholds (source: `{threshold_registry.source_path}`):

```json
{thresholds_json}
```

## 8. Single Image Inference

```python
result = engine.predict_image("/path/to/image.png")
```

## 9. Batch Inference

```python
results = engine.predict_batch(
    ["/path/a.png", "/path/b.png"],
    image_identifiers=["patient_a", "patient_b"],
)
```

## 10. Expected Output (JSON Response)

```json
{{
  "image_identifier": "patient_a",
  "success": true,
  "predicted_diseases": ["Effusion"],
  "confidence_scores": {{"Effusion": 0.81}},
  "probabilities": {{"...": "..."}},
  "thresholds_used": {{"...": "..."}},
  "inference_timestamp_utc": "2026-07-05T00:00:00+00:00",
  "model_fingerprint_sha256": "{model_registry.checkpoint_sha256}",
  "model_version": "{inference_engine.model_version}",
  "error": null
}}
```

## 11. Error Handling

On decode/preprocessing failure, `success` is `false` and `error.error_type` is one of:

{error_types_md}

## 12. Deployment Checklist

{checklist_md}

Overall readiness: **{deployment_readiness.get('readiness_level')}** \
({deployment_readiness.get('overall_score')}/100)
"""

    _write_text(
        dirs["deployment_guides"] / "deployment_guide.md", md,
        "deployment_guide_md", "deployment_guides", generated_files, logger,
    )

    elapsed = time.time() - t0
    logger.info("MODULE FINISH module=4 name=DeploymentGuide elapsed=%.3fs", elapsed)
    return md


# ======================================================================
# MODULE 5 -- API DOCUMENTATION
# ======================================================================
def build_module5_api_documentation(
    dirs: Dict[str, Path], logger: logging.Logger, generated_files: List[ReleaseArtifactEntry],
    model_registry: Any, threshold_registry: Any, inference_engine: Any,
) -> Dict[str, Any]:
    logger.info("MODULE START module=5 name=APIDocumentation")
    t0 = time.time()

    class_names = _class_names(threshold_registry, model_registry, inference_engine)

    contract = {
        "endpoint": "/v1/predict",
        "method": "POST",
        "request_schema": {
            "type": "object",
            "properties": {
                "images": {
                    "type": "array",
                    "items": {"type": "string", "description": "Absolute or accessible image path/URI."},
                    "minItems": 1,
                },
                "image_identifiers": {
                    "type": "array",
                    "items": {"type": "string"},
                    "description": "Optional caller-supplied identifiers, one per image (defaults to the path).",
                },
            },
            "required": ["images"],
        },
        "response_schema": {
            "type": "object",
            "properties": {
                "predictions": {"type": "array", "items": {"$ref": "#/definitions/PredictionObject"}},
            },
        },
        "prediction_object": {
            "image_identifier": "string",
            "success": "boolean",
            "predicted_diseases": "array[string]",
            "confidence_scores": "object<string, float 0-1>",
            "probabilities": "object<string, float 0-1>",
            "thresholds_used": "object<string, float 0-1>",
            "inference_timestamp_utc": "string (ISO-8601 UTC) | null",
            "model_fingerprint_sha256": "string | null",
            "model_version": "string | null",
            "error": "object{error_type, message} | null",
        },
        "class_names": class_names,
        "top_k_predictions": {
            "note": "Not thresholded -- caller may rank `probabilities` descending and slice top-k client-side.",
            "example_k": 5,
        },
        "batch_inference": {
            "supported": True,
            "max_batch_size": "unbounded (memory-bound); documented validated sizes: 1, 2, 4, 8, 16, 32",
        },
        "error_codes": {
            "400": "Malformed request body / missing required field.",
            "415": "Unsupported media type (see `unsupported_extension`).",
            "422": "Image failed decode/validation (see per-item `error.error_type`).",
            "500": "Unhandled server/inference error.",
        },
        "validation_rules": [
            "images must be a non-empty array.",
            "Each image must decode to 1, 3, or 4 source bands and convert to RGB.",
            "Each image must be non-zero-byte and have a supported extension "
            "(.png, .jpg, .jpeg, .bmp, .tiff, .tif).",
        ],
        "example_request": {
            "images": ["/data/patient_001.png", "/data/patient_002.png"],
            "image_identifiers": ["patient_001", "patient_002"],
        },
        "example_response": {
            "predictions": [
                {
                    "image_identifier": "patient_001",
                    "success": True,
                    "predicted_diseases": [class_names[0]] if class_names else [],
                    "confidence_scores": {class_names[0]: 0.81} if class_names else {},
                    "model_version": inference_engine.model_version,
                    "error": None,
                },
                {
                    "image_identifier": "patient_002",
                    "success": False,
                    "error": {"error_type": "corrupted_image", "message": "Failed to decode image: ..."},
                },
            ]
        },
        "generated_utc": _utc_now(),
    }

    _write_json(
        dirs["api"] / "api_contract.json", contract,
        "api_contract", "api", generated_files, logger,
    )

    md = [
        f"# {MODEL_DISPLAY_NAME} -- API Reference", "",
        f"## Endpoint: `{contract['method']} {contract['endpoint']}`", "",
        "### Request Schema",
        "```json", json.dumps(contract["request_schema"], indent=2), "```", "",
        "### Response Schema",
        "```json", json.dumps(contract["response_schema"], indent=2), "```", "",
        "### Prediction Object",
        "```json", json.dumps(contract["prediction_object"], indent=2), "```", "",
        "### Error Codes",
    ]
    for code, desc in contract["error_codes"].items():
        md.append(f"- `{code}`: {desc}")
    md += ["", "### Validation Rules"]
    md += [f"- {r}" for r in contract["validation_rules"]]
    md += [
        "", "### Example Request",
        "```json", json.dumps(contract["example_request"], indent=2), "```", "",
        "### Example Response",
        "```json", json.dumps(contract["example_response"], indent=2), "```", "",
    ]
    _write_text(
        dirs["api"] / "api_reference.md", "\n".join(md),
        "api_reference_md", "api", generated_files, logger,
    )

    elapsed = time.time() - t0
    logger.info("MODULE FINISH module=5 name=APIDocumentation elapsed=%.3fs", elapsed)
    return contract


# ======================================================================
# MODULE 6 -- RELEASE NOTES
# ======================================================================
def build_module6_release_notes(
    dirs: Dict[str, Path], logger: logging.Logger, generated_files: List[ReleaseArtifactEntry],
    package_summary: Dict[str, Any], deployment_readiness: Dict[str, Any],
    compatibility_report: Dict[str, Any], integrity_report: Dict[str, Any],
    threshold_registry: Any, stage_summaries: Dict[str, Optional[Dict[str, Any]]],
) -> str:
    logger.info("MODULE START module=6 name=ReleaseNotes")
    t0 = time.time()

    perf = stage_summaries.get("stage05") or {}
    explain = stage_summaries.get("stage06") or {}

    limitations = list(threshold_registry.validation_errors)
    limitations.extend(compatibility_report.get("warnings", []))
    limitations.extend(integrity_report.get("warnings", []))
    if not limitations:
        limitations = ["No open warnings recorded at release time."]

    # Precompute every dynamic block as a plain variable *before* building the
    # f-string, for the same portability reason documented in Module 4: no
    # nested "..." / f"..." inside the f\"\"\"...\"\"\" expression parts below.
    completed_work_md = "\n".join(f"- {s['stage']}: {s['name']}" for s in STAGE_PIPELINE)
    supported_runtimes_text = ", ".join(package_summary.get("supported_runtimes") or [])
    supported_devices_text = ", ".join(package_summary.get("supported_devices") or [])
    methods_passed_text = explain.get("methods_passed", "see stage06_summary.json")
    if perf:
        performance_md = "```json\n" + json.dumps(_json_safe(perf), indent=2) + "\n```"
    else:
        performance_md = "_Stage 05 summary not found on disk._"
    package_size_text = _format_bytes(package_summary.get("total_package_size_bytes"))
    limitations_md = "\n".join(f"- {l}" for l in limitations)
    future_roadmap_md = "\n".join(f"- {r}" for r in FUTURE_ROADMAP)

    md = f"""# Release Notes -- {PROJECT_NAME} {package_summary.get('package_version')}

**Release date (UTC):** {_utc_now()}

## Sprint 05 Overview

Sprint 05 took the Sprint 04 training checkpoint through reconstruction, dual-format export
(TorchScript + ONNX), a production inference runtime, performance/robustness validation,
an explainability runtime, deployment packaging, and this final documentation & release
engineering stage.

## Completed Work

{completed_work_md}

## Major Improvements

- Deterministic, strictly-validated model reconstruction from checkpoint (Stage 02).
- Dual-runtime export (TorchScript + ONNX) with cross-runtime numerical validation (Stage 03).
- Deterministic batch-capable inference runtime with structured error handling (Stage 04).

## Deployment Features

- Supported runtimes: {supported_runtimes_text}
- Supported devices: {supported_devices_text}
- Checksum-verified deployment manifest and reproducibility manifest (Stage 07).

## Explainability Features

- Methods available: {methods_passed_text}

## Performance Summary

{performance_md}

## Package Summary

- Package version: {package_summary.get('package_version')}
- Artifacts packaged: {package_summary.get('artifacts_packaged')}
- Package size: {package_size_text}
- Readiness: {deployment_readiness.get('readiness_level')} ({deployment_readiness.get('overall_score')}/100)

## Known Limitations

{limitations_md}

## Future Roadmap

{future_roadmap_md}
"""

    _write_text(
        dirs["release"] / "release_notes.md", md,
        "release_notes_md", "release", generated_files, logger,
    )

    elapsed = time.time() - t0
    logger.info("MODULE FINISH module=6 name=ReleaseNotes elapsed=%.3fs", elapsed)
    return md


# ======================================================================
# MODULE 8 -- RELEASE MANIFEST
# ======================================================================
def build_module8_release_manifest(
    dirs: Dict[str, Path], logger: logging.Logger, generated_files: List[ReleaseArtifactEntry],
    package_summary: Dict[str, Any], deployment_readiness: Dict[str, Any],
    reproducibility_manifest: Dict[str, Any], deployment_manifest: Any,
) -> Dict[str, Any]:
    logger.info("MODULE START module=8 name=ReleaseManifest")
    t0 = time.time()

    dependencies = {
        "torch": _lib_version("torch"),
        "torchvision": _lib_version("torchvision"),
        "onnx": _lib_version("onnx"),
        "onnxruntime": _lib_version("onnxruntime"),
        "numpy": _lib_version("numpy"),
        "pillow": _lib_version("pillow"),
    }

    manifest = {
        "version": package_summary.get("package_version"),
        "build_timestamp_utc": _utc_now(),
        "artifacts": [asdict(entry) for entry in generated_files],
        "checksums": {entry.artifact_id: entry.sha256 for entry in generated_files},
        "dependencies": dependencies,
        "framework_versions": reproducibility_manifest.get("framework_versions"),
        "deployment_score": deployment_readiness.get("overall_score"),
        "readiness": deployment_readiness.get("readiness_level"),
        "package_directory": package_summary.get("package_directory"),
        "output_directory": package_summary.get("output_directory"),
        "stage07_total_artifacts": deployment_manifest.total_artifacts,
        "stage07_total_size_bytes": deployment_manifest.total_size_bytes,
    }

    _write_json(
        dirs["release"] / "release_manifest.json", manifest,
        "release_manifest", "release", generated_files, logger,
    )

    elapsed = time.time() - t0
    logger.info("MODULE FINISH module=8 name=ReleaseManifest elapsed=%.3fs", elapsed)
    return manifest


# ======================================================================
# MODULE 9 -- FINAL SPRINT REPORT
# ======================================================================
def build_module9_final_sprint_report(
    dirs: Dict[str, Path], logger: logging.Logger, generated_files: List[ReleaseArtifactEntry],
    model_registry: Any, export_registry: Any, deployment_readiness: Dict[str, Any],
    compatibility_report: Dict[str, Any], integrity_report: Dict[str, Any],
    package_summary: Dict[str, Any], stage_summaries: Dict[str, Optional[Dict[str, Any]]],
) -> Dict[str, Any]:
    logger.info("MODULE START module=9 name=FinalSprintReport")
    t0 = time.time()

    perf = stage_summaries.get("stage05") or {}
    explain = stage_summaries.get("stage06") or {}

    report = {
        "sprint": "Sprint 05",
        "final_stage": "Stage 08 -- Deployment Documentation & Release Engineering",
        "stages": [
            {
                **stage_info,
                "summary": stage_summaries.get(f"stage{str(i + 1).zfill(2)}") if i < 7 else None,
            }
            for i, stage_info in enumerate(STAGE_PIPELINE)
        ],
        "engineering_achievements": [
            "Deterministic checkpoint reconstruction with strict state_dict validation.",
            "Dual-runtime export (TorchScript + ONNX) with cross-runtime numerical parity checks.",
            "Deterministic, structured, batch-capable production inference runtime.",
            "Automated performance, robustness, and memory-leak validation.",
            "Seven-method explainability runtime (GradCAM family, gradient-based, perturbation-based).",
            "Checksum-verified, versioned deployment package with automated readiness scoring.",
            "Complete production documentation set (this stage).",
        ],
        "deployment_readiness": deployment_readiness,
        "performance": perf if perf else {"available": False},
        "explainability": explain if explain else {"available": False},
        "compatibility": {"passed": compatibility_report.get("passed"), "warnings": compatibility_report.get("warnings")},
        "integrity": {"passed": integrity_report.get("passed"), "warnings": integrity_report.get("warnings")},
        "production_readiness": {
            "score": deployment_readiness.get("overall_score"),
            "level": deployment_readiness.get("readiness_level"),
            "blocking_issues": deployment_readiness.get("blocking_issues"),
        },
        "package_summary": package_summary,
        "future_sprint": {
            "sprint": "Sprint 06 (proposed)",
            "roadmap": FUTURE_ROADMAP,
        },
        "model_backbone": model_registry.backbone,
        "checkpoint_sha256": model_registry.checkpoint_sha256,
        "export_lineage_consistent": (
            export_registry.torchscript.model_hash == model_registry.checkpoint_sha256
            and export_registry.onnx.model_hash == model_registry.checkpoint_sha256
        ),
        "generated_utc": _utc_now(),
    }

    _write_json(
        dirs["reports"] / "Sprint05_Final_Report.json", report,
        "sprint05_final_report", "reports", generated_files, logger,
    )

    md = [
        "# Sprint 05 -- Final Report", "",
        f"**Generated (UTC):** {report['generated_utc']}", "",
        "## Stages",
    ]
    for s in report["stages"]:
        status = (s["summary"] or {}).get("status", "N/A") if s["summary"] else "N/A"
        md.append(f"- **{s['stage']}: {s['name']}** -- status={status}")
    md += ["", "## Engineering Achievements"]
    md += [f"- {a}" for a in report["engineering_achievements"]]
    md += [
        "", "## Production Readiness",
        f"- Score: {report['production_readiness']['score']}/100",
        f"- Level: {report['production_readiness']['level']}",
        f"- Blocking issues: {report['production_readiness']['blocking_issues'] or 'None'}",
        "", "## Compatibility & Integrity",
        f"- Compatibility passed: {report['compatibility']['passed']}",
        f"- Integrity passed: {report['integrity']['passed']}",
        "", "## Future Sprint Roadmap",
    ]
    md += [f"- {r}" for r in FUTURE_ROADMAP]
    md.append("")
    md.append(f"**Sprint 05 Status: {'COMPLETE' if report['production_readiness']['level'] in ('READY', 'READY_WITH_WARNINGS') else 'NOT COMPLETE'}**")

    _write_text(
        dirs["reports"] / "Sprint05_Final_Report.md", "\n".join(md),
        "sprint05_final_report_md", "reports", generated_files, logger,
    )

    elapsed = time.time() - t0
    logger.info("MODULE FINISH module=9 name=FinalSprintReport elapsed=%.3fs", elapsed)
    return report


# ======================================================================
# MODULE 7 -- DEPLOYMENT VALIDATION REPORT
# (walks the actual stage08_release tree on disk -- independent of the
#  in-memory generated_files ledger, so it catches everything.)
# ======================================================================
REQUIRED_STAGE08_FILES = [
    "reports/deployment_summary.json", "reports/deployment_summary.md",
    "documentation/deployment_report.json", "documentation/deployment_report.md",
    "model_card/model_card.md", "model_card/model_card.json",
    "deployment_guides/deployment_guide.md",
    "api/api_reference.md", "api/api_contract.json",
    "release/release_notes.md", "release/release_manifest.json",
    "reports/Sprint05_Final_Report.md", "reports/Sprint05_Final_Report.json",
]

REQUIRED_STAGE08_SUBDIRS = [
    "logs", "reports", "documentation", "model_card",
    "deployment_guides", "api", "release", "validation",
]


def build_module7_deployment_validation(
    dirs: Dict[str, Path], logger: logging.Logger, generated_files: List[ReleaseArtifactEntry],
    deployment_manifest: Any, integrity_report: Dict[str, Any],
) -> Dict[str, Any]:
    logger.info("MODULE START module=7 name=DeploymentValidationReport")
    logger.info("VALIDATION starting: filesystem + manifest + checksum + reference checks")
    t0 = time.time()

    checks: Dict[str, bool] = {}
    warnings: List[str] = []
    fatal_errors: List[str] = []

    stage_root = dirs["stage_root"]

    # 1. Correct folder hierarchy.
    missing_dirs = [d for d in REQUIRED_STAGE08_SUBDIRS if not (stage_root / d).is_dir()]
    checks["correct_folder_hierarchy"] = len(missing_dirs) == 0
    if missing_dirs:
        fatal_errors.append(f"Missing required Stage 08 subdirectories: {missing_dirs}")

    # 2. All required reports/documents exist.
    missing_files = [f for f in REQUIRED_STAGE08_FILES if not (stage_root / f).exists()]
    checks["all_reports_exist"] = len(missing_files) == 0
    if missing_files:
        fatal_errors.append(f"Missing required Stage 08 outputs: {missing_files}")

    # 3. All JSON files on disk under stage08_release are syntactically valid and non-empty.
    json_files = sorted(stage_root.rglob("*.json"))
    invalid_json: List[str] = []
    empty_outputs: List[str] = []
    for jf in json_files:
        try:
            content = jf.read_text()
            if not content.strip():
                empty_outputs.append(str(jf))
                continue
            parsed = json.loads(content)
            if isinstance(parsed, (dict, list)) and len(parsed) == 0:
                empty_outputs.append(str(jf))
        except Exception as exc:
            invalid_json.append(f"{jf}: {exc}")
    checks["all_json_valid"] = len(invalid_json) == 0
    checks["no_empty_outputs"] = len(empty_outputs) == 0
    fatal_errors.extend(f"Invalid JSON: {e}" for e in invalid_json)
    warnings.extend(f"Empty JSON output: {e}" for e in empty_outputs)

    # 4. No duplicate files (identical sha256 content living at >1 path).
    hash_to_paths: Dict[str, List[str]] = {}
    for entry in generated_files:
        hash_to_paths.setdefault(entry.sha256, []).append(entry.file_path)
    duplicate_groups = [paths for paths in hash_to_paths.values() if len(paths) > 1]
    checks["no_duplicate_files"] = len(duplicate_groups) == 0
    if duplicate_groups:
        warnings.append(f"Duplicate-content file groups detected: {duplicate_groups}")

    # 5. Manifest consistency -- Stage 07's deployment manifest is internally coherent.
    manifest_dict = deployment_manifest.to_dict()
    checks["manifest_consistency"] = (
        manifest_dict["total_artifacts"] == len(manifest_dict["entries"])
        and len(manifest_dict["missing_critical"]) == 0
    )
    if not checks["manifest_consistency"]:
        fatal_errors.append("Stage 07 deployment manifest is internally inconsistent or has missing critical artifacts.")

    # 6. Checksum consistency -- reuse Stage 07's integrity verdict (do not recompute file hashes
    #    of frozen upstream artifacts; that is Stage 07's responsibility).
    checks["checksum_consistency"] = len(integrity_report.get("checksum_mismatches", [])) == 0
    if not checks["checksum_consistency"]:
        fatal_errors.append("Stage 07 integrity report recorded checksum drift.")

    # 7. Metadata consistency -- every generated Stage 08 JSON carries the expected envelope keys.
    metadata_inconsistent: List[str] = []
    for jf in json_files:
        try:
            parsed = json.loads(jf.read_text())
        except Exception:
            continue
        if isinstance(parsed, dict) and not all(k in parsed for k in ("generator", "stage", "version", "generated_utc")):
            metadata_inconsistent.append(str(jf))
    checks["metadata_consistency"] = len(metadata_inconsistent) == 0
    if metadata_inconsistent:
        warnings.append(f"JSON files missing standard metadata envelope: {metadata_inconsistent}")

    # 8. Deployment package consistency -- Stage 08 references, but does not duplicate, the
    #    Stage 07 package directory.
    checks["deployment_package_consistency"] = deployment_manifest.total_artifacts > 0

    # 9. Correct references -- every generated_files entry actually exists at its recorded path.
    broken_references = [e.file_path for e in generated_files if not Path(e.file_path).exists()]
    checks["correct_references"] = len(broken_references) == 0
    if broken_references:
        fatal_errors.append(f"Broken references in generated_files ledger: {broken_references}")

    passed = len(fatal_errors) == 0
    for w in warnings:
        logger.warning("VALIDATION WARNING: %s", w)
    for e in fatal_errors:
        logger.error("VALIDATION FAILURE: %s", e)

    report = {
        "generated_utc": _utc_now(),
        "checks": checks,
        "checks_passed": sum(1 for v in checks.values() if v),
        "checks_total": len(checks),
        "warnings": warnings,
        "fatal_errors": fatal_errors,
        "passed": passed,
        "json_files_scanned": len(json_files),
        "tracked_artifacts": len(generated_files),
    }

    _write_json(
        dirs["validation"] / "deployment_validation.json", report,
        "deployment_validation", "validation", generated_files, logger,
    )

    md = [
        "# Stage 08 -- Deployment Validation Report", "",
        f"**Passed:** {report['passed']}  ",
        f"**Checks:** {report['checks_passed']}/{report['checks_total']}",
        "",
        "| Check | Result |", "|---|---|",
    ]
    md += [f"| {name} | {'PASS' if ok else 'FAIL'} |" for name, ok in checks.items()]
    md += ["", "## Warnings"]
    md += [f"- {w}" for w in warnings] or ["- None"]
    md += ["", "## Fatal Errors"]
    md += [f"- {e}" for e in fatal_errors] or ["- None"]

    _write_text(
        dirs["validation"] / "deployment_validation.md", "\n".join(md),
        "deployment_validation_md", "validation", generated_files, logger,
    )

    elapsed = time.time() - t0
    logger.info(
        "VALIDATION complete: %d/%d checks passed, passed=%s",
        report["checks_passed"], report["checks_total"], report["passed"],
    )
    logger.info("MODULE FINISH module=7 name=DeploymentValidationReport elapsed=%.3fs", elapsed)
    return report


# ======================================================================
# MODULE 10 -- ENGINEERING VALIDATION (Stage 08 gate)
# ======================================================================
def build_module10_engineering_validation(
    dirs: Dict[str, Path], logger: logging.Logger, generated_files: List[ReleaseArtifactEntry],
    deployment_validation: Dict[str, Any],
) -> Dict[str, Any]:
    logger.info("MODULE START module=10 name=EngineeringValidation")
    t0 = time.time()

    checks: Dict[str, bool] = {}
    warnings: List[str] = list(deployment_validation.get("warnings", []))
    fatal_errors: List[str] = []

    checks["no_missing_reports"] = deployment_validation["checks"].get("all_reports_exist", False)
    checks["no_missing_metadata"] = deployment_validation["checks"].get("metadata_consistency", False)
    checks["no_missing_documentation"] = deployment_validation["checks"].get("all_reports_exist", False)
    checks["no_broken_references"] = deployment_validation["checks"].get("correct_references", False)
    checks["no_duplicate_artifacts"] = deployment_validation["checks"].get("no_duplicate_files", False)
    checks["correct_folder_hierarchy"] = deployment_validation["checks"].get("correct_folder_hierarchy", False)

    version_values = set()
    for jf in Path(dirs["stage_root"]).rglob("*.json"):
        try:
            parsed = json.loads(jf.read_text())
        except Exception:
            continue
        if isinstance(parsed, dict) and "version" in parsed:
            version_values.add(parsed["version"])
    checks["no_inconsistent_versions"] = len(version_values) <= 1
    if not checks["no_inconsistent_versions"]:
        fatal_errors.append(f"Inconsistent 'version' envelope values across Stage 08 outputs: {sorted(version_values)}")

    checks["everything_internally_consistent"] = deployment_validation.get("passed", False)

    for name, ok in checks.items():
        if not ok and name not in ("everything_internally_consistent",):
            fatal_errors.append(f"Engineering check failed: {name}")

    passed = len(fatal_errors) == 0
    for w in warnings:
        logger.warning("ENGINEERING VALIDATION WARNING: %s", w)
    for e in fatal_errors:
        logger.error("ENGINEERING VALIDATION FAILURE: %s", e)

    result = {
        "generated_utc": _utc_now(),
        "checks": checks,
        "checks_passed": sum(1 for v in checks.values() if v),
        "checks_total": len(checks),
        "warnings": warnings,
        "fatal_errors": fatal_errors,
        "passed": passed,
    }

    _write_json(
        dirs["validation"] / "engineering_validation.json", result,
        "engineering_validation", "validation", generated_files, logger,
    )

    elapsed = time.time() - t0
    logger.info("MODULE FINISH module=10 name=EngineeringValidation elapsed=%.3fs", elapsed)
    return result


# ======================================================================
# MAIN STAGE ENTRY POINT
# ======================================================================
def run_stage08(
    config_registry: Any,
    artifact_registry: Any,
    model_registry: Any,
    metadata_registry: Any,
    threshold_registry: Any,
    export_registry: Any,
    inference_engine: Any,
    deployment_manifest: Any,
    deployment_metadata: Dict[str, Any],
    compatibility_report: Dict[str, Any],
    reproducibility_manifest: Dict[str, Any],
    integrity_report: Dict[str, Any],
    deployment_readiness: Dict[str, Any],
    package_summary: Dict[str, Any],
    reconstructed_model: Any,
) -> Dict[str, Any]:
    start_time = time.time()

    stage_root = OUTPUT_ROOT / STAGE8_DIR_NAME
    dirs = make_stage_dirs(stage_root)
    logger = build_logger(dirs["logs"])

    logger.info("START Sprint05-Stage08 Deployment Documentation & Release Engineering")
    log_resources(logger, "RESOURCE START")

    generated_files: List[ReleaseArtifactEntry] = []
    stage_summaries = _stage_summaries(logger)

    # ---- MODULE 1: EXECUTIVE DEPLOYMENT SUMMARY ------------------------
    build_module1_executive_summary(
        dirs, logger, generated_files, config_registry, model_registry, export_registry,
        threshold_registry, deployment_readiness, package_summary, deployment_manifest,
        reproducibility_manifest,
    )

    # ---- MODULE 2: COMPLETE DEPLOYMENT REPORT --------------------------
    build_module2_deployment_report(
        dirs, logger, generated_files, config_registry, artifact_registry, model_registry,
        metadata_registry, threshold_registry, export_registry, inference_engine,
        deployment_manifest, deployment_metadata, compatibility_report, integrity_report,
        deployment_readiness, package_summary, stage_summaries,
    )

    # ---- MODULE 3: MODEL CARD ------------------------------------------
    build_module3_model_card(
        dirs, logger, generated_files, model_registry, metadata_registry, threshold_registry,
        export_registry, inference_engine, compatibility_report, integrity_report, package_summary,
    )

    # ---- MODULE 4: DEPLOYMENT GUIDE -------------------------------------
    build_module4_deployment_guide(
        dirs, logger, generated_files, model_registry, threshold_registry, export_registry,
        inference_engine, deployment_readiness, compatibility_report, package_summary,
    )

    # ---- MODULE 5: API DOCUMENTATION -------------------------------------
    build_module5_api_documentation(
        dirs, logger, generated_files, model_registry, threshold_registry, inference_engine,
    )

    # ---- MODULE 6: RELEASE NOTES ------------------------------------------
    build_module6_release_notes(
        dirs, logger, generated_files, package_summary, deployment_readiness,
        compatibility_report, integrity_report, threshold_registry, stage_summaries,
    )

    # ---- MODULE 9: FINAL SPRINT REPORT (generated before the manifest so
    #      the manifest can checksum it too) -------------------------------
    build_module9_final_sprint_report(
        dirs, logger, generated_files, model_registry, export_registry, deployment_readiness,
        compatibility_report, integrity_report, package_summary, stage_summaries,
    )

    # ---- MODULE 8: RELEASE MANIFEST ---------------------------------------
    build_module8_release_manifest(
        dirs, logger, generated_files, package_summary, deployment_readiness,
        reproducibility_manifest, deployment_manifest,
    )

    # ---- MODULE 7: DEPLOYMENT VALIDATION REPORT ---------------------------
    deployment_validation = build_module7_deployment_validation(
        dirs, logger, generated_files, deployment_manifest, integrity_report,
    )

    # ---- MODULE 10: ENGINEERING VALIDATION (fail-fast gate) ----------------
    engineering_validation = build_module10_engineering_validation(
        dirs, logger, generated_files, deployment_validation,
    )

    if not engineering_validation["passed"]:
        log_resources(logger, "FINISH-FAILED")
        elapsed = time.time() - start_time
        logger.error("FINISH Sprint05-Stage08 status=FAILED elapsed=%.2fs", elapsed)
        raise RuntimeError(
            "Sprint05 Stage08 engineering validation FAILED (fail-fast):\n  - "
            + "\n  - ".join(engineering_validation["fatal_errors"])
        )

    log_resources(logger, "RESOURCE FINISH")
    elapsed = time.time() - start_time

    reports_generated = sum(1 for e in generated_files if e.file_path.endswith(".json"))
    documents_generated = sum(1 for e in generated_files if e.file_path.endswith(".md"))

    stage08_summary = {
        "stage": STAGE8_DIR_NAME,
        "status": "OK",
        "elapsed_seconds": round(elapsed, 3),
        "reports_generated": reports_generated,
        "documents_generated": documents_generated,
        "total_artifacts_generated": len(generated_files),
        "validation_checks_passed": deployment_validation["checks_passed"],
        "validation_checks_total": deployment_validation["checks_total"],
        "engineering_checks_passed": engineering_validation["checks_passed"],
        "engineering_checks_total": engineering_validation["checks_total"],
        "deployment_score": deployment_readiness.get("overall_score"),
        "readiness_level": deployment_readiness.get("readiness_level"),
        "warnings_count": len(engineering_validation["warnings"]),
        "output_directory": str(dirs["stage_root"]),
    }
    _write_json(
        dirs["stage_root"] / "stage08_summary.json", stage08_summary,
        "stage08_summary", "stage_root", generated_files, logger,
    )

    logger.info("SUMMARY reports=%d documents=%d artifacts=%d", reports_generated, documents_generated, len(generated_files))
    logger.info("FINISH Sprint05-Stage08 status=OK elapsed=%.2fs", elapsed)

    # ---- CONSOLE REPORT ---------------------------------------------------
    print("=" * 70)
    print("SPRINT05 — STAGE08")
    print("DEPLOYMENT DOCUMENTATION & RELEASE ENGINEERING")
    print("=" * 70)
    print(f"Reports Generated     : {reports_generated:02d}")
    print(f"Documents Generated   : {documents_generated:02d}")
    print(f"Validation Checks     : {deployment_validation['checks_passed']:02d}/{deployment_validation['checks_total']:02d}")
    print(f"Deployment Score      : {deployment_readiness.get('overall_score'):.2f}")
    print(f"Readiness             : {deployment_readiness.get('readiness_level')}")
    print(f"Engineering Checks    : {engineering_validation['checks_passed']:02d}/{engineering_validation['checks_total']:02d}")
    print(f"Warnings              : {len(engineering_validation['warnings'])}")
    print(f"Output Directory      : {dirs['stage_root']}")
    print()
    print("Sprint 05 : COMPLETE")
    print("=" * 70)

    return {
        "generated_files": generated_files,
        "deployment_validation": deployment_validation,
        "engineering_validation": engineering_validation,
        "summary": stage08_summary,
    }


# ======================================================================
# STAGE 7 -> STAGE 8 PREREQUISITE RESOLUTION LAYER
# (replaces the previous `if __name__ == "__main__":` block only —
#  no other Stage 8 module is touched)
# ======================================================================
_HARD_PREREQUISITES = [
    "CONFIG_REGISTRY", "ARTIFACT_REGISTRY", "MODEL_REGISTRY", "METADATA_REGISTRY",
    "THRESHOLD_REGISTRY", "EXPORT_REGISTRY", "INFERENCE_ENGINE", "DEPLOYMENT_MANIFEST",
    "RECONSTRUCTED_MODEL",
]  # live Python objects -- cannot be rebuilt from JSON, must come from a live Stage 01-07 run

_SOFT_PREREQUISITES = {
    "DEPLOYMENT_METADATA": ["deployment_metadata.json", "stage07_summary.json"],
    "COMPATIBILITY_REPORT": ["compatibility_report.json", "reports/compatibility_report.json", "stage07_summary.json"],
    "REPRODUCIBILITY_MANIFEST": ["reproducibility_manifest.json", "reports/reproducibility_manifest.json", "stage07_summary.json"],
    "INTEGRITY_REPORT": ["integrity_report.json", "reports/integrity_report.json", "stage07_summary.json"],
    "DEPLOYMENT_READINESS": ["deployment_readiness.json", "reports/deployment_readiness.json", "stage07_summary.json"],
    "PACKAGE_SUMMARY": ["stage07_summary.json", "package_summary.json", "reports/package_summary.json"],
}  # plain dicts -- safe to re-read from Stage 07's already-written JSON if the global is absent

_SOFT_KEY_HINTS = {
    "PACKAGE_SUMMARY": ["package_summary", "data", None],
    "DEPLOYMENT_METADATA": ["deployment_metadata", "data", None],
    "COMPATIBILITY_REPORT": ["compatibility_report", "data", None],
    "REPRODUCIBILITY_MANIFEST": ["reproducibility_manifest", "data", None],
    "INTEGRITY_REPORT": ["integrity_report", "data", None],
    "DEPLOYMENT_READINESS": ["deployment_readiness", "data", None],
}


def _find_in_globals(name: str, glb: Dict[str, Any]):
    """Exact name, common casings, or nested inside a *_result dict left
    by a stage's driver cell (e.g. _stage07_result['package_summary'])."""
    if name in glb:
        return glb[name], f"global '{name}'"
    lowered = name.lower()
    for candidate in (lowered, name.upper(), name.title().replace("_", "")):
        if candidate in glb:
            return glb[candidate], f"global '{candidate}'"
    for gname, gval in glb.items():
        if gname.startswith("_") or not isinstance(gval, dict):
            continue
        if lowered in gval:
            return gval[lowered], f"nested inside global '{gname}[\"{lowered}\"]'"
    return None, None


def _find_in_stage07_json(name: str, logger: logging.Logger):
    """Read-only recovery from Stage 07's own already-written JSON --
    not recomputation, just re-reading a frozen artifact."""
    for fname in _SOFT_PREREQUISITES.get(name, []):
        path = STAGE7_DIR / fname
        if not path.exists():
            continue
        try:
            parsed = json.loads(path.read_text())
        except Exception as exc:
            logger.warning("Found %s but could not parse it as JSON: %s", path, exc)
            continue
        for key in _SOFT_KEY_HINTS.get(name, [None]):
            if key is None:
                candidate = parsed
            elif isinstance(parsed, dict) and key in parsed:
                candidate = parsed[key]
            elif isinstance(parsed, dict) and isinstance(parsed.get("data"), dict) and key in parsed["data"]:
                candidate = parsed["data"][key]
            else:
                continue
            if isinstance(candidate, dict) and candidate:
                return candidate, str(path)
    return None, None


def resolve_stage08_prerequisites() -> Dict[str, Any]:
    """Gathers everything run_stage08() needs. Tolerates a missing
    PACKAGE_SUMMARY (or any other soft prerequisite) by re-reading
    Stage 07's on-disk JSON. Raises ONE descriptive RuntimeError naming
    every unresolvable prerequisite instead of a bare NameError."""
    bootstrap_logger = logging.getLogger("sprint05.stage08.bootstrap")
    if not bootstrap_logger.handlers:
        h = logging.StreamHandler(sys.stdout)
        h.setFormatter(logging.Formatter("%(asctime)s | %(levelname)-8s | %(message)s", datefmt="%Y-%m-%d %H:%M:%S"))
        bootstrap_logger.addHandler(h)
        bootstrap_logger.setLevel(logging.INFO)

    glb = globals()
    resolved: Dict[str, Any] = {}
    missing_hard, missing_soft, notes = [], [], []

    for name in _HARD_PREREQUISITES:
        value, source = _find_in_globals(name, glb)
        if value is None:
            missing_hard.append(name)
        else:
            resolved[name] = value
            if source != f"global '{name}'":
                notes.append(f"{name} recovered from {source} (expected a bare global -- check Stage 07's export cell).")

    for name in _SOFT_PREREQUISITES:
        value, source = _find_in_globals(name, glb)
        if value is None:
            value, source = _find_in_stage07_json(name, bootstrap_logger)
            if value is not None:
                notes.append(f"{name} was not found as a notebook global; reconstructed from {source}.")
        if value is None:
            missing_soft.append(name)
        else:
            resolved[name] = value

    for n in notes:
        bootstrap_logger.warning("PREREQUISITE RECOVERY: %s", n)

    if missing_hard or missing_soft:
        lines = ["Stage 08 cannot start -- required Stage 01-07 outputs are missing from this kernel session."]
        if missing_hard:
            lines.append(
                "These are live objects Stage 08 cannot reconstruct from disk; "
                "re-run Stages 01-07 in order in this kernel session, then re-run Stage 08:"
            )
            lines += [f"  - {n}" for n in missing_hard]
        if missing_soft:
            lines.append(
                f"These JSON-shaped summaries could not be found or parsed under {STAGE7_DIR}:"
            )
            lines += [f"  - {n}" for n in missing_soft]
        raise RuntimeError("\n".join(lines))

    bootstrap_logger.info(
        "All %d Stage 08 prerequisites resolved (%d recovered via fallback, %d as direct globals).",
        len(_HARD_PREREQUISITES) + len(_SOFT_PREREQUISITES), len(notes),
        len(_HARD_PREREQUISITES) + len(_SOFT_PREREQUISITES) - len(notes),
    )
    return resolved


if __name__ == "__main__":
    _prereqs = resolve_stage08_prerequisites()
    _stage08_result = run_stage08(
        config_registry=_prereqs["CONFIG_REGISTRY"],
        artifact_registry=_prereqs["ARTIFACT_REGISTRY"],
        model_registry=_prereqs["MODEL_REGISTRY"],
        metadata_registry=_prereqs["METADATA_REGISTRY"],
        threshold_registry=_prereqs["THRESHOLD_REGISTRY"],
        export_registry=_prereqs["EXPORT_REGISTRY"],
        inference_engine=_prereqs["INFERENCE_ENGINE"],
        deployment_manifest=_prereqs["DEPLOYMENT_MANIFEST"],
        deployment_metadata=_prereqs["DEPLOYMENT_METADATA"],
        compatibility_report=_prereqs["COMPATIBILITY_REPORT"],
        reproducibility_manifest=_prereqs["REPRODUCIBILITY_MANIFEST"],
        integrity_report=_prereqs["INTEGRITY_REPORT"],
        deployment_readiness=_prereqs["DEPLOYMENT_READINESS"],
        package_summary=_prereqs["PACKAGE_SUMMARY"],
        reconstructed_model=_prereqs["RECONSTRUCTED_MODEL"],
    )

2026-07-08 11:02:17 | WARNING  | PREREQUISITE RECOVERY: PACKAGE_SUMMARY was not found as a notebook global; reconstructed from /kaggle/working/sprint05_deployment/stage07_deployment_package/stage07_summary.json.
2026-07-08 11:02:17 | WARNING  | PREREQUISITE RECOVERY: PACKAGE_SUMMARY was not found as a notebook global; reconstructed from /kaggle/working/sprint05_deployment/stage07_deployment_package/stage07_summary.json.
2026-07-08 11:02:17 | INFO     | All 15 Stage 08 prerequisites resolved (1 recovered via fallback, 14 as direct globals).
2026-07-08 11:02:17 | INFO     | All 15 Stage 08 prerequisites resolved (1 recovered via fallback, 14 as direct globals).
2026-07-08 11:02:17 | INFO     | START Sprint05-Stage08 Deployment Documentation & Release Engineering
2026-07-08 11:02:17 | INFO     | RESOURCES [RESOURCE START] cpu=1.2% ram=8.5%(2.2GB/31.4GB)
2026-07-08 11:02:17 | INFO     | MODULE START module=1 name=ExecutiveDeploymentSummary
2026-07-08 11:02:17 | INFO     | ARTIFACT WRITTEN 

In [26]:
from pathlib import Path
import shutil
import os

deployment_root = Path("/kaggle/working/sprint05_deployment")

zip_name = "/kaggle/working/VisionServeAI_Sprint05_Complete"

# remove old zip if exists
if os.path.exists(zip_name + ".zip"):
    os.remove(zip_name + ".zip")

shutil.make_archive(
    base_name=zip_name,
    format="zip",
    root_dir=deployment_root
)

print("=" * 60)
print("Sprint 05 archived successfully")
print("=" * 60)
print(zip_name + ".zip")
print("=" * 60)

zip_path = Path(zip_name + ".zip")

print(f"Size : {zip_path.stat().st_size / (1024*1024):.2f} MB")

Sprint 05 archived successfully
/kaggle/working/VisionServeAI_Sprint05_Complete.zip
Size : 110.21 MB
